# Revised baseline: c4_adapted, codenet, seed 123

One model/seed/condition per Save & Run, with a 10-hour session and 45-minute
output reserve. Training uses validation-based selection; final test remains
off while choosing the protocol. Time-limited jobs checkpoint and remain
incomplete; reattach progress.pt to continue. A 100-epoch ceiling does not
guarantee convergence or completion in one session.

Legacy encoders are the project's portable adaptations. C4 is explicitly a
task-masked contrastive adaptation of the authors' CodeBERT method, not an
exact reproduction (https://github.com/Chenning-Tao/C4). Qwen uses LoRA and
classification plus pair-contrastive training. CodeBERT is a symmetric Siamese
classifier. None of these is relabeled as FSD-CLCD, ZC3, or GraphCodeBERT.

Transfer jobs use 6,000 pairs and 30 epochs / 1,410 optimizer steps. Physical
batches differ for memory, so in-batch contrastive negatives are not equivalent
to a 128-example batch; report physical and effective sizes separately.
Main jobs use validation stopping. Learning rate is method-specific and editable;
if tuning it, compare prespecified candidates on validation only and freeze the
choice before final testing and running the remaining seeds.

Use the SAME derived common-support dataset for all compared methods. Missing
required graph views cause an error rather than silent test-pair exclusions.
For transformers enable Internet and install the optional requirements first.


In [ ]:
# SPECTRA notebook progress: begin
from tqdm.auto import tqdm

def _spectra_enable_cell_progress():
    from IPython import get_ipython
    _shell = get_ipython()
    if _shell is None:
        return
    _old = getattr(_shell, "_spectra_cell_progress", None)
    if _old is not None:
        _old[0].close()
        for _event, _callback in _old[1:]:
            _shell.events.unregister(_event, _callback)
    _bar = tqdm(total=3, desc="Notebook cells", unit="cell", mininterval=2)
    def _after_cell(_result):
        if _bar.n < _bar.total:
            _bar.update(1)
        if getattr(_result, "error_in_exec", None) or getattr(_result, "error_before_exec", None):
            _bar.set_description("Notebook cell failed")
            _bar.close()
        elif _bar.n >= _bar.total:
            _bar.close()
    _shell.events.register("post_run_cell", _after_cell)
    _shell._spectra_cell_progress = (_bar, ("post_run_cell", _after_cell))

_spectra_enable_cell_progress()
# SPECTRA notebook progress: end

from pathlib import Path
import time
SESSION_STARTED = time.monotonic()
METHOD = 'c4_adapted'
DATASET = 'codenet'
SEED = 123
SESSION_HOURS = 10.0
MAX_EPOCHS = 100
MIN_EPOCHS = 10
PATIENCE = 10
BATCH_SIZE = 8
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = None  # method default; freeze validation-selected changes across seeds
TRANSFER_KEY = None
MATCHED_TRAIN_PAIRS = 6000
CLEAN_DATA_DIR = None
RESUME_CHECKPOINT = None
ALLOW_NUMERICAL_UPGRADE = False
RESUME_TOKENS_DIR = None
MODEL_REVISION = None  # resolved to a fixed model commit at first token-cache creation
FINAL_TEST = False
MEASURE_COST = True


In [ ]:
# Embedded source snapshot; generated by scripts/build_revision_notebooks.py.
# No repository checkout or network installation is required for graph experiments.
import sys
_sources = {'__init__.py': '"""Versioned, resumable reviewer-response experiments."""\n', 'alignment.py': '"""Recover fallback DDG->AST identity only after exact syntax-tree validation.\n\nFallback parsers number AST nodes and statement DDG nodes independently. Their\nnumeric/string IDs must never be joined heuristically. This local preparation\nstep reproduces the original AST traversal, checks exported node types AND\nedges, and writes an explicit mapping sidecar for portable Kaggle consumers.\nExternal dependency nodes without an AST counterpart remain unmapped.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast as py_ast\nimport gzip\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef fallback_mapping(source: str, language: str, ast: dict, ddg: dict) -> tuple[list, str]:\n    types, edges, objects = [], [], {}\n    nodes = []\n    if language == "python" and ast.get("node_types", [None])[0] == "Module":\n        try:\n            # Same Python-2/operator repair path as the existing exporter,\n            # used only locally. The generated sidecar needs no parser on Kaggle.\n            from spectral_code.preprocessing.graph_parser import _parse_python_source, _ordered_statements\n            tree = _parse_python_source(source)\n        except ImportError:\n            tree = py_ast.parse(source)\n        if tree is None:\n            return [], "source_parse_failed"\n\n        def visit(node, parent=None):\n            i = len(types)\n            objects[id(node)] = i\n            types.append(type(node).__name__)\n            if parent is not None:\n                edges.append((parent, i))\n            for _, value in py_ast.iter_fields(node):\n                for child in value if isinstance(value, list) else [value]:\n                    if isinstance(child, py_ast.AST):\n                        visit(child, i)\n        visit(tree)\n        statements = _ordered_statements(tree)\n        mapping = {f"s{i}": objects[id(node)] for i, node in enumerate(statements)}\n    elif language == "csharp" and ast.get("node_types", [None])[0] == "compilation_unit":\n        from tree_sitter import Language, Parser\n        import tree_sitter_c_sharp\n        from spectral_code.preprocessing.graph_parser import CSHARP_METHOD_NODES\n        tree = Parser(Language(tree_sitter_c_sharp.language())).parse(source.encode("utf-8"))\n        root = tree.root_node\n        stack = [(root, None)]\n        while stack:\n            node, parent = stack.pop()\n            i = len(types)\n            objects[node.id] = i\n            types.append(node.type); nodes.append(node)\n            if parent is not None:\n                edges.append((parent, i))\n            stack.extend((child, i) for child in reversed(node.named_children))\n        methods = [node for node in nodes if node.type in CSHARP_METHOD_NODES] or [root]\n        mapping = {}\n        statement_index = 0\n        for method_index, method in enumerate(methods):\n            mapping[f"entry:{method_index}"] = objects[method.id]\n            body = method.child_by_field_name("body") if method is not root else root\n            if body is None:\n                continue\n            local, stack = [], [body]\n            while stack:\n                node = stack.pop()\n                if node.type.endswith("_statement") and node.type != "global_statement":\n                    local.append(node)\n                stack.extend(reversed(node.named_children))\n            for node in sorted(local, key=lambda n: (n.start_byte, n.end_byte)):\n                mapping[f"s{statement_index}"] = objects[node.id]\n                statement_index += 1\n    else:\n        return [], "not_fallback_ast"\n    expected_edges = set(zip(ast.get("row", []), ast.get("col", [])))\n    if types != ast.get("node_types") or set(edges) != expected_edges:\n        return [], "ast_reproduction_mismatch"\n    ast_ids = ast.get("node_ids", [])\n    result = []\n    for i, node_id in enumerate(ddg.get("node_ids", [])):\n        position = mapping.get(str(node_id))\n        # Confirm statement type too; a stale DDG from another source is rejected.\n        if position is not None and i < len(ddg.get("node_types", [])):\n            ddg_type = ddg["node_types"][i]\n            if not str(node_id).startswith("entry:") and ddg_type != types[position]:\n                return [], "ddg_statement_type_mismatch"\n        result.append(ast_ids[position] if position is not None else None)\n    return result, "validated_ast_and_ddg_statement_types"\n\n\ndef _retained(ast, ddg, mapping, max_nodes):\n    valid_ast = set(str(x) for x in ast.get("node_ids", [])[:max_nodes])\n    count = 0\n    for a, b in zip(ddg.get("row", []), ddg.get("col", [])):\n        a, b = int(a), int(b)\n        if 0 <= a < len(mapping) and 0 <= b < len(mapping):\n            count += str(mapping[a]) in valid_ast and str(mapping[b]) in valid_ast\n    return count\n\n\ndef build_alignment(root: Path, max_nodes: int = 256) -> dict:\n    codes = {}\n    with gzip.open(root / "codes.jsonl.gz", "rt", encoding="utf-8") as stream:\n        for line in stream:\n            code = json.loads(line)\n            if code["language"] in {"python", "csharp"}:\n                codes[str(code["code_id"])] = code\n    output = root / "ddg_ast_alignment.jsonl.gz"\n    stats = defaultdict(Counter)\n    with gzip.open(root / "graph_spectra.jsonl.gz", "rt", encoding="utf-8") as stream, gzip.open(output, "wt", encoding="utf-8", compresslevel=1) as dest:\n        for line in stream:\n            record = json.loads(line)\n            code = codes.get(str(record["code_id"]))\n            if code is None:\n                continue\n            ast = record["graphs"].get("ast", {}).get("adjacency", {})\n            ddg = record["graphs"].get("ddg", {}).get("adjacency", {})\n            if not ast.get("node_types"):\n                continue\n            s = stats[code["language"]]\n            s["codes"] += 1\n            s["exported_edges"] += len(ddg.get("row", []))\n            s["projectable_before"] += _retained(ast, ddg, ddg.get("node_ids", []), max_nodes)\n            mapping, reason = fallback_mapping(code["code"], code["language"], ast, ddg)\n            s[reason] += 1\n            if mapping:\n                dest.write(json.dumps({"code_id": str(record["code_id"]), "ast_node_ids": mapping}) + "\\n")\n                s["projectable_after"] += _retained(ast, ddg, mapping, max_nodes)\n            else:\n                s["projectable_after"] += _retained(ast, ddg, ddg.get("node_ids", []), max_nodes)\n    report = {"method": "source reparse, exact exported AST types/edges, DDG statement type validation",\n              "external_nodes": "unmapped; no synthetic AST correspondence", "max_nodes": max_nodes,\n              "languages": {lang: dict(counts) for lang, counts in stats.items()}}\n    (root / "ddg_alignment_audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    digest = hashlib.sha256(output.read_bytes()).hexdigest()\n    path = root / "metadata.json"\n    manifest = json.loads(path.read_text(encoding="utf-8"))\n    content = hashlib.sha256()\n    with gzip.open(output, "rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            content.update(block)\n    manifest["ddg_ast_alignment"] = {"file": output.name, "sha256": digest,\n                                     "content_sha256": content.hexdigest(), "audit": report}\n    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("root", type=Path)\n    args = parser.parse_args()\n    print(json.dumps(build_alignment(args.root), indent=2))\n', 'audit.py': '"""Language-wise exported-graph coverage without claiming unavailable parser logs."""\nfrom __future__ import annotations\n\nimport argparse\nimport gzip\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .canonical_types import get_canonical_id\nfrom .data import CleanDataCorpus, _open_text, _project_ddg\n\n\ndef audit_graphs(root: Path, max_nodes: int = 256) -> dict:\n    languages = {}\n    with gzip.open(root / "codes.jsonl.gz", "rt", encoding="utf-8") as stream:\n        for line in stream:\n            row = json.loads(line)\n            languages[str(row["code_id"])] = row["language"]\n    stats, types = defaultdict(Counter), defaultdict(Counter)\n    with gzip.open(root / "graph_spectra.jsonl.gz", "rt", encoding="utf-8") as stream:\n        for line in stream:\n            row = json.loads(line)\n            language = languages.get(str(row["code_id"]))\n            if language is None:\n                continue\n            s = stats[language]\n            s["codes"] += 1\n            layers = row.get("graphs", {})\n            ast = layers.get("ast", {}).get("adjacency", {})\n            raw_types = ast.get("node_types", [])\n            s["ast_backbone_available"] += bool(raw_types)\n            s["truncated_codes"] += len(raw_types) > max_nodes\n            s["retained_nodes"] += min(max_nodes, len(raw_types))\n            types[language].update(str(t) for t in raw_types[:max_nodes])\n            for layer in ("ast", "cfg", "ddg", "cpg"):\n                info = layers.get(layer, {})\n                s[f"{layer}_missing_spectra"] += not bool(info.get("eigenvalues"))\n                status = str(info.get("spectral_status", "unreported"))\n                s[f"{layer}_status_{status}"] += 1\n            ddg = layers.get("ddg", {}).get("adjacency", {})\n            ast_index = {str(node): i for i, node in enumerate(ast.get("node_ids", [])[:max_nodes])}\n            ddg_ids = ddg.get("node_ids", [])\n            edges = list(zip(ddg.get("row", []), ddg.get("col", [])))\n            s["ddg_empty_export"] += not bool(edges)\n            s["ddg_exported_edges"] += len(edges)\n            for a, b in edges:\n                a, b = int(a), int(b)\n                if 0 <= a < len(ddg_ids) and 0 <= b < len(ddg_ids):\n                    s["ddg_edges_projectable_at_cap"] += str(ddg_ids[a]) in ast_index and str(ddg_ids[b]) in ast_index\n    unknown = get_canonical_id("Canonical_Unknown")\n    report = {"max_nodes": max_nodes, "parser_success": "unavailable from exports alone; AST availability is a proxy",\n              "ddg_warning": "An empty DDG is not necessarily a parser failure", "languages": {}}\n    for language, counts in stats.items():\n        counts["unknown_retained_nodes"] = sum(n for raw, n in types[language].items()\n                                               if get_canonical_id(raw, language=language) == unknown)\n        report["languages"][language] = {\n            **dict(counts), "truncated_fraction": counts["truncated_codes"] / max(1, counts["codes"]),\n            "unknown_retained_node_fraction": counts["unknown_retained_nodes"] / max(1, counts["retained_nodes"]),\n            "ddg_projectable_edge_fraction": counts["ddg_edges_projectable_at_cap"] / max(1, counts["ddg_exported_edges"]),\n        }\n    (root / "coverage_by_language.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\ndef audit_split_coverage(root: Path, max_nodes: int = 256) -> dict:\n    """Coverage on actual retained pairs, with raw and repaired DDG counts."""\n    corpus = CleanDataCorpus(root)\n    membership = {}\n    for split in ("train", "valid", "test"):\n        frame = corpus.pairs(split)\n        for key in set(frame.left_id) | set(frame.right_id):\n            if key in membership and membership[key] != split:\n                raise ValueError("Endpoint occurs in multiple splits")\n            membership[key] = split\n    stats, seen = defaultdict(Counter), set()\n    unknown = get_canonical_id("Canonical_Unknown")\n    with _open_text(corpus.graphs_path) as stream:\n        for line in stream:\n            row = json.loads(line)\n            key = str(row.get("code_id"))\n            if key not in membership:\n                continue\n            seen.add(key)\n            language, split = corpus.code_languages[key], membership[key]\n            counts = stats[(language, split)]\n            counts["programs"] += 1\n            layers = row.get("graphs", {})\n            ast = layers.get("ast", {}).get("adjacency", {})\n            types = ast.get("node_types", [])\n            counts["ast_available"] += bool(types)\n            counts["truncated_codes"] += len(types) > max_nodes\n            retained = types[:max_nodes]\n            counts["retained_nodes"] += len(retained)\n            counts["unknown_retained_nodes"] += sum(get_canonical_id(t, language=language) == unknown for t in retained)\n            ddg = layers.get("ddg", {}).get("adjacency", {})\n            counts["empty_ddg_export"] += not bool(ddg.get("row"))\n            counts["ddg_exported_edges"] += len(ddg.get("row", []))\n            counts["ddg_projectable_raw"] += len(_project_ddg(ast, ddg, max_nodes))\n            repaired = {**ddg, "ast_node_ids": corpus.ddg_alignment[key]} if key in corpus.ddg_alignment else ddg\n            counts["ddg_projectable_after_alignment"] += len(_project_ddg(ast, repaired, max_nodes))\n    for key in set(membership) - seen:\n        stats[(corpus.code_languages[key], membership[key])]["missing_graph_export"] += 1\n    rows = []\n    for (language, split), counts in sorted(stats.items()):\n        rows.append({"language": language, "split": split, **dict(counts),\n                     "truncated_fraction": counts["truncated_codes"] / max(1, counts["programs"]),\n                     "unknown_node_fraction": counts["unknown_retained_nodes"] / max(1, counts["retained_nodes"])})\n    pd.DataFrame(rows).to_csv(root / "coverage_by_language_split.csv", index=False)\n    result = {"max_nodes": max_nodes, "records": rows,\n              "parser_success": "Unavailable from exports; AST availability is not a parser success log",\n              "ddg_failure": "Empty DDGs are reported as empty exports, not extraction failures"}\n    (root / "coverage_by_language_split.json").write_text(json.dumps(result, indent=2), encoding="utf-8")\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--max-nodes", type=int, default=256)\n    parser.add_argument("--by-split", action="store_true")\n    args = parser.parse_args()\n    print(json.dumps((audit_split_coverage if args.by_split else audit_graphs)(args.root, args.max_nodes), indent=2))\n', 'batching.py': '"""PyTorch Dataset/DataLoader adapters for canonical code graphs."""\n\nfrom __future__ import annotations\n\nfrom .data import CanonicalGraph\n\nfrom dataclasses import dataclass\nfrom typing import Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\n\n\n\n@dataclass\nclass CanonicalGraphBatch:\n    canonical_ids: torch.Tensor\n    lexical_ids: torch.Tensor\n    source_lexical_ids: torch.Tensor\n    mask: torch.Tensor\n    edges: torch.Tensor\n\n    def to(self, device: str | torch.device) -> "CanonicalGraphBatch":\n        return CanonicalGraphBatch(\n            self.canonical_ids.to(device, non_blocking=True),\n            self.lexical_ids.to(device, non_blocking=True),\n            self.source_lexical_ids.to(device, non_blocking=True),\n            self.mask.to(device, non_blocking=True),\n            self.edges.to(device, non_blocking=True),\n        )\n\n\nclass CanonicalPairDataset(Dataset):\n    def __init__(self, pairs: pd.DataFrame, graphs: dict[str, CanonicalGraph]) -> None:\n        usable = pairs[pairs["left_id"].isin(graphs) & pairs["right_id"].isin(graphs)].copy()\n        self.dropped_pairs = int(len(pairs) - len(usable))\n        if usable.empty:\n            raise RuntimeError("No usable pairs remain after joining pair endpoints to graph records.")\n        self.left_ids = usable["left_id"].astype(str).tolist()\n        self.right_ids = usable["right_id"].astype(str).tolist()\n        self.labels = usable["label"].to_numpy(dtype=np.float32)\n        self.graphs = graphs\n\n    def __len__(self) -> int:\n        return len(self.labels)\n\n    def __getitem__(self, index: int):\n        return self.graphs[self.left_ids[index]], self.graphs[self.right_ids[index]], self.labels[index]\n\n\ndef pack_graphs(graphs: Sequence[CanonicalGraph], max_nodes: int) -> CanonicalGraphBatch:\n    batch_size = len(graphs)\n    lexical_slots = graphs[0].lexical_ids.shape[1]\n    canonical_ids = torch.zeros(batch_size, max_nodes, dtype=torch.long)\n    lexical_ids = torch.zeros(batch_size, max_nodes, lexical_slots, dtype=torch.long)\n    source_slots = graphs[0].source_lexical_ids.shape[0]\n    source_lexical = torch.zeros(batch_size, source_slots, dtype=torch.long)\n    mask = torch.zeros(batch_size, max_nodes, dtype=torch.bool)\n    edges = torch.zeros(batch_size, 3, max_nodes, max_nodes, dtype=torch.float32)\n    for batch_index, graph in enumerate(graphs):\n        node_count = min(graph.node_count, max_nodes)\n        canonical_ids[batch_index, :node_count] = torch.from_numpy(graph.canonical_ids[:node_count])\n        lexical_ids[batch_index, :node_count] = torch.from_numpy(graph.lexical_ids[:node_count])\n        source_lexical[batch_index] = torch.from_numpy(graph.source_lexical_ids)\n        mask[batch_index, :node_count] = True\n        for relation, relation_edges in enumerate(graph.edges):\n            if not len(relation_edges):\n                continue\n            valid = relation_edges[(relation_edges[:, 0] < node_count) & (relation_edges[:, 1] < node_count)]\n            if len(valid):\n                rows = torch.from_numpy(valid[:, 0].astype(np.int64))\n                cols = torch.from_numpy(valid[:, 1].astype(np.int64))\n                edges[batch_index, relation, rows, cols] = 1.0\n    return CanonicalGraphBatch(canonical_ids, lexical_ids, source_lexical, mask, edges)\n\n\ndef make_collate(max_nodes: int):\n    def collate(items):\n        left, right, labels = zip(*items)\n        return pack_graphs(left, max_nodes), pack_graphs(right, max_nodes), torch.tensor(labels, dtype=torch.float32)\n\n    return collate\n\n\ndef make_loader(\n    pairs: pd.DataFrame,\n    graphs: dict[str, CanonicalGraph],\n    *,\n    max_nodes: int,\n    batch_size: int,\n    shuffle: bool,\n    num_workers: int = 0,\n    pin_memory: bool = True,\n) -> DataLoader:\n    dataset = CanonicalPairDataset(pairs, graphs)\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        collate_fn=make_collate(max_nodes),\n    )\n', 'benchmark.py': '"""Comparable measured inference and forward/backward costs on fixed support."""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .model import canonical_spectra_loss\nfrom .protocol import fingerprint\nfrom .training import _loader, SessionBudgetReached\n\n\ndef benchmark_model(model, frame, packed, config, output, *, losses=None, clock=None, warmup=2, batches=None):\n    if (batches is not None and batches < 1) or warmup < 0:\n        raise ValueError("Positive measured batch count and nonnegative warmup required")\n    device = next(model.parameters()).device\n    # No optimizer updates: timing never modifies the learned checkpoint.\n    loss_function = getattr(model, "compute_loss", canonical_spectra_loss)\n    def sync():\n        if device.type == "cuda":\n            torch.cuda.synchronize(device)\n    rows = frame.sample(frac=1, random_state=9173).head(256).reset_index(drop=True)\n    results = {}\n    was_training = model.training\n    try:\n        for phase in ("inference", "forward_backward"):\n            model.train(phase == "forward_backward")\n            if device.type == "cuda":\n                torch.cuda.reset_peak_memory_stats(device)\n            times, counts = [], []\n            warmup_loader = _loader(rows.head(warmup * config.batch_size), packed, config) if warmup else ()\n            for left, right, _ in warmup_loader:\n                if clock:\n                    clock.check()\n                with torch.no_grad(), torch.autocast(device_type=device.type, enabled=config.amp and device.type == "cuda"):\n                    model(left.to(device), right.to(device), None)\n            for index, (left, right, labels) in enumerate(_loader(rows, packed, config)):\n                if batches is not None and index >= batches:\n                    break\n                if clock:\n                    clock.check()\n                left, right, labels = left.to(device), right.to(device), labels.to(device)\n                sync()\n                started = time.perf_counter()\n                with torch.set_grad_enabled(phase == "forward_backward"):\n                    with torch.autocast(device_type=device.type, enabled=config.amp and device.type == "cuda"):\n                        logits, auxiliary = model(left, right, None)\n                        if phase == "forward_backward":\n                            loss, _ = loss_function(logits, auxiliary, labels,\n                                                     positive_class_weight=1.0, **(losses or {}))\n                    if phase == "forward_backward":\n                        loss.backward()\n                        model.zero_grad(set_to_none=True)\n                sync()\n                elapsed = time.perf_counter() - started\n                times.append(elapsed); counts.append(len(labels))\n            if not times:\n                raise ValueError("Not enough examples after warmup to measure timing")\n            results[phase] = {"measured_batches": len(times), "pairs": sum(counts),\n                              "seconds_per_pair": sum(times) / sum(counts),\n                              "pairs_per_second": sum(counts) / sum(times),\n                              "batch_seconds_median": float(np.median(times)),\n                              "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None}\n        results.update(status="complete", physical_batch=config.batch_size, effective_batch=config.effective_batch_size,\n                       trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),\n                       gpu=torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",\n                       torch_version=str(torch.__version__), warmup_batches=warmup,\n                       support_hash=fingerprint(rows.iloc[:sum(counts)][["left_id", "right_id", "label"]].to_dict("records")),\n                       scope="model compute after inputs reach device; excludes loading, transfers, optimizer step and checkpoint I/O")\n    except SessionBudgetReached:\n        results.update(status="incomplete_session_budget")\n    finally:\n        model.train(was_training)\n        model.zero_grad(set_to_none=True)\n    Path(output).parent.mkdir(parents=True, exist_ok=True)\n    Path(output).write_text(json.dumps(results, indent=2), encoding="utf-8")\n    return results\n\n\ndef benchmark_checkpoint(checkpoint, frame, packed, config, output, *, model_factory,\n                         model_config_factory, clock=None):\n    from .training import _load_checkpoint\n    if clock:\n        try:\n            clock.check()\n        except SessionBudgetReached:\n            result = {"status": "pending_session_budget", "continuation": "Reattach completed checkpoint and rerun timing"}\n            Path(output).write_text(json.dumps(result, indent=2), encoding="utf-8")\n            return result\n    state = _load_checkpoint(checkpoint, "cpu")\n    if state["best_model"] is None:\n        raise ValueError("A validation-selected checkpoint is required for timing")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model_factory(model_config_factory(**state["model_config"])).to(device)\n    model.load_state_dict(state["best_model"])\n    return benchmark_model(model, frame, packed, config, output, losses=state["losses"], clock=clock)\n', 'canonical_types.py': '"""Language-agnostic canonical node types for code graphs.\n\nParser node names are an implementation detail: the same construct may be an\n``If``, ``if_statement`` or ``CONTROL_STRUCTURE_IF`` depending on whether the\ngraph came from Python\'s :mod:`ast`, tree-sitter, or Joern.  This module maps\nthose heterogeneous names to a small, stable vocabulary suitable for a shared\nembedding table.\n\nThe mapper deliberately consumes *node types*, not source-code tokens.  It is\nalso conservative: ambiguous tags such as bare ``LITERAL`` and\n``CONTROL_STRUCTURE`` map to their explicit generic categories, rather than\nbeing guessed as a numeric literal or a particular branch construct.\n\nIDs are stable, contiguous, and zero-based.  If a downstream model reserves\nzero for padding, store ``get_canonical_id(raw_type) + 1`` and allocate\n``NUM_CANONICAL_TYPES + 1`` embeddings.\n"""\n\nfrom __future__ import annotations\n\n\nimport re\nimport html\nfrom collections.abc import Iterable\nfrom typing import Final\n\n\n# Do not reorder this tuple: its position is the serialized embedding ID.\nCANONICAL_CATEGORIES: Final[tuple[str, ...]] = (\n    "Control_If",\n    "Control_Loop",\n    "Control_Switch",\n    "Control_Return",\n    "Control_Break",\n    "Var_Decl",\n    "Func_Decl",\n    "Class_Decl",\n    "Param_Decl",\n    "Assign_Op",\n    "Binary_Op",\n    "Unary_Op",\n    "Call_Expr",\n    "Access_Expr",\n    "Literal_Num",\n    "Literal_Str",\n    "Literal_Bool",\n    "Literal_Null",\n    "Structural_Block",\n    "Identifier_Context",\n    "Literal_Generic",\n    "Control_Generic",\n    "Type_Meta",\n    "Canonical_Unknown",\n)\n\nCANONICAL_TYPE_TO_ID: Final[dict[str, int]] = {\n    name: index for index, name in enumerate(CANONICAL_CATEGORIES)\n}\nCANONICAL_ID_TO_TYPE: Final[tuple[str, ...]] = CANONICAL_CATEGORIES\nNUM_CANONICAL_TYPES: Final[int] = len(CANONICAL_CATEGORIES)\nCANONICAL_UNKNOWN_ID: Final[int] = CANONICAL_TYPE_TO_ID["Canonical_Unknown"]\n\n\n_CAMEL_WORD_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")\n_ACRONYM_WORD_BOUNDARY = re.compile(r"(?<=[A-Z])(?=[A-Z][a-z])")\n_NON_ALNUM = re.compile(r"[^a-z0-9]+")\n\n\ndef _normalize(raw_type: object) -> str:\n    """Normalize parser spelling without reading source-level token content."""\n    if not isinstance(raw_type, str):\n        return ""\n    text = html.unescape(raw_type).strip()\n    if not text:\n        return ""\n    text = _ACRONYM_WORD_BOUNDARY.sub("_", text)\n    text = _CAMEL_WORD_BOUNDARY.sub("_", text)\n    return _NON_ALNUM.sub("_", text.lower()).strip("_")\n\n\n# Exact aliases cover the common output of Joern, Python ast, JavaParser,\n# srcML, clang/tree-sitter C, and tree-sitter Python/Java.  Keys are normalized\n# once at import time, so lookup remains inexpensive inside a DataLoader.\n_ALIASES_BY_CATEGORY: Final[dict[str, tuple[str, ...]]] = {\n    "Control_If": (\n        "operator_conditional",\n        "if",\n        "if_stmt",\n        "if_statement",\n        "if_expression",\n        "if_exp",\n        "if_then_statement",\n        "if_then_else_statement",\n        "conditional_expression",\n        "ternary_expression",\n        "control_structure_if",\n    ),\n    "Control_Loop": (\n        "for",\n        "for_stmt",\n        "for_statement",\n        "for_each",\n        "for_each_statement",\n        "foreach_statement",\n        "enhanced_for_statement",\n        "async_for",\n        "while",\n        "while_stmt",\n        "while_statement",\n        "do",\n        "do_stmt",\n        "do_statement",\n        "do_while_statement",\n        "loop_statement",\n        "list_comp",\n        "set_comp",\n        "comprehension",\n        "dict_comp",\n        "generator_exp",\n        "control_structure_for",\n        "control_structure_while",\n        "control_structure_do",\n    ),\n    "Control_Switch": (\n        "switch",\n        "switch_stmt",\n        "switch_statement",\n        "switch_expression",\n        "switch_entry",\n        "case",\n        "case_statement",\n        "case_label",\n        "match",\n        "match_statement",\n        "match_case",\n        "control_structure_switch",\n    ),\n    "Control_Return": (\n        "return",\n        "return_stmt",\n        "return_statement",\n        "yield",\n        "yield_expr",\n        "yield_expression",\n        "control_structure_return",\n    ),\n    "Control_Break": (\n        "break",\n        "break_stmt",\n        "break_statement",\n        "continue",\n        "continue_stmt",\n        "continue_statement",\n        "goto",\n        "goto_statement",\n        "control_structure_break",\n        "control_structure_continue",\n    ),\n    "Var_Decl": (\n        "local",\n        "member",\n        "field_declaration",\n        "variable_declaration",\n        "variable_declarator",\n        "local_variable_declaration",\n        "local_declaration_statement",\n        "declaration",\n        "declaration_statement",\n        "init_declarator",\n        "var_decl",\n        "let_declaration",\n        "const_declaration",\n        "global",\n        "nonlocal",\n    ),\n    "Func_Decl": (\n        "method",\n        "method_declaration",\n        "method_definition",\n        "function_declaration",\n        "function_definition",\n        "function_def",\n        "async_function_def",\n        "constructor_declaration",\n        "constructor_definition",\n        "lambda",\n        "lambda_expression",\n        "func_decl",\n    ),\n    "Class_Decl": (\n        "type_decl",\n        "class_decl",\n        "class_declaration",\n        "class_definition",\n        "class_def",\n        "interface_declaration",\n        "annotation_type_declaration",\n        "enum_declaration",\n        "struct_specifier",\n        "struct_declaration",\n        "union_specifier",\n        "record_declaration",\n    ),\n    "Param_Decl": (\n        "keyword",\n        "arg",\n        "arguments",\n        "parameter",\n        "parameter_declaration",\n        "formal_parameter",\n        "receiver_parameter",\n        "spread_parameter",\n        "typed_parameter",\n        "default_parameter",\n        "method_parameter_in",\n        "method_parameter_out",\n        "param_decl",\n    ),\n    "Assign_Op": (\n        "assign",\n        "ann_assign",\n        "aug_assign",\n        "named_expr",\n        "assignment",\n        "assignment_expression",\n        "assignment_operator",\n        "operator_assignment",\n        "operator_assignment_plus",\n        "operator_assignment_minus",\n        "operator_assignment_multiplication",\n        "operator_assignment_division",\n        "operator_assignment_modulo",\n        "operator_assignment_and",\n        "operator_assignment_or",\n        "operator_assignment_xor",\n        "operator_assignment_shift_left",\n        "operator_assignment_arithmetic_shift_right",\n        "operator_assignment_logical_shift_right",\n    ),\n    "Binary_Op": (\n        "bin_op",\n        "bool_op",\n        "compare",\n        "binary_expression",\n        "binary_operator",\n        "infix_expression",\n        "add",\n        "sub",\n        "mult",\n        "mat_mult",\n        "div",\n        "floor_div",\n        "mod",\n        "pow",\n        "l_shift",\n        "r_shift",\n        "bit_or",\n        "bit_xor",\n        "bit_and",\n        "and",\n        "or",\n        "eq",\n        "not_eq",\n        "lt",\n        "lt_e",\n        "gt",\n        "gt_e",\n        "is",\n        "is_not",\n        "in",\n        "not_in",\n        "operator_addition",\n        "operator_subtraction",\n        "operator_multiplication",\n        "operator_division",\n        "operator_modulo",\n        "operator_exponentiation",\n        "operator_shift_left",\n        "operator_arithmetic_shift_right",\n        "operator_logical_shift_right",\n        "operator_and",\n        "operator_or",\n        "operator_xor",\n        "operator_logical_and",\n        "operator_logical_or",\n        "operator_equals",\n        "operator_not_equals",\n        "operator_less_than",\n        "operator_less_equals_than",\n        "operator_greater_than",\n        "operator_greater_equals_than",\n    ),\n    "Unary_Op": (\n        "operator_size_of",\n        "operator_cast",\n        "unary_op",\n        "unary_expression",\n        "prefix_expression",\n        "postfix_expression",\n        "u_add",\n        "u_sub",\n        "not",\n        "invert",\n        "operator_not",\n        "operator_logical_not",\n        "operator_minus",\n        "operator_plus",\n        "operator_pre_increment",\n        "operator_post_increment",\n        "operator_pre_decrement",\n        "operator_post_decrement",\n        "operator_address_of",\n        "operator_indirection",\n    ),\n    "Call_Expr": (\n        "init",\n        "operator_alloc",\n        "operator_new",\n        "operator_constructor",\n        "call",\n        "call_expr",\n        "call_expression",\n        "method_call",\n        "method_call_expression",\n        "method_invocation",\n        "function_call",\n        "function_call_expression",\n        "object_creation_expression",\n        "class_instance_creation_expression",\n        "new_expression",\n    ),\n    "Access_Expr": (\n        "operator_index_access",\n        "slice",\n        "identifier",\n        "field_identifier",\n        "name",\n        "qualified_name",\n        "attribute",\n        "attribute_access",\n        "member_access",\n        "member_expression",\n        "field_access",\n        "field_expression",\n        "array_access",\n        "array_access_expression",\n        "subscript",\n        "index_expression",\n        "pointer_expression",\n        "this",\n        "super",\n        "external",\n    ),\n    "Literal_Num": (\n        "num",\n        "number",\n        "number_literal",\n        "numeric_literal",\n        "integer_literal",\n        "decimal_integer_literal",\n        "hex_integer_literal",\n        "octal_integer_literal",\n        "binary_integer_literal",\n        "floating_point_literal",\n        "float_literal",\n        "double_literal",\n        "imaginary_literal",\n        "literal_int",\n        "literal_integer",\n        "literal_float",\n        "literal_double",\n        "constant_int",\n        "constant_integer",\n        "constant_float",\n        "constant_double",\n        "constant_complex",\n    ),\n    "Literal_Str": (\n        "str",\n        "joined_str",\n        "formatted_value",\n        "string",\n        "string_literal",\n        "character_literal",\n        "char_literal",\n        "concatenated_string",\n        "template_string",\n        "literal_str",\n        "literal_string",\n        "literal_char",\n        "constant_str",\n        "constant_string",\n        "constant_bytes",\n    ),\n    "Literal_Bool": (\n        "bool",\n        "boolean",\n        "boolean_literal",\n        "true",\n        "false",\n        "true_literal",\n        "false_literal",\n        "literal_bool",\n        "literal_boolean",\n        "constant_bool",\n        "constant_boolean",\n    ),\n    "Literal_Null": (\n        "null",\n        "none",\n        "nil",\n        "nullptr",\n        "null_literal",\n        "none_literal",\n        "nil_literal",\n        "literal_null",\n        "literal_none",\n        "constant_null",\n        "constant_none",\n    ),\n    "Structural_Block": (\n        "expr",\n        "pass",\n        "block",\n        "module",\n        "compilation_unit",\n        "statement_list",\n        "expression_list",\n        "suite",\n    ),\n    "Identifier_Context": (\n        "load",\n        "store",\n        "del",\n        "starred",\n    ),\n    "Literal_Generic": (\n        "tuple",\n        "list",\n        "dict",\n        "set",\n        "operator_array_initializer",\n        "literal",\n        "constant",\n    ),\n    "Control_Generic": (\n        "try",\n        "except_handler",\n        "with",\n        "withitem",\n        "assert",\n        "raise",\n        "delete",\n        "operator_throw",\n        "control_structure",\n    ),\n    "Type_Meta": (\n        "alias",\n        "method_return",\n        "modifier",\n        "type",\n        "type_ref",\n        "annotation",\n        "import",\n        "import_from",\n        "package",\n        "namespace_block",\n    ),\n    "Canonical_Unknown": (\n        "canonical_unknown",\n        "unknown",\n        "unknown_node",\n    ),\n}\n\n\ndef _build_alias_table() -> dict[str, int]:\n    aliases: dict[str, int] = {}\n    for category, raw_aliases in _ALIASES_BY_CATEGORY.items():\n        category_id = CANONICAL_TYPE_TO_ID[category]\n        for raw_alias in (category, *raw_aliases):\n            alias = _normalize(raw_alias)\n            previous = aliases.setdefault(alias, category_id)\n            if previous != category_id:\n                raise RuntimeError(f"Conflicting canonical alias: {raw_alias!r}")\n    return aliases\n\n\nRAW_TYPE_TO_CANONICAL_ID: Final[dict[str, int]] = _build_alias_table()\n\n\n# Namespace-qualified parser classes are common (for example,\n# ``com.github.javaparser.ast.stmt.IfStmt``).  Exact aliases cover normal\n# inputs; these ordered patterns provide a conservative fallback for qualified\n# names and parser-specific prefixes/suffixes.\n_PATTERN_RULES: Final[tuple[tuple[int, re.Pattern[str]], ...]] = tuple(\n    (CANONICAL_TYPE_TO_ID[category], re.compile(pattern))\n    for category, pattern in (\n        (\n            "Control_If",\n            r"(?:^|_)(?:if)(?:_stmt|_statement|_expression|_exp)?(?:_|$)",\n        ),\n        (\n            "Control_Loop",\n            r"(?:^|_)(?:for|for_each|foreach|enhanced_for|while|do_while|loop)(?:_stmt|_statement|_expression)?(?:_|$)",\n        ),\n        (\n            "Control_Switch",\n            r"(?:^|_)(?:switch|case|match)(?:_stmt|_statement|_expression|_entry|_label)?(?:_|$)",\n        ),\n        ("Control_Return", r"(?:^|_)(?:return|yield)(?:_stmt|_statement|_expr|_expression)?(?:_|$)"),\n        ("Control_Break", r"(?:^|_)(?:break|continue|goto)(?:_stmt|_statement)?(?:_|$)"),\n        (\n            "Param_Decl",\n            r"(?:^|_)(?:formal_|method_)?(?:parameter|param|argument)(?:_in|_out|_declaration|_decl)?(?:_|$)",\n        ),\n        (\n            "Func_Decl",\n            r"(?:^|_)(?:function|method|constructor)(?:_declaration|_definition|_def|_decl)(?:_|$)",\n        ),\n        (\n            "Class_Decl",\n            r"(?:^|_)(?:class|interface|enum|struct|union|record|type)(?:_declaration|_definition|_specifier|_decl|_def)(?:_|$)",\n        ),\n        (\n            "Var_Decl",\n            r"(?:^|_)(?:variable|local_variable|field|member|var|let|const)(?:_declaration|_declarator|_statement|_decl)?(?:_|$)",\n        ),\n        (\n            "Assign_Op",\n            r"(?:^|_)(?:assign|assignment|ann_assign|aug_assign|named_expr)(?:_operator|_expression)?(?:_|$)",\n        ),\n        (\n            "Unary_Op",\n            r"(?:^|_)(?:unary|prefix|postfix)(?:_op|_operator|_expression)?(?:_|$)",\n        ),\n        (\n            "Binary_Op",\n            r"(?:^|_)(?:binary|infix|bin_op|bool_op|compare)(?:_op|_operator|_expression)?(?:_|$)",\n        ),\n        (\n            "Call_Expr",\n            r"(?:^|_)(?:call|invocation|method_call|function_call|object_creation|class_instance_creation)(?:_expr|_expression)?(?:_|$)",\n        ),\n        (\n            "Access_Expr",\n            r"(?:^|_)(?:identifier|qualified_name|attribute|member_access|field_access|field_expression|array_access|subscript|index_expression)(?:_|$)",\n        ),\n        (\n            "Literal_Bool",\n            r"(?:^|_)(?:bool|boolean|true|false)(?:_literal)?(?:_|$)",\n        ),\n        (\n            "Literal_Null",\n            r"(?:^|_)(?:null|none|nil|nullptr)(?:_literal)?(?:_|$)",\n        ),\n        (\n            "Literal_Str",\n            r"(?:^|_)(?:str|string|char|character|bytes)(?:_literal)?(?:_|$)",\n        ),\n        (\n            "Literal_Num",\n            r"(?:^|_)(?:num|number|numeric|integer|int|float|double|decimal|hex|octal|binary_integer)(?:_literal)?(?:_|$)",\n        ),\n    )\n)\n\n\ndef get_canonical_id(raw_type_string: str, *, language: str | None = None) -> int:\n    """Return a language-agnostic canonical ID for one parser/Joern node.\n\n    ``language`` is optional to preserve the public mapping API.  It is used\n    only for a Joern export quirk: some Java CALL nodes are stored as their\n    callee text (for example ``println`` or ``nextLong``) instead of ``CALL``.\n    Those labels are safely treated as ``Call_Expr`` after all exact and\n    parser-generic rules have failed; Java identifiers themselves are already\n    emitted as ``IDENTIFIER`` and are handled before this fallback.\n    """\n    normalized = _normalize(raw_type_string)\n    if not normalized:\n        return CANONICAL_UNKNOWN_ID\n\n    exact = RAW_TYPE_TO_CANONICAL_ID.get(normalized)\n    if exact is not None:\n        return exact\n\n    for category_id, pattern in _PATTERN_RULES:\n        if pattern.search(normalized):\n            return category_id\n\n    if str(language or "").strip().lower() == "java" and not normalized.startswith("operator_"):\n        return CANONICAL_TYPE_TO_ID["Call_Expr"]\n    return CANONICAL_UNKNOWN_ID\n\n\ndef get_canonical_name(raw_type_string: str) -> str:\n    """Return the canonical category name for a raw node type."""\n    return CANONICAL_ID_TO_TYPE[get_canonical_id(raw_type_string)]\n\n\ndef canonicalize_node_types(raw_types: Iterable[str]) -> list[int]:\n    """Map an iterable of raw node types while preserving input order."""\n    return [get_canonical_id(raw_type) for raw_type in raw_types]\n\n\n__all__ = [\n    "CANONICAL_CATEGORIES",\n    "CANONICAL_ID_TO_TYPE",\n    "CANONICAL_TYPE_TO_ID",\n    "CANONICAL_UNKNOWN_ID",\n    "NUM_CANONICAL_TYPES",\n    "RAW_TYPE_TO_CANONICAL_ID",\n    "canonicalize_node_types",\n    "get_canonical_id",\n    "get_canonical_name",\n]\n', 'clone_types.py': '"""Clone-type reporting from supplied authoritative positive-pair annotations."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .protocol import metrics\n\n\ndef report_clone_types(predictions, annotations, threshold, output):\n    frame = pd.read_csv(predictions, dtype={"pair_id": str})\n    types = pd.read_csv(annotations, dtype={"pair_id": str, "clone_type": str})\n    if not {"pair_id", "clone_type", "annotation_source"}.issubset(types):\n        raise ValueError("Annotations need pair_id, clone_type and annotation_source; do not infer Type-IV from token overlap")\n    if types.pair_id.duplicated().any() or types[["clone_type", "annotation_source"]].isna().any().any():\n        raise ValueError("Each pair needs one nonmissing authoritative annotation")\n    merged = frame.merge(types, on="pair_id", how="left", validate="one_to_one")\n    if merged.loc[merged.label.eq(0), "clone_type"].notna().any():\n        raise ValueError("Clone-type labels apply to positive pairs only")\n    negatives = merged[merged.label.eq(0)]\n    rows = []\n    for name, positives in merged[merged.label.eq(1) & merged.clone_type.notna()].groupby("clone_type"):\n        subset = pd.concat((positives, negatives))\n        rows.append({"clone_type": name, **metrics(subset.label, subset.probability, threshold),\n                     "positive_pairs": len(positives), "shared_negative_pairs": len(negatives),\n                     "annotation_sources": ";".join(sorted(positives.annotation_source.unique()))})\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(rows).to_csv(output / "clone_type_metrics.csv", index=False)\n    status = {"annotated_positives": int((merged.label.eq(1) & merged.clone_type.notna()).sum()),\n              "unannotated_positives": int((merged.label.eq(1) & merged.clone_type.isna()).sum()),\n              "threshold_source": "fixed validation-selected threshold supplied by completed experiment",\n              "interpretation": "Each type uses all shared negatives; report recall by type. F1/AP depend on this constructed prevalence; do not average types as independent tests."}\n    (output / "clone_type_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")\n    return status\n', 'data.py': '"""Portable clean-data loading for canonical mono/cross-language experiments.\n\nThe module deliberately separates parser-specific raw node strings from the\nmodel-facing representation.  Every node receives a canonical type id and a\nsmall, fixed-size hashed lexical sketch.  Raw strings never become vocabulary\nindices, which prevents a Java-only vocabulary from defining the Python/C\nfeature space during zero-shot evaluation.\n"""\n\nfrom __future__ import annotations\n\nfrom .canonical_types import NUM_CANONICAL_TYPES, get_canonical_id\n\nimport csv\nimport gzip\nimport hashlib\nimport html\nimport json\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable, Iterator, Sequence\n\nimport numpy as np\nimport pandas as pd\n\n\n\n\nRELATIONS = ("ast", "next_token", "ddg")\nLEXICAL_PAD_ID = 0\nLEXICAL_UNKNOWN_ID = 1\nDEFAULT_LEXICAL_BUCKETS = 4096\nDEFAULT_LEXICAL_SLOTS = 4\nINPUT_REPRESENTATION_VERSION = "java_callee_lexical_v2"\n\n\n@dataclass(frozen=True)\nclass PairLanguageFilter:\n    """Language constraints for both endpoints of a pair."""\n\n    left: str | None = None\n    right: str | None = None\n\n    @classmethod\n    def mono(cls, language: str) -> "PairLanguageFilter":\n        normalized = normalize_language(language)\n        return cls(normalized, normalized)\n\n    @classmethod\n    def cross(cls, left: str, right: str) -> "PairLanguageFilter":\n        return cls(normalize_language(left), normalize_language(right))\n\n\n@dataclass(frozen=True)\nclass CanonicalGraph:\n    code_id: str\n    language: str\n    canonical_ids: np.ndarray\n    lexical_ids: np.ndarray\n    source_lexical_ids: np.ndarray\n    edges: tuple[np.ndarray, np.ndarray, np.ndarray]\n    eigenvalues: dict[str, np.ndarray]\n\n    @property\n    def node_count(self) -> int:\n        return int(len(self.canonical_ids))\n\n\ndef normalize_language(language: str | None) -> str:\n    value = (language or "unknown").strip().lower()\n    aliases = {\n        "javasrc": "java",\n        "py": "python",\n        "pythonsrc": "python",\n        "csrc": "c",\n        "c++": "cpp",\n        "cxx": "cpp",\n    }\n    return aliases.get(value, value)\n\n\ndef _is_gzip_file(path: Path) -> bool:\n    with path.open("rb") as stream:\n        return stream.read(2) == b"\\x1f\\x8b"\n\n\ndef _open_text(path: Path):\n    return gzip.open(path, "rt", encoding="utf-8") if _is_gzip_file(path) else path.open("r", encoding="utf-8")\n\n\ndef _find_file(root: Path, *names: str) -> Path:\n    """Find normal gzip/JSON files and Kaggle\'s occasional ``.tmp`` form.\n\n    Kaggle can expose a ZIP member such as ``codes.jsonl.gz`` as\n    ``codes.jsonl/codes.jsonl.gz.tmp``.  The content is still valid gzip, so\n    accepting the suffix is safe; ``_open_text`` identifies compression from\n    the file magic rather than the filename.\n    """\n    matches = []\n    for name in names:\n        variants = (name, f"{name}.tmp")\n        for variant in variants:\n            direct = root / variant\n            if direct.is_file():\n                matches.append(direct)\n            matches.extend(path for path in root.rglob(variant) if path.is_file())\n    if not matches:\n        visible = [str(path.relative_to(root)) for path in root.rglob("*") if path.is_file()][:30] if root.exists() else []\n        raise FileNotFoundError(f"Missing {names} below {root}. Files seen: {visible}")\n    return min(set(matches), key=lambda path: (len(path.parts), str(path)))\n\n\ndef verify_alignment(root: Path, specification: dict) -> Path:\n    """Accept Kaggle gzip/plain/tmp layouts while verifying the same payload."""\n    name = specification["file"]\n    try:\n        path = _find_file(root, name, name.removesuffix(".gz"))\n    except FileNotFoundError as exc:\n        raise ValueError("Required DDG/AST alignment sidecar is missing") from exc\n    if hashlib.sha256(path.read_bytes()).hexdigest() == specification["sha256"]:\n        return path\n    if specification.get("content_sha256"):\n        digest = hashlib.sha256()\n        opener = gzip.open if _is_gzip_file(path) else open\n        with opener(path, "rb") as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b""):\n                digest.update(block)\n        if digest.hexdigest() == specification["content_sha256"]:\n            return path\n    raise ValueError("Required DDG/AST alignment sidecar has changed")\n\n\ndef _stable_bucket(value: str, bucket_count: int) -> int:\n    if bucket_count <= 2:\n        raise ValueError("lexical bucket count must be greater than two")\n    digest = hashlib.blake2b(value.encode("utf-8", errors="replace"), digest_size=8, person=b"spectral").digest()\n    return 2 + int.from_bytes(digest, "little") % (bucket_count - 2)\n\n\n_CAMEL_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")\n_TOKEN_SPLIT = re.compile(r"[^A-Za-z0-9]+")\n\n\ndef lexical_feature_ids(\n    raw_label: str,\n    *,\n    bucket_count: int = DEFAULT_LEXICAL_BUCKETS,\n    slots: int = DEFAULT_LEXICAL_SLOTS,\n) -> np.ndarray:\n    """Return a bounded hashed subtoken/character sketch for a node label.\n\n    Pure line-number labels are intentionally ignored. Numeric, string,\n    boolean, and null values are normalized before hashing. The output space\n    is fixed across languages and never fitted on the training language.\n    """\n    text = html.unescape(str(raw_label or "")).strip()\n    result = np.full(slots, LEXICAL_PAD_ID, dtype=np.int64)\n    if not text or text.isdigit():\n        return result\n    text = _CAMEL_BOUNDARY.sub(" ", text)\n    raw_tokens = [token.lower() for token in _TOKEN_SPLIT.split(text) if token]\n    normalized = []\n    for token in raw_tokens:\n        if token.isdigit():\n            token = "<num>"\n        elif token in {"true", "false"}:\n            token = "<bool>"\n        elif token in {"null", "none", "nullptr", "nil"}:\n            token = "<null>"\n        normalized.append(token)\n    # Keep the first few subtokens before character n-grams.  In particular,\n    # ``Name,total`` must retain ``total`` even when ``Name`` has many\n    # trigrams.  This is a bounded *fallback*, not a train-fitted vocabulary.\n    features: list[str] = [f"sub:{token}" for token in normalized]\n    for token in normalized[:2]:\n        compact = f"^{token}$"\n        features.extend(f"tri:{compact[index:index + 3]}" for index in range(max(0, len(compact) - 2)))\n    for index, feature in enumerate(features[:slots]):\n        result[index] = _stable_bucket(feature, bucket_count)\n    return result\n\n\n_SOURCE_TOKEN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*|\\d+|==|!=|<=|>=|&&|\\|\\||[+*/%<>=!&|~-]")\n_SOURCE_KEYWORDS = {\n    "def": "func", "function": "func", "void": "func", "class": "class",\n    "if": "if", "else": "else", "elif": "else", "switch": "switch", "case": "switch",\n    "for": "loop", "while": "loop", "do": "loop", "foreach": "loop",\n    "return": "return", "break": "break", "continue": "break",\n    "try": "exception", "catch": "exception", "except": "exception", "throw": "exception",\n    "import": "import", "from": "import", "package": "import",\n    "true": "bool", "false": "bool", "none": "null", "null": "null",\n}\n\n\ndef source_lexical_ids(\n    source_code: str,\n    *,\n    bucket_count: int = DEFAULT_LEXICAL_BUCKETS,\n    slots: int = 96,\n) -> np.ndarray:\n    """Return a bounded, language-neutral hashed sketch of source tokens.\n\n    Keywords are normalized to functional concepts and identifiers are split\n    into subtokens before hashing.  No train-fitted vocabulary and no raw AST\n    node labels are used.  Uniform sampling retains information from the whole\n    method instead of favouring Java boilerplate at the beginning.\n    """\n    output = np.full(slots, LEXICAL_PAD_ID, dtype=np.int64)\n    text = html.unescape(str(source_code or ""))\n    features = []\n    for token in _SOURCE_TOKEN.findall(text):\n        normalized = token.lower()\n        if normalized.isdigit():\n            features.append("src:num")\n        elif normalized in _SOURCE_KEYWORDS:\n            features.append(f"src:kw:{_SOURCE_KEYWORDS[normalized]}")\n        elif normalized[0].isalpha() or normalized[0] == "_":\n            split = [part.lower() for part in _CAMEL_BOUNDARY.sub(" ", token).replace("_", " ").split() if part]\n            features.extend(f"src:id:{part}" for part in split[:3])\n        else:\n            features.append(f"src:op:{normalized}")\n    if not features:\n        return output\n    if len(features) <= slots:\n        chosen = features\n    else:\n        chosen = [features[round(index * (len(features) - 1) / (slots - 1))] for index in range(slots)]\n    for index, feature in enumerate(chosen[:slots]):\n        output[index] = _stable_bucket(feature, bucket_count)\n    return output\n\n\ndef lexical_source(raw_type: str, raw_label: str, canonical_id: int, *, language: str | None = None) -> str:\n    """Select a source-level lexical fallback without using numeric line IDs.\n\n    Python fallback graphs serialize labels such as ``Name,total`` while many\n    Joern graphs serialize only a line number.  In the former case the parser\n    type prefix is removed; in the latter, unknown raw node text and recovered\n    Java callee names supply the fallback. Known structural parser tags are deliberately\n    not hashed, because their canonical category already represents them.\n    """\n    raw_type_text = html.unescape(str(raw_type or "")).strip()\n    label = html.unescape(str(raw_label or "")).strip()\n    if label:\n        prefix = raw_type_text + ","\n        if prefix and label.lower().startswith(prefix.lower()):\n            label = label[len(prefix):].strip()\n        if label and not label.isdigit() and label.lower() != raw_type_text.lower():\n            return label\n    # Java exports can put the callee (e.g. ``println``) in node_types while\n    # node_labels contains only a line number. Canonical mapping deliberately\n    # recognizes that as Call_Expr; retain the callee in the separate hashed\n    # lexical channel as well. Exact/parser-pattern structural tags remain\n    # excluded, so CALL/IDENTIFIER/etc. do not become lexical features.\n    unknown = get_canonical_id("Canonical_Unknown")\n    java_callee = (str(language or "").strip().lower() == "java"\n                   and canonical_id == get_canonical_id("CALL")\n                   and get_canonical_id(raw_type_text) == unknown)\n    if (canonical_id == unknown or java_callee) and raw_type_text and not raw_type_text.isdigit():\n        return raw_type_text\n    return ""\n\n\ndef _edge_array(rows: Sequence, cols: Sequence, node_count: int) -> np.ndarray:\n    pairs = [\n        (int(source), int(target))\n        for source, target in zip(rows, cols)\n        if 0 <= int(source) < node_count and 0 <= int(target) < node_count\n    ]\n    return np.asarray(pairs, dtype=np.int32).reshape(-1, 2) if pairs else np.empty((0, 2), dtype=np.int32)\n\n\ndef _project_ddg(ast: dict, ddg: dict, node_count: int) -> np.ndarray:\n    ast_ids = [str(value) for value in ast.get("node_ids", [])[:node_count]]\n    ast_index = {node_id: index for index, node_id in enumerate(ast_ids)}\n    ddg_ids = [str(value) for value in ddg.get("ast_node_ids", ddg.get("node_ids", []))]\n    pairs = []\n    for raw_source, raw_target in zip(ddg.get("row", []), ddg.get("col", [])):\n        source_position, target_position = int(raw_source), int(raw_target)\n        if not (0 <= source_position < len(ddg_ids) and 0 <= target_position < len(ddg_ids)):\n            continue\n        source = ast_index.get(ddg_ids[source_position])\n        target = ast_index.get(ddg_ids[target_position])\n        if source is not None and target is not None:\n            pairs.extend(((source, target), (target, source)))\n    return np.asarray(sorted(set(pairs)), dtype=np.int32).reshape(-1, 2) if pairs else np.empty((0, 2), dtype=np.int32)\n\n\ndef canonical_graph_from_record(\n    record: dict,\n    *,\n    language: str,\n    max_nodes: int = 512,\n    lexical_buckets: int = DEFAULT_LEXICAL_BUCKETS,\n    lexical_slots: int = DEFAULT_LEXICAL_SLOTS,\n    source_code: str = "",\n    source_slots: int = 96,\n    strip_node_types: bool = False,\n    strip_topology: bool = False,\n) -> CanonicalGraph | None:\n    graphs = record.get("graphs", {})\n    ast_layer = graphs.get("ast", {})\n    ddg_layer = graphs.get("ddg", {})\n    ast = ast_layer.get("adjacency", ast_layer) if isinstance(ast_layer, dict) else {}\n    ddg = ddg_layer.get("adjacency", ddg_layer) if isinstance(ddg_layer, dict) else {}\n    raw_types = list(ast.get("node_types", []))\n    if not raw_types:\n        return None\n    node_count = min(len(raw_types), max_nodes)\n    raw_labels = list(ast.get("node_labels", []))\n    if len(raw_labels) < node_count:\n        raw_labels.extend([""] * (node_count - len(raw_labels)))\n    # Zero is reserved for padding in the embedding table. Canonical IDs are\n    # serialized as 1..N while the mapping API itself remains the requested\n    # stable zero-based 0..N-1 vocabulary.\n    raw_type_ids = [get_canonical_id(str(raw), language=language) for raw in raw_types[:node_count]]\n    canonical_ids = np.asarray([raw_id + 1 for raw_id in raw_type_ids], dtype=np.int64)\n    if strip_node_types:\n        # One shared id for every node: adjacency survives, categories do not.\n        canonical_ids = np.ones_like(canonical_ids)\n    lexical_ids = np.stack(\n        [\n            lexical_feature_ids(\n                lexical_source(raw_types[index], raw_labels[index], raw_type_ids[index], language=language),\n                bucket_count=lexical_buckets,\n                slots=lexical_slots,\n            )\n            for index in range(node_count)\n        ]\n    )\n    ast_edges = _edge_array(ast.get("row", []), ast.get("col", []), node_count)\n    ast_edges = np.asarray(sorted(set(map(tuple, np.vstack((ast_edges, ast_edges[:, ::-1]))))), dtype=np.int32) if len(ast_edges) else ast_edges\n    next_edges = np.asarray(\n        [(source, target) for index in range(node_count - 1) for source, target in ((index, index + 1), (index + 1, index))],\n        dtype=np.int32,\n    ).reshape(-1, 2)\n    ddg_edges = _project_ddg(ast, ddg, node_count)\n    if strip_topology:\n        # Input-only ablation: the encoder receives no observed graph edges.\n        empty_edges = np.empty((0, 2), dtype=np.int32)\n        ast_edges, next_edges, ddg_edges = empty_edges, empty_edges.copy(), empty_edges.copy()\n    eigenvalues = {\n        graph_type: np.asarray(layer.get("eigenvalues", []), dtype=np.float32)\n        for graph_type, layer in graphs.items()\n        if isinstance(layer, dict)\n    }\n    return CanonicalGraph(\n        code_id=str(record.get("code_id", record.get("id", ""))),\n        language=normalize_language(language),\n        canonical_ids=canonical_ids,\n        lexical_ids=lexical_ids,\n        source_lexical_ids=source_lexical_ids(source_code, bucket_count=lexical_buckets, slots=source_slots),\n        edges=(ast_edges, next_edges, ddg_edges),\n        eigenvalues=eigenvalues,\n    )\n\n\nclass CleanDataCorpus:\n    """Reader for ``spectral_clean_data_v1`` exports with language filters."""\n\n    def __init__(self, root: str | Path, *, default_language: str | None = None) -> None:\n        self.root = Path(root)\n        self.codes_path = _find_file(self.root, "codes.jsonl.gz", "codes.jsonl")\n        self.pairs_path = _find_file(self.root, "pairs.csv.gz", "pairs.csv")\n        self.graphs_path = _find_file(self.root, "graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")\n        self.default_language = normalize_language(default_language) if default_language else None\n        self.code_metadata = {}\n        self.ddg_alignment = {}\n        try:\n            alignment_path = _find_file(self.root, "ddg_ast_alignment.jsonl.gz", "ddg_ast_alignment.jsonl")\n        except FileNotFoundError:\n            alignment_path = None\n        if alignment_path is not None:\n            with _open_text(alignment_path) as stream:\n                for line in stream:\n                    item = json.loads(line)\n                    self.ddg_alignment[str(item["code_id"])] = item["ast_node_ids"]\n        self.code_languages, self.code_texts = self._load_code_metadata()\n\n    def _load_code_metadata(self) -> tuple[dict[str, str], dict[str, str]]:\n        languages, code_texts = {}, {}\n        with _open_text(self.codes_path) as stream:\n            for line in stream:\n                if not line.strip():\n                    continue\n                record = json.loads(line)\n                code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))\n                raw_language = record.get("language", record.get("lang", self.default_language))\n                if raw_language is None:\n                    raise ValueError(\n                        f"Code {code_id} in {self.codes_path} has no language; provide default_language explicitly."\n                    )\n                languages[code_id] = normalize_language(str(raw_language))\n                code_texts[code_id] = str(record.get("code", record.get("source", "")) or "")\n                self.code_metadata[code_id] = {key: record.get(key) for key in\n                                              ("problem_id", "source_code_id", "source_sha256")}\n        return languages, code_texts\n\n    def pairs(\n        self,\n        split: str,\n        *,\n        language_filter: PairLanguageFilter | None = None,\n        maximum: int | None = None,\n        seed: int = 42,\n    ) -> pd.DataFrame:\n        frame = pd.read_csv(\n            self.pairs_path,\n            compression="gzip" if _is_gzip_file(self.pairs_path) else None,\n            dtype={"left_id": str, "right_id": str, "label": np.int8},\n        )\n        frame = frame[frame["split"].astype(str).str.lower() == split.strip().lower()].copy()\n        frame["left_language"] = frame["left_id"].map(self.code_languages)\n        frame["right_language"] = frame["right_id"].map(self.code_languages)\n        for side in ("left", "right"):\n            frame[f"{side}_problem_id"] = frame[f"{side}_id"].map(\n                lambda value: self.code_metadata.get(str(value), {}).get("problem_id"))\n        if frame[["left_language", "right_language"]].isna().any().any():\n            raise RuntimeError("At least one pair endpoint is missing from codes.jsonl(.gz).")\n        if language_filter:\n            if language_filter.left:\n                frame = frame[frame["left_language"] == normalize_language(language_filter.left)]\n            if language_filter.right:\n                frame = frame[frame["right_language"] == normalize_language(language_filter.right)]\n        if maximum is not None and len(frame) > maximum:\n            pieces = []\n            for _, group in frame.groupby("label", sort=True):\n                size = max(1, round(maximum * len(group) / len(frame)))\n                pieces.append(group.sample(n=min(size, len(group)), random_state=seed))\n            frame = pd.concat(pieces).sample(frac=1, random_state=seed).head(maximum)\n        return frame.reset_index(drop=True)\n\n    def load_graphs(\n        self,\n        code_ids: Iterable[str],\n        *,\n        max_nodes: int = 512,\n        lexical_buckets: int = DEFAULT_LEXICAL_BUCKETS,\n        lexical_slots: int = DEFAULT_LEXICAL_SLOTS,\n        source_slots: int = 96,\n        strip_node_types: bool = False,\n        strip_topology: bool = False,\n    ) -> dict[str, CanonicalGraph]:\n        required = {str(value) for value in code_ids}\n        graphs = {}\n        with _open_text(self.graphs_path) as stream:\n            for line in stream:\n                record = json.loads(line)\n                code_id = str(record.get("code_id", record.get("id", "")))\n                if code_id not in required:\n                    continue\n                if code_id in self.ddg_alignment:\n                    record["graphs"]["ddg"]["adjacency"]["ast_node_ids"] = self.ddg_alignment[code_id]\n                graph = canonical_graph_from_record(\n                    record,\n                    language=self.code_languages[code_id],\n                    max_nodes=max_nodes,\n                    lexical_buckets=lexical_buckets,\n                    lexical_slots=lexical_slots,\n                    source_code=self.code_texts.get(code_id, ""),\n                    source_slots=source_slots,\n                    strip_node_types=strip_node_types,\n                    strip_topology=strip_topology,\n                )\n                if graph is not None:\n                    graphs[code_id] = graph\n        return graphs\n\n\ndef pair_code_ids(*frames: pd.DataFrame) -> set[str]:\n    values: set[str] = set()\n    for frame in frames:\n        values.update(frame["left_id"].astype(str))\n        values.update(frame["right_id"].astype(str))\n    return values\n', 'datasets.py': '"""Freeze audited common graph support without relabeling existing pairs.\n\nAtCoder problem IDs come from the local source metadata, never inferred from\nclone-pair connected components. BigCloneBench problem and clone-type labels\nare not fabricated when the source export does not supply them.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport gzip\nimport hashlib\nimport io\nimport json\nimport shutil\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .data import _find_file, _open_text\nfrom .prepare import pair_overlap\nfrom .protocol import fingerprint\n\n\ndef _component_assignments(codes, provided=None):\n    parent = {str(row["code_id"]): str(row["code_id"]) for row in codes}\n    def find(key):\n        while parent[key] != key:\n            parent[key] = parent[parent[key]]\n            key = parent[key]\n        return key\n    seen = {}\n    for row in codes:\n        keys = [("source", hashlib.sha256(row["code"].encode()).hexdigest())]\n        if row.get("source_code_id"):\n            keys.append(("submission", str(row["source_code_id"])))\n        if row.get("problem_id") is not None:\n            keys.append(("problem", str(row["problem_id"])))\n        for key in keys:\n            code_id = str(row["code_id"])\n            if key in seen:\n                a, b = find(code_id), find(seen[key])\n                parent[max(a, b)] = min(a, b)\n            else:\n                seen[key] = code_id\n    owners = {}\n    if provided is not None:\n        for row in codes:\n            owners.setdefault(find(str(row["code_id"])), set()).add(provided.get(str(row.get("problem_id"))))\n    assignments = {}\n    for key in parent:\n        component = find(key)\n        if provided is not None:\n            splits = owners[component]\n            assignments[key] = next(iter(splits)) if len(splits) == 1 and None not in splits else "excluded_conflicting_component"\n        else:\n            value = int(hashlib.sha256(f"endpoint-reference-v1/42/{component}".encode()).hexdigest()[:16], 16) / 2**64\n            assignments[key] = "train" if value < .7 else "valid" if value < .85 else "test"\n    return assignments\n\n\ndef _cap_preserving_prevalence(frame, cap, reference_prevalence, seed=42):\n    available = frame.label.value_counts().to_dict()\n    fraction = reference_prevalence\n    total = min(len(frame), cap) if cap else len(frame)\n    total = min(total, int(available.get(1, 0) / fraction), int(available.get(0, 0) / (1 - fraction)))\n    positives = min(available.get(1, 0), round(total * fraction))\n    negatives = min(available.get(0, 0), total - positives)\n    if not positives or not negatives:\n        raise ValueError("Filtering left too few pairs in one class")\n    return pd.concat((frame[frame.label.eq(1)].sample(positives, random_state=seed),\n                      frame[frame.label.eq(0)].sample(negatives, random_state=seed))).sample(frac=1, random_state=seed)\n\n\ndef prepare_bundle(source, output, dataset, *, source_metadata_zip=None,\n                   required_layers=("ast", "cfg", "ddg", "cpg"),\n                   caps=(70000, 15000, 15000)):\n    source, output = Path(source), Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError("Choose an empty destination; existing datasets are preserved")\n    output.mkdir(parents=True, exist_ok=True)\n    archive = zipfile.ZipFile(source) if source.is_file() else None\n    try:\n        for name in ("codes.jsonl.gz", "pairs.csv.gz", "graph_spectra.jsonl.gz", "metadata.json"):\n            if archive:\n                matches = [n for n in archive.namelist() if Path(n).name == name]\n                if len(matches) != 1:\n                    raise ValueError(f"Expected exactly one {name}")\n                with archive.open(matches[0]) as stream, (output / name).open("wb") as target:\n                    shutil.copyfileobj(stream, target)\n            else:\n                shutil.copyfile(_find_file(source, name), output / name)\n        original_metadata = json.loads((output / "metadata.json").read_text(encoding="utf-8"))\n        if not archive and original_metadata.get("ddg_ast_alignment"):\n            shutil.copyfile(_find_file(source, "ddg_ast_alignment.jsonl.gz"), output / "ddg_ast_alignment.jsonl.gz")\n            for name in ("ddg_alignment_audit.json", "coverage_by_language.json", "problem_splits.csv"):\n                if (source / name).exists():\n                    shutil.copyfile(source / name, output / name)\n        with _open_text(output / "codes.jsonl.gz") as stream:\n            codes = [json.loads(line) for line in stream]\n        for row in codes:\n            row["code_id"] = str(row["code_id"])\n        provided = None\n        if dataset == "atcoder":\n            if source_metadata_zip is None:\n                raise ValueError("AtCoder requires the original problem metadata archive")\n            with zipfile.ZipFile(source_metadata_zip) as metadata_archive:\n                metadata = pd.read_parquet(io.BytesIO(metadata_archive.read("ATCoder/function_metadata.parquet")),\n                                           columns=["function_id", "problem_id"])\n                splits = pd.read_parquet(io.BytesIO(metadata_archive.read("ATCoder/splits_v3_problem_disjoint/problem_split_assignments.parquet")),\n                                         columns=["problem_id", "split"])\n            lookup = dict(zip(metadata.function_id.astype(str), metadata.problem_id.astype(str)))\n            provided = dict(zip(splits.problem_id.astype(str), splits.split.replace({"validation": "valid"})))\n            for row in codes:\n                row["problem_id"] = lookup.get(str(row["source_code_id"]))\n            if any(row["problem_id"] is None for row in codes):\n                raise ValueError("Some AtCoder programs have no authoritative problem ID")\n        pairs = pd.read_csv(output / "pairs.csv.gz", dtype={"left_id": str, "right_id": str})\n        pairs["split"] = pairs.split.replace({"validation": "valid"})\n        original_counts = pairs.groupby(["split", "label"]).size().to_dict()\n        usable, coverage = set(), Counter()\n        wanted = {row["code_id"] for row in codes}\n        with _open_text(output / "graph_spectra.jsonl.gz") as stream:\n            for line in stream:\n                row = json.loads(line)\n                if str(row.get("code_id")) not in wanted:\n                    continue\n                layers = row.get("graphs", {})\n                good = True\n                for layer in required_layers:\n                    view = layers.get(layer, {})\n                    nodes = view.get("adjacency", view).get("node_types", [])\n                    coverage[f"{layer}_missing_or_empty_nodes"] += not bool(nodes)\n                    good &= bool(nodes)\n                if good:\n                    usable.add(str(row["code_id"]))\n        original_pairs = pairs.copy()\n        if dataset == "codenet":\n            # The existing CodeNet split was already problem/source disjoint.\n            if original_metadata.get("protocol") != "revision_problem_disjoint_v1":\n                raise ValueError("Start CodeNet common-support preparation from its audited revision directory")\n            membership = None\n        else:\n            membership = _component_assignments(codes, provided)\n            pairs = pairs[pairs.left_id.map(membership).eq(pairs.split) & pairs.right_id.map(membership).eq(pairs.split)]\n        pairs = pairs[pairs.left_id.isin(usable) & pairs.right_id.isin(usable)].copy()\n        pairs["_unordered_pair"] = ["/".join(sorted((str(a), str(b)))) for a, b in\n                                    pairs[["left_id", "right_id"]].itertuples(index=False, name=None)]\n        if pairs.groupby("_unordered_pair").label.nunique().gt(1).any():\n            raise ValueError("Conflicting labels for the same unordered code pair")\n        before_deduplication = len(pairs)\n        pairs = pairs.drop_duplicates(["split", "_unordered_pair"]).drop(columns="_unordered_pair")\n        duplicate_pairs_removed = before_deduplication - len(pairs)\n        filtered_counts = pairs.groupby(["split", "label"]).size().to_dict()\n        selected = []\n        for split, cap in zip(("train", "valid", "test"), caps):\n            reference = original_pairs[original_pairs.split.eq(split)]\n            subset = pairs[pairs.split.eq(split)]\n            selected.append(_cap_preserving_prevalence(subset, cap, float(reference.label.mean())))\n        pairs = pd.concat(selected, ignore_index=True)\n        pairs["configuration_id"] = pairs.get("configuration_id", pd.Series(index=pairs.index, dtype=str))\n        index = {row["code_id"]: row for row in codes}\n        missing = pairs.configuration_id.isna()\n        pairs.loc[missing, "configuration_id"] = [\n            a if a == b else "_".join(sorted((a, b))) for a, b in\n            ((index[left]["language"], index[right]["language"]) for left, right in\n             pairs.loc[missing, ["left_id", "right_id"]].itertuples(index=False, name=None))]\n        if "pair_id" not in pairs:\n            pairs["pair_id"] = [hashlib.sha256(f"{split}/{a}/{b}".encode()).hexdigest()[:24]\n                                for split, a, b in pairs[["split", "left_id", "right_id"]].itertuples(index=False, name=None)]\n        if pairs.pair_id.duplicated().any():\n            raise ValueError("Duplicate pair IDs in retained support")\n        problems_known = all(row.get("problem_id") is not None for row in codes)\n        audit_codes = codes if problems_known else [{**row, "problem_id": row["code_id"]} for row in codes]\n        overlap = pair_overlap(pairs, audit_codes)\n        if not problems_known:\n            for item in overlap.values():\n                item.pop("problem")  # endpoint identity is not a functionality label\n        if any(value for item in overlap.values() for value in item.values()):\n            raise ValueError(f"Disjointness audit failed: {overlap}")\n        pairs.to_csv(output / "pairs.csv.gz", index=False, compression="gzip")\n        with gzip.open(output / "codes.jsonl.gz", "wt", encoding="utf-8") as stream:\n            for row in codes:\n                stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\n        manifest = {"format": "spectral_clean_data_v1", "dataset_key": dataset,\n                    "dataset": f"{dataset}: audited common support; derived benchmark",\n                    "protocol": "revision_problem_disjoint_v1" if dataset == "codenet" else "revision_reference_audited_v1",\n                    "source_metadata": original_metadata, "required_graph_layers": required_layers,\n                    "overlap": overlap, "problem_disjoint": problems_known,\n                    "problem_id_status": "authoritative source metadata" if problems_known else "unavailable; not inferred",\n                    "user_disjoint": "unverified: user metadata unavailable",\n                    "clone_type_status": "unavailable unless authoritative annotations are supplied separately",\n                    "selection_seed": 42, "caps": caps,\n                    "pair_table_sha256": fingerprint(pairs.to_dict("records")),\n                    "counts": pairs.groupby(["split", "label"]).size().rename("pairs").reset_index().to_dict("records"),\n                    "graph_coverage": dict(coverage), "usable_graph_endpoints": len(usable),\n                    "duplicate_unordered_pairs_removed": duplicate_pairs_removed,\n                    "original_counts": {str(k): int(v) for k, v in original_counts.items()},\n                    "after_support_and_disjointness": {str(k): int(v) for k, v in filtered_counts.items()},\n                    "split_policy": "retain original pair labels and partition; remove endpoints/components incompatible with disjoint support",\n                    "prevalence_policy": "subsample labels to preserve each original split\'s prevalence; no label rebalance to 50/50 for BigCloneBench",\n                    "claim_boundary": "derived common-support endpoint/source-disjoint evaluation, not a replacement score on the unfiltered official release"}\n        if original_metadata.get("ddg_ast_alignment"):\n            manifest["ddg_ast_alignment"] = original_metadata["ddg_ast_alignment"]\n        (output / "metadata.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n        return manifest\n    finally:\n        if archive:\n            archive.close()\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--dataset", choices=("codenet", "atcoder", "bigclonebench"), required=True)\n    parser.add_argument("--source-metadata-zip", type=Path)\n    parser.add_argument("--required-layers", nargs="+", default=["ast", "cfg", "ddg", "cpg"])\n    args = parser.parse_args()\n    print(json.dumps(prepare_bundle(args.source, args.output, args.dataset,\n                                    source_metadata_zip=args.source_metadata_zip, required_layers=tuple(args.required_layers)), indent=2))\n', 'diagnostics.py': '"""Validation-only spectral distance, size/density and slot-profile diagnostics."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import roc_auc_score\n\nfrom .batching import pack_graphs\nfrom .model import RevisionModelConfig, RevisionPairModel\nfrom .protocol import RunConfig, fingerprint\nfrom .training import _load_checkpoint, load_pair_data, SessionBudgetReached\nfrom .versioning import training_implementation_hash\nfrom tqdm.auto import tqdm\n\n\ndef _fixed_features(eigenvalues):\n    values = np.asarray(eigenvalues, dtype=float)\n    if not len(values) or not np.isfinite(values).all():\n        return None\n    density = np.exp(-.5 * ((values[:, None] - np.linspace(0, 2, 32)) / .07) ** 2).mean(0)\n    density /= max(density.sum(), 1e-12)\n    heat = np.exp(-values[:, None] * np.logspace(-2, 2, 24)).mean(0)\n    return np.r_[density, heat]\n\n\ndef _distance(a, b):\n    a, b = np.asarray(a), np.asarray(b)\n    return float(np.linalg.norm(a / max(np.linalg.norm(a), 1e-12) - b / max(np.linalg.norm(b), 1e-12)))\n\n\n@torch.no_grad()\ndef validation_diagnostics(root, checkpoint, output, *, clock=None, maximum_pairs=4000, seed=9173):\n    state = _load_checkpoint(checkpoint, "cpu")\n    if state.get("implementation_hash") != training_implementation_hash():\n        raise ValueError("Diagnostic input/runtime differs from checkpoint; use its original notebook or an input-v2 checkpoint")\n    if state["model_config"].get("family") is not None or state["best_model"] is None:\n        raise ValueError("Use a validation-selected SPECTRA checkpoint")\n    corpus, frames, data_hash = load_pair_data(Path(root), require_disjoint=state["training"]["dataset"] == "codenet")\n    if data_hash != state["data_hash"]:\n        raise ValueError("Diagnostic bundle differs from checkpoint; use a main-experiment checkpoint, not a transfer job")\n    frame = frames["valid"]\n    if len(frame) > maximum_pairs:\n        # Uniform seeded sample independent of predictions; report all strata.\n        frame = frame.sample(maximum_pairs, random_state=seed).reset_index(drop=True)\n    ids = sorted(set(frame.left_id) | set(frame.right_id))\n    config = RunConfig(**state["training"])\n    graphs = corpus.load_graphs(ids, max_nodes=config.max_nodes)\n    if set(ids) - set(graphs):\n        raise ValueError("Validation endpoints are missing graphs")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = RevisionPairModel(RevisionModelConfig(**state["model_config"])).to(device).eval()\n    model.load_state_dict(state["best_model"])\n    model.encoder.capture_diagnostics = True\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    artifacts, profiles, profile_counts = {}, {}, {}\n    try:\n        for start in tqdm(range(0, len(ids), config.batch_size), desc="Validation descriptors",\n                          unit="batch", mininterval=2):\n            if clock:\n                clock.check()\n            keys = ids[start:start + config.batch_size]\n            batch = pack_graphs([graphs[key] for key in keys], config.max_nodes).to(device)\n            _, descriptor, auxiliary = model.encoder(batch, None)\n            assignments = auxiliary["assignment"].float()\n            onehot = torch.nn.functional.one_hot(batch.canonical_ids, model.config.canonical_vocab_size).float()\n            onehot = onehot * batch.mask[:, :, None]\n            masses = torch.einsum("bnk,bnc->bkc", assignments, onehot).cpu().numpy()\n            for i, key in enumerate(keys):\n                graph = graphs[key]\n                adjacency = auxiliary["adjacency"][i].float().cpu().numpy()\n                values = auxiliary["eigenvalues"][i].float().cpu().numpy()\n                vector = descriptor[i].float().cpu().numpy()\n                n = graph.node_count\n                observed_edges = {tuple(edge) for relation in (graph.edges[0], graph.edges[2]) for edge in relation if edge[0] != edge[1]}\n                artifacts[key] = {"descriptor": vector, "eigenvalues": values,\n                                  "nodes": n, "observed_density": len(observed_edges) / max(1, n * (n - 1)),\n                                  "latent_density": float(adjacency.sum() / (len(adjacency) * (len(adjacency) - 1)))}\n                language = graph.language\n                normalized = masses[i] / masses[i].sum(-1, keepdims=True).clip(1e-12)\n                profiles[language] = profiles.get(language, np.zeros_like(normalized)) + normalized\n                profile_counts[language] = profile_counts.get(language, 0) + 1\n        rows = []\n        for row in tqdm(frame.to_dict("records"), desc="Pair distances", unit="pair", mininterval=2):\n            a, b = str(row["left_id"]), str(row["right_id"])\n            base = {key: row.get(key) for key in ("pair_id", "label", "configuration_id", "left_problem_id", "right_problem_id")}\n            base.update(mean_nodes=(artifacts[a]["nodes"] + artifacts[b]["nodes"]) / 2,\n                        mean_observed_density=(artifacts[a]["observed_density"] + artifacts[b]["observed_density"]) / 2,\n                        mean_latent_density=(artifacts[a]["latent_density"] + artifacts[b]["latent_density"]) / 2)\n            rows.append({**base, "view": "learned_eigenvalue_density_heat", "distance": _distance(\n                _fixed_features(artifacts[a]["eigenvalues"]), _fixed_features(artifacts[b]["eigenvalues"]))})\n            rows.append({**base, "view": "learned_full_descriptor", "distance": _distance(artifacts[a]["descriptor"], artifacts[b]["descriptor"])})\n            for layer in ("ast", "cfg", "ddg", "cpg"):\n                left = _fixed_features(graphs[a].eigenvalues.get(layer, []))\n                right = _fixed_features(graphs[b].eigenvalues.get(layer, []))\n                if left is not None and right is not None:\n                    rows.append({**base, "view": "fixed_" + layer, "distance": _distance(left, right)})\n        distances = pd.DataFrame(rows)\n        # Quantile boundaries use validation covariates only. The same strata\n        # apply to every view. Match label counts within each stratum.\n        covariates = distances.drop_duplicates("pair_id")\n        size_edges = np.unique(np.quantile(covariates.mean_nodes, [0, .25, .5, .75, 1]))\n        density_edges = np.unique(np.quantile(covariates.mean_observed_density, [0, .25, .5, .75, 1]))\n        distances["size_bin"] = np.searchsorted(size_edges[1:-1], distances.mean_nodes)\n        distances["density_bin"] = np.searchsorted(density_edges[1:-1], distances.mean_observed_density)\n        summaries = []\n        for view, group in distances.groupby("view"):\n            matched = []\n            for _, cell in group.groupby(["configuration_id", "size_bin", "density_bin"]):\n                count = min(int(cell.label.eq(0).sum()), int(cell.label.eq(1).sum()))\n                if count:\n                    matched += [cell[cell.label.eq(label)].sample(count, random_state=seed) for label in (0, 1)]\n            for selection, subset in (("all_available", group), ("size_density_label_matched", pd.concat(matched) if matched else group.iloc[:0])):\n                if subset.empty or subset.label.nunique() != 2:\n                    continue\n                summaries.append({"view": view, "selection": selection, "pairs": len(subset),\n                                  "distance_AUROC": float(roc_auc_score(subset.label, -subset.distance)),\n                                  "clone_mean_distance": float(subset[subset.label.eq(1)].distance.mean()),\n                                  "nonclone_mean_distance": float(subset[subset.label.eq(0)].distance.mean())})\n        distances.to_csv(output / "validation_pair_distances.csv.gz", index=False, compression="gzip")\n        pd.DataFrame(summaries).to_csv(output / "validation_distance_summary.csv", index=False)\n        profiles = {language: values / profile_counts[language] for language, values in profiles.items()}\n        profile_rows = []\n        for i, a in enumerate(sorted(profiles)):\n            for b in sorted(profiles)[i + 1:]:\n                # Slot identities are shared within this checkpoint only.\n                profile_rows.append({"language_a": a, "language_b": b,\n                                     "mean_slot_profile_L1": float(np.abs(profiles[a] - profiles[b]).sum(-1).mean())})\n        pd.DataFrame(profile_rows).to_csv(output / "cross_language_slot_profiles.csv", index=False)\n        np.savez_compressed(output / "slot_profiles.npz", **profiles)\n        np.savez_compressed(output / "validation_code_spectra.npz", code_ids=np.asarray(ids),\n                            eigenvalues=np.stack([artifacts[key]["eigenvalues"] for key in ids]),\n                            descriptors=np.stack([artifacts[key]["descriptor"] for key in ids]))\n        result = {"status": "complete", "split": "valid", "checkpoint_identity": state["identity"],\n                  "validation_pair_sample_hash": fingerprint(frame.pair_id.astype(str).tolist()),\n                  "pairs": len(frame), "codes": len(ids), "selection_seed": seed,\n                  "size_quantiles": size_edges.tolist(), "density_quantiles": density_edges.tolist(),\n                  "distance_definition": "L2 between individually L2-normalized descriptors",\n                  "limitations": ["Diagnostics, not a theorem of language invariance", "Fixed views can have different available support; counts are reported",\n                                  "Slot profiles are marginal language distributions, not aligned semantics or causal explanations",\n                                  "Slots must not be compared across checkpoints without permutation alignment", "Truncated node counts, not original program sizes"]}\n    except SessionBudgetReached:\n        result = {"status": "incomplete_session_budget", "split": "valid", "codes_processed": len(artifacts),\n                  "continuation": "Rerun this bounded diagnostic job; training checkpoint remains unchanged"}\n    (output / "diagnostic_status.json").write_text(json.dumps(result, indent=2), encoding="utf-8")\n    return result\n', 'embeddings.py': '"""A modern frozen Qwen3 embedding baseline with a train-only linear probe.\n\nModel loading/pooling follows the publisher\'s model card:\nhttps://huggingface.co/Qwen/Qwen3-Embedding-0.6B\nThis is explicitly a frozen representation baseline, not ZC3 or a fine-tuned LLM.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport pickle\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.special import expit\nfrom sklearn.linear_model import SGDClassifier\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .protocol import choose_threshold, fingerprint, metrics\nfrom .io_utils import replace_file\nfrom .training import SessionBudgetReached\n\n\ndef _json_save(value, path):\n    temporary = path.with_suffix(".tmp")\n    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")\n    replace_file(temporary, path)\n\n\ndef extract_embeddings(codes: dict[str, str], ids: list[str], output: Path, *, clock,\n                       resume: Path | None = None, max_length: int = 1024, batch_size: int = 8,\n                       model_id: str = "Qwen/Qwen3-Embedding-0.6B", revision: str | None = None):\n    output.mkdir(parents=True, exist_ok=True)\n    code_hashes = [(key, hashlib.sha256(codes[key].encode()).hexdigest()) for key in ids]\n    identity = fingerprint({"codes": code_hashes, "model": model_id, "requested_revision": revision,\n                            "max_length": max_length, "pooling": "last_token", "prompt": "none",\n                            "source_version": 1,\n                            "implementation_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})\n    manifest_path = output / "embeddings.json"\n    matrix_path = output / "embeddings.npy"\n    if resume and not manifest_path.exists():\n        for name in ("embeddings.json", "embeddings.npy", "embedding_ids.json"):\n            shutil.copyfile(resume / name, output / name)\n    state = json.loads(manifest_path.read_text()) if manifest_path.exists() else None\n    if state is not None:\n        if state["identity"] != identity:\n            raise ValueError("Embedding resume mismatch: code pool/model/token limit changed")\n        if state["complete"]:\n            return state\n    clock.check()\n    from transformers import AutoModel, AutoTokenizer\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    resolved = state["resolved_revision"] if state else revision\n    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=resolved, padding_side="left", trust_remote_code=False)\n    model = AutoModel.from_pretrained(model_id, revision=resolved, trust_remote_code=False,\n                                      torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()\n    model.requires_grad_(False)\n    dimension = model.config.hidden_size\n    if state is None:\n        state = {"identity": identity, "model": model_id, "resolved_revision": model.config._commit_hash,\n                 "next_index": 0, "complete": False, "dimension": dimension,\n                 "parameters": sum(p.numel() for p in model.parameters()),\n                 "max_length": max_length, "batch_size": batch_size, "truncated_codes": 0,\n                 "encoding_seconds": 0.0, "codes": len(ids), "encoder_training": "frozen",\n                 "pooling": "last token; left padding; L2 normalized; no task prompt"}\n        np.lib.format.open_memmap(matrix_path, mode="w+", dtype=np.float32, shape=(len(ids), dimension)).flush()\n        (output / "embedding_ids.json").write_text(json.dumps(ids), encoding="utf-8")\n        _json_save(state, manifest_path)\n    matrix = np.load(matrix_path, mmap_mode="r+")\n    try:\n        while state["next_index"] < len(ids):\n            clock.check()\n            start = state["next_index"]\n            stop = min(len(ids), start + state["batch_size"])\n            texts = [codes[key] for key in ids[start:stop]]\n            started = time.monotonic()\n            lengths = [len(row) for row in tokenizer(texts, truncation=False)["input_ids"]]\n            tokens = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)\n            try:\n                with torch.inference_mode():\n                    hidden = model(**tokens).last_hidden_state\n                    vectors = torch.nn.functional.normalize(hidden[:, -1].float(), dim=-1).cpu().numpy()\n            except torch.cuda.OutOfMemoryError:\n                del tokens\n                torch.cuda.empty_cache()\n                if state["batch_size"] <= 1:\n                    raise\n                state["batch_size"] = max(1, state["batch_size"] // 2)\n                _json_save(state, manifest_path)\n                continue\n            if not np.isfinite(vectors).all():\n                raise FloatingPointError("Non-finite pretrained embeddings")\n            matrix[start:stop] = vectors\n            matrix.flush()\n            state["next_index"] = stop\n            state["truncated_codes"] += sum(n > max_length for n in lengths)\n            state["encoding_seconds"] += time.monotonic() - started\n            state["complete"] = stop == len(ids)\n            _json_save(state, manifest_path)\n            print(f"Embedded {stop}/{len(ids)}", flush=True)\n    except SessionBudgetReached:\n        _json_save(state, manifest_path)\n    return state\n\n\ndef pair_vectors(matrix, left, right):\n    a, b = np.asarray(matrix[left]), np.asarray(matrix[right])\n    return np.concatenate((np.abs(a - b), a * b, (a * b).sum(-1, keepdims=True),\n                           np.linalg.norm(a - b, axis=-1, keepdims=True)), axis=-1)\n\n\ndef fit_probe(frames: dict, embeddings: Path, output: Path, *, clock, seed: int = 42,\n              primary: str = "Accuracy", resume: Path | None = None, final_test: bool = False):\n    output.mkdir(parents=True, exist_ok=True)\n    manifest = json.loads((embeddings / "embeddings.json").read_text())\n    if not manifest["complete"]:\n        raise ValueError("Complete embedding extraction before fitting the supervised probe")\n    ids = json.loads((embeddings / "embedding_ids.json").read_text())\n    index = {key: i for i, key in enumerate(ids)}\n    arrays = {name: (f.left_id.map(index).to_numpy(), f.right_id.map(index).to_numpy(), f.label.to_numpy())\n              for name, f in frames.items()}\n    if any(pd.isna(a).any() or pd.isna(b).any() for a, b, _ in arrays.values()):\n        raise ValueError("An evaluation endpoint has no pretrained embedding")\n    matrix = np.load(embeddings / "embeddings.npy", mmap_mode="r")\n    identity = fingerprint({"embedding": manifest["identity"], "seed": seed, "metric": primary,\n                            "pairs": {k: fingerprint(v.to_csv(index=False)) for k, v in frames.items()},\n                            "implementation_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                            "learner": "train-scaled SGD logistic alpha1e-4, 100 epochs, patience10, min10"})\n    path = output / "probe.pkl"\n    if resume and not path.exists():\n        shutil.copyfile(resume / "probe.pkl", path)\n    if path.exists():\n        with path.open("rb") as stream:\n            state = pickle.load(stream)  # trusted checkpoint from this experiment only\n        if state["identity"] != identity:\n            raise ValueError("Linear probe resume identity mismatch")\n    else:\n        state = {"identity": identity, "stage": "scale", "offset": 0, "epoch": 1,\n                 "scaler": StandardScaler(), "model": SGDClassifier(loss="log_loss", alpha=1e-4,\n                                                                       random_state=seed, shuffle=False),\n                 "history": [], "best": None, "bad_epochs": 0, "complete": False}\n\n    def save():\n        temporary = path.with_suffix(".tmp")\n        with temporary.open("wb") as stream:\n            pickle.dump(state, stream)\n        replace_file(temporary, path)\n\n    def scores(name):\n        a, b, _ = arrays[name]\n        return np.concatenate([expit(state["model"].decision_function(state["scaler"].transform(pair_vectors(matrix, a[i:i+1024], b[i:i+1024]))))\n                               for i in range(0, len(a), 1024)])\n\n    a, b, y = arrays["train"]\n    try:\n        while state["stage"] == "scale":\n            clock.check()\n            i = state["offset"]\n            state["scaler"].partial_fit(pair_vectors(matrix, a[i:i+1024], b[i:i+1024]))\n            state["offset"] += 1024\n            if state["offset"] >= len(a):\n                state.update(stage="train", offset=0)\n            save()\n        while not state["complete"]:\n            order = np.random.default_rng(seed + state["epoch"]).permutation(len(a))\n            for i in range(state["offset"], len(a), 1024):\n                clock.check()\n                rows = order[i:i+1024]\n                features = state["scaler"].transform(pair_vectors(matrix, a[rows], b[rows]))\n                state["model"].partial_fit(features, y[rows], classes=np.array([0, 1]))\n                state["offset"] = i + 1024\n                save()\n            clock.check()\n            threshold, valid = choose_threshold(arrays["valid"][2], scores("valid"), primary)\n            state["history"].append({"epoch": state["epoch"], "valid": valid})\n            if state["best"] is None or valid[primary] > state["best"]["valid"][primary]:\n                state["best"] = {"valid": valid, "threshold": threshold,\n                                 "model": pickle.dumps(state["model"]), "epoch": state["epoch"]}\n                state["bad_epochs"] = 0\n            else:\n                state["bad_epochs"] += 1\n            state["epoch"] += 1; state["offset"] = 0\n            state["complete"] = state["epoch"] > 100 or (state["epoch"] > 10 and state["bad_epochs"] >= 10)\n            save()\n    except SessionBudgetReached:\n        save()\n    summary = {"identity": identity, "complete": state["complete"], "history": state["history"],\n               "encoder_training": "frozen", "head": "train-only standardized logistic SGD"}\n    if final_test and state["complete"]:\n        result_file = output / "final_test.json"\n        if result_file.exists():\n            return json.loads(result_file.read_text())\n        clock.check()\n        state["model"] = pickle.loads(state["best"]["model"])\n        p = scores("test")\n        cosine_valid = np.sum(matrix[arrays["valid"][0]] * matrix[arrays["valid"][1]], axis=-1)\n        cosine_test = np.sum(matrix[arrays["test"][0]] * matrix[arrays["test"][1]], axis=-1)\n        threshold, _ = choose_threshold(arrays["valid"][2], cosine_valid, primary)\n        summary.update(seed=seed, model=manifest["model"], resolved_revision=manifest["resolved_revision"],\n                       probe_test=metrics(arrays["test"][2], p, state["best"]["threshold"]),\n                       cosine_test=metrics(arrays["test"][2], cosine_test, threshold))\n        predictions = frames["test"].copy()\n        predictions["probability"] = p\n        predictions.to_csv(output / "test_predictions.csv.gz", index=False, compression="gzip")\n        _json_save(summary, result_file)\n    _json_save(summary, output / "probe_status.json")\n    return summary\n', 'io_utils.py': '"""Atomic replacement with a bounded retry for transient Windows file locks."""\nfrom __future__ import annotations\n\nimport os\nimport time\nfrom pathlib import Path\n\n\ndef replace_file(source: Path, destination: Path):\n    for attempt in range(6):\n        try:\n            os.replace(source, destination)\n            return\n        except PermissionError:\n            if attempt == 5:\n                raise\n            time.sleep(.025 * 2**attempt)\n', 'jobs.py': '"""Notebook entry points for revised baseline and matched-transfer jobs."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport torch\n\nfrom .protocol import RunConfig, fingerprint\nfrom .training import SessionClock, evaluate_final, load_pair_data, train_session\nfrom .transfer import select_transfer_frames\n\n\ndef find_bundle(explicit=None, dataset="codenet"):\n    if explicit is not None:\n        return Path(explicit)\n    candidates = []\n    for path in Path("/kaggle/input").rglob("metadata.json"):\n        try:\n            metadata = json.loads(path.read_text(encoding="utf-8"))\n            if metadata.get("protocol") in {"revision_problem_disjoint_v1", "revision_reference_audited_v1"}:\n                if dataset in metadata.get("dataset_key", "codenet"):\n                    candidates.append(path.parent)\n        except (ValueError, OSError):\n            pass\n    if len(candidates) != 1:\n        raise ValueError(f"Attach exactly one {dataset} revision bundle, or set CLEAN_DATA_DIR: {candidates}")\n    return candidates[0]\n\n\ndef run_external(*, method, seed, started, clean_data=None, session_hours=10.0,\n                 max_epochs=100, min_epochs=10, patience=10, batch_size=8,\n                 effective_batch_size=128, learning_rate=None, transfer_key=None,\n                 matched_pairs=6000, resume_checkpoint=None, resume_tokens=None,\n                 final_test=False, output_root=Path("/kaggle/working"),\n                 dataset="codenet", model_revision=None, measure_cost=True,\n                 allow_numerical_upgrade=False):\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU for these full training jobs")\n    is_text = method in {"codebert", "c4_adapted", "qwen3_lora"}\n    if learning_rate is None:\n        learning_rate = (5e-5 if method == "c4_adapted" else 2e-5) if is_text else {"astnn": 1e-3, "cdlh": 1e-3}.get(method, 5e-4)\n    if transfer_key:\n        max_epochs, min_epochs, patience = 30, 30, 1000\n    config = RunConfig(dataset=dataset, seed=seed, selection_metric="F1" if dataset == "bigclonebench" else "Accuracy",\n                       max_epochs=max_epochs, min_epochs=min_epochs, patience=patience,\n                       batch_size=batch_size, effective_batch_size=effective_batch_size,\n                       weight_decay=0.0 if method in {"astnn", "cdlh"} else 1e-3 if method == "deepsim" else 1e-4,\n                       learning_rate=learning_rate, session_hours=session_hours)\n    clock = SessionClock(config, started=started)\n    root = find_bundle(clean_data, dataset)\n    corpus, frames, data_hash = load_pair_data(root, require_disjoint=dataset == "codenet")\n    group_hash = data_hash\n    label = f"baseline/{method}"\n    suffix = ""\n    if transfer_key:\n        frames, transfer_metadata = select_transfer_frames(frames, transfer_key, matched_pairs, seed)\n        data_hash = fingerprint({"bundle": data_hash, "transfer": transfer_metadata})\n        label += "/" + transfer_key\n        suffix = "_" + transfer_key.replace("|", "_").replace("->", "_")\n    output = Path(output_root) / f"revision_{dataset}_{method}{suffix}_seed{seed}"\n    output.mkdir(parents=True, exist_ok=True)\n    if transfer_key:\n        (output / "transfer_metadata.json").write_text(json.dumps(transfer_metadata, indent=2), encoding="utf-8")\n    if is_text:\n        from .transformer import TransformerConfig, TransformerPairModel, prepare_tokens\n        settings = TransformerConfig(revision=model_revision)\n        if method == "c4_adapted":\n            settings = TransformerConfig(revision=model_revision, objective="c4_adapted")\n        elif method == "qwen3_lora":\n            settings = TransformerConfig(model_id="Qwen/Qwen3-Embedding-0.6B", revision=model_revision,\n                                         pooling="last_token", lora_rank=16, objective="pair_bce_contrastive")\n        packed, model_config, token_status = prepare_tokens(corpus, frames, settings, output / "tokens",\n                                                            clock=clock, resume=resume_tokens)\n        if packed is None:\n            return {"complete": False, "status": "incomplete_tokenization", "token_status": token_status,\n                    "continuation": "Attach output; set RESUME_TOKENS_DIR to its tokens directory"}\n        factory, config_factory = TransformerPairModel, TransformerConfig\n    else:\n        from .legacy import LegacyConfig, prepare_legacy\n        packed, model_config, factory = prepare_legacy(root, frames, method, config, output)\n        config_factory = LegacyConfig\n    summary = train_session(frames, packed, config, model_config, {}, output, data_hash=data_hash,\n                            data_group_hash=group_hash, model_factory=factory, clock=clock,\n                            resume=Path(resume_checkpoint) if resume_checkpoint else None, experiment_label=label,\n                            allow_numerical_upgrade=allow_numerical_upgrade)\n    if measure_cost and summary["complete"]:\n        from .benchmark import benchmark_checkpoint\n        benchmark_checkpoint(output / "progress.pt", frames["train"], packed, config, output / "benchmark.json",\n                             model_factory=factory, model_config_factory=config_factory, clock=clock)\n    if final_test and summary["complete"]:\n        return evaluate_final(output / "progress.pt", frames, packed, config, output,\n                              expected_data_hash=data_hash, clock=clock, model_factory=factory,\n                              model_config_factory=config_factory)\n    return summary\n', 'legacy.py': '"""Existing graph-baseline adaptations under the revision training protocol.\n\nThe encoder implementations are reused, not renamed SPECTRA encoders. Exported\nnode vocabularies are learned from training endpoints only. Differences from\nthe original papers remain documented in faithful_graph_baselines/core.py.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport types\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .data import CleanDataCorpus, _open_text, pair_code_ids\n\n\n@dataclass(frozen=True)\nclass LegacyConfig:\n    method: str\n    vocab_size: int\n    support_code_hash: str\n    family: str = "legacy_graph_adaptation"\n\n\ndef runtime(max_nodes=256):\n    """Isolate each experiment\'s packing settings from module globals."""\n    from spectral_code import faithful_graph_baselines\n    root = Path(faithful_graph_baselines.__file__).parent\n    source = (root / "notebook_runtime.py").read_text(encoding="utf-8")\n    module = types.ModuleType("revision_legacy_runtime")\n    module.__dict__.update(MAX_NODES=max_nodes, MAX_EDGES=512, MAX_STATEMENTS=64)\n    exec(compile(source, str(root / "notebook_runtime.py"), "exec"), module.__dict__)\n    digest = hashlib.sha256(((root / "core.py").read_text(encoding="utf-8") + source).encode()).hexdigest()\n    return module, digest\n\n\nclass _Rows(Dataset):\n    def __init__(self, frame, index):\n        self.rows = [(index[str(a)], index[str(b)], float(y))\n                     for a, b, y in frame[["left_id", "right_id", "label"]].itertuples(index=False, name=None)]\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, i):\n        return self.rows[i]\n\n\nclass LegacyCorpus:\n    def __init__(self, arrays, index):\n        self.arrays, self.index = arrays, index\n\n    def pair_loader(self, frame, config, batches=None):\n        kwargs = {"batch_sampler": batches} if batches is not None else {"batch_size": config.batch_size}\n        return DataLoader(_Rows(frame, self.index), num_workers=0,\n                          generator=torch.Generator().manual_seed(config.seed + 9187), **kwargs)\n\n\nclass LegacyPairModel(torch.nn.Module):\n    def __init__(self, config, corpus, module):\n        super().__init__()\n        self.config = config\n        self.network = module._PairModel(config.method, corpus.arrays, config.vocab_size)\n\n    def forward(self, left, right, temperature=None):\n        logits, reconstruction, a, b = self.network(left.long(), right.long())\n        return logits, {"reconstruction": reconstruction, "left": a, "right": b}\n\n    def compute_loss(self, logits, auxiliary, labels, *, positive_class_weight=1.0, **unused):\n        classification = F.binary_cross_entropy_with_logits(\n            logits.float(), labels.float(), pos_weight=logits.new_tensor(positive_class_weight))\n        extra = logits.new_zeros(())\n        if self.config.method == "rtvnn":\n            extra = .05 * auxiliary["reconstruction"]\n        elif self.config.method == "cdlh":\n            extra = .01 * sum((auxiliary[side].abs() - 1).square().mean() for side in ("left", "right"))\n        return classification + extra, {"classification": classification.detach(), "auxiliary": extra.detach()}\n\n\ndef prepare_legacy(root, frames, method, config, work_dir):\n    """Stream exports into compact arrays; fail rather than change test support."""\n    if method not in {"astnn", "rtvnn", "cdlh", "deepsim", "fa_ast_ggnn", "fa_ast_gmn"}:\n        raise ValueError(f"Unsupported baseline: {method}")\n    module, digest = runtime(config.max_nodes)\n    corpus = CleanDataCorpus(root)\n    ids = sorted(pair_code_ids(*frames.values()))\n    train_ids = pair_code_ids(frames["train"])\n    if method in {"astnn", "rtvnn", "cdlh"}:\n        arrays, vocab, usable, index = module._load_tree_arrays_streaming(corpus.graphs_path, ids, train_ids)\n    elif method.startswith("fa_ast"):\n        arrays, vocab, usable, index = module._load_fa_arrays_streaming(\n            corpus.graphs_path, ids, train_ids, ("ast", "cfg", "ddg", "cpg"))\n    else:\n        # Two passes avoid holding decoded JSON graph dictionaries in host RAM.\n        frequency = {}\n        with _open_text(corpus.graphs_path) as stream:\n            for line in stream:\n                row = json.loads(line)\n                if str(row.get("code_id")) in train_ids:\n                    adjacency = module._adjacency(row.get("graphs", {}).get("cfg"))\n                    for node in adjacency.get("node_types", [])[:config.max_nodes]:\n                        key = str(node)\n                        frequency[key] = frequency.get(key, 0) + 1\n        vocab = {"<pad>": 0, "<unk>": 1}\n        vocab.update({key: i + 2 for i, (key, _) in enumerate(sorted(frequency.items(), key=lambda x: (-x[1], x[0])))})\n        index, usable = {key: i for i, key in enumerate(ids)}, set()\n        arrays = {"node_types": np.zeros((len(ids), config.max_nodes), np.int32),\n                  "numeric": np.zeros((len(ids), config.max_nodes, 12), np.float32),\n                  "edge_src": np.full((len(ids), module.MAX_EDGES), -1, np.int32),\n                  "edge_dst": np.full((len(ids), module.MAX_EDGES), -1, np.int32)}\n        with _open_text(corpus.graphs_path) as stream:\n            for line in stream:\n                row = json.loads(line)\n                key = str(row.get("code_id"))\n                if key not in index:\n                    continue\n                layers = {name: module._adjacency(info) for name, info in row.get("graphs", {}).items()}\n                if not layers.get("cfg", {}).get("node_types"):\n                    continue\n                values = module._deepsim_arrays(layers, vocab)\n                for name, value in zip(arrays, values):\n                    arrays[name][index[key]] = value\n                usable.add(key)\n    missing = sorted(set(ids) - usable)\n    work_dir = Path(work_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    support = {"method": method, "required_endpoints": len(ids), "usable_endpoints": len(usable),\n               "missing_ids": missing, "vocabulary_training_endpoints_only": True,\n               "max_nodes": config.max_nodes, "max_edges_per_relation": module.MAX_EDGES,\n               "interpretation": "portable adaptation of existing project baseline; not original-paper reproduction"}\n    (work_dir / "baseline_support.json").write_text(json.dumps(support, indent=2), encoding="utf-8")\n    if missing:\n        raise ValueError(f"{method}: {len(missing)} endpoints missing required graph views. Freeze common support for ALL methods first; see baseline_support.json")\n    packed = LegacyCorpus(arrays, index)\n    model_config = LegacyConfig(method, len(vocab), digest)\n    def factory(settings):\n        if settings.support_code_hash != digest:\n            raise ValueError("Baseline encoder source changed")\n        return LegacyPairModel(settings, packed, module)\n    return packed, model_config, factory\n', 'model.py': '"""Canonical SPECTRA-Siam model for mono- and zero-shot cross-language runs.\n\nThe encoder consumes only stable canonical category IDs plus a bounded hashed\nlexical sketch. Parser-specific raw node strings never index an embedding\ntable. Input topology is retained as a soft latent-graph prior.\n"""\n\nfrom __future__ import annotations\n\nfrom .canonical_types import NUM_CANONICAL_TYPES\nADAPTIVE_TEMPERATURE_MIN = 0.20\nADAPTIVE_TEMPERATURE_MAX = 1.20\nUSE_ADAPTIVE_TEMPERATURE = True\nUSE_SPECTRAL_ATTENTION = False\nSPECTRAL_VARIANCE_FLOOR = 0.03\nTEMPERATURE_END = 0.30\n\nimport math\nfrom dataclasses import dataclass, replace\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n\n\n@dataclass(frozen=True)\nclass RevisionModelConfig:\n    canonical_vocab_size: int = NUM_CANONICAL_TYPES + 1  # zero is padding\n    lexical_buckets: int = 4096\n    hidden_dim: int = 256\n    max_input_nodes: int = 256\n    latent_nodes: int = 32\n    slot_iterations: int = 3\n    attention_heads: int = 4\n    lexical_dropout: float = 0.30\n    use_node_lexical: bool = True\n    use_source_lexical: bool = False\n    readout_mode: str = "graph_signal_spectral"\n    adjacency_mode: str = "learned"\n    pooling: str = "spectral"\n    target_density: float = 0.15\n    chebyshev_degree: int = 12\n    density_bins: int = 32\n    heat_samples: int = 24\n    spectral_bands: int = 12\n    spectral_signals: int = 8\n    dropout: float = 0.10\n    relation_indices: tuple[int, ...] = (0, 2)  # AST plus DDG projected onto the AST backbone\n    prior_relation_indices: tuple[int, ...] | None = None  # keep encoder inputs fixed in adjacency-view controls\n\n    @property\n    def descriptor_dim(self) -> int:\n        return self.density_bins + self.heat_samples + self.spectral_bands * self.spectral_signals\n\n    @property\n    def eigenvalue_descriptor_dim(self) -> int:\n        return self.density_bins + self.heat_samples\n\n    @property\n    def readout_descriptor_dim(self) -> int:\n        if self.pooling == "energy_only":\n            return self.spectral_bands * self.spectral_signals\n        if self.readout_mode == "eigenvalue_only":\n            return self.eigenvalue_descriptor_dim\n        return self.descriptor_dim\n\n\nclass RelationGraphLayer(nn.Module):\n    def __init__(self, dim: int, relations: int = 3, dropout: float = 0.10) -> None:\n        super().__init__()\n        self.self_linear = nn.Linear(dim, dim)\n        self.relation_linears = nn.ModuleList(nn.Linear(dim, dim, bias=False) for _ in range(relations))\n        self.norm = nn.LayerNorm(dim)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, states: torch.Tensor, edges: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        message = self.self_linear(states)\n        for relation, linear in enumerate(self.relation_linears):\n            adjacency = edges[:, relation]\n            degree = adjacency.sum(-1, keepdim=True).clamp_min(1.0)\n            message = message + torch.bmm(adjacency / degree, linear(states))\n        return self.norm(states + self.dropout(F.gelu(message))) * mask.unsqueeze(-1)\n\n\nclass SlotAttentionInduction(nn.Module):\n    def __init__(self, dim: int, slots: int, iterations: int) -> None:\n        super().__init__()\n        self.slots = slots\n        self.iterations = iterations\n        self.scale = dim**-0.5\n        self.slot_seed = nn.Parameter(torch.randn(1, slots, dim) * 0.02)\n        self.input_norm = nn.LayerNorm(dim)\n        self.slot_norm = nn.LayerNorm(dim)\n        self.to_query = nn.Linear(dim, dim, bias=False)\n        self.to_key = nn.Linear(dim, dim, bias=False)\n        self.to_value = nn.Linear(dim, dim, bias=False)\n        self.update = nn.GRUCell(dim, dim)\n        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))\n\n    def forward(self, states: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        normalized = self.input_norm(states)\n        key, value = self.to_key(normalized), self.to_value(normalized)\n        slots = self.slot_seed.expand(states.size(0), -1, -1)\n        for _ in range(self.iterations):\n            scores = torch.einsum("bsd,bnd->bns", self.to_query(self.slot_norm(slots)), key) * self.scale\n            scores = scores.masked_fill(~mask.unsqueeze(-1), -1e4)\n            assignments = torch.softmax(scores, dim=-1) * mask.unsqueeze(-1)\n            weights = assignments / assignments.sum(1, keepdim=True).clamp_min(1e-6)\n            updates = torch.einsum("bns,bnd->bsd", weights, value)\n            slots = self.update(updates.reshape(-1, updates.size(-1)), slots.reshape(-1, slots.size(-1))).view_as(slots)\n            slots = slots + self.mlp(slots)\n        return slots, assignments\n\n\nclass LatentGraphRefinement(nn.Module):\n    def __init__(self, dim: int, dropout: float) -> None:\n        super().__init__()\n        self.message = nn.Linear(dim, dim, bias=False)\n        self.self_linear = nn.Linear(dim, dim)\n        self.norm = nn.LayerNorm(dim)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, states: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:\n        degree = adjacency.sum(-1, keepdim=True).clamp_min(1e-6)\n        message = torch.bmm(adjacency / degree, self.message(states))\n        return self.norm(states + self.dropout(F.gelu(self.self_linear(states) + message)))\n\n\nclass MultiScaleSpectralExtractor(nn.Module):\n    """Graph spectrum plus attributed graph-signal spectral energies.\n\n    In ``graph_signal_spectral`` mode, canonical/lexical information can reach\n    the classifier only after node signals have been filtered by polynomial\n    functions of the learned latent Laplacian.  Squared energy is aggregated\n    over nodes, making the descriptor invariant to latent-node permutations.\n    There is no state pooling or raw-feature concatenation in this branch.\n    """\n\n    def __init__(self, dim: int, config: RevisionModelConfig) -> None:\n        super().__init__()\n        self.config = config\n        if config.readout_mode in {"hybrid", "graph_signal_spectral"}:\n            self.signal = nn.Linear(dim, config.spectral_signals, bias=False)\n        else:\n            self.signal = None\n        self.band_attention = (\n            nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, config.spectral_bands))\n            if config.readout_mode == "hybrid"\n            else None\n        )\n\n        # Chebyshev coefficients use uniform angular quadrature, not uniform\n        # Laplacian eigenvalues (which gives a different integration measure).\n        theta = (torch.arange(256) + 0.5) * math.pi / 256\n        grid = 1 + torch.cos(theta)\n        coefficients = []\n        for center in torch.linspace(0.05, 1.95, config.spectral_bands):\n            response = torch.exp(-0.5 * ((grid - center) / 0.18) ** 2)\n            coefficients.append(\n                torch.stack(\n                    [\n                        (response * torch.cos(order * theta)).mean() * (1 if order == 0 else 2)\n                        for order in range(config.chebyshev_degree + 1)\n                    ]\n                )\n            )\n        self.register_buffer("chebyshev_coefficients", torch.stack(coefficients))\n        self.register_buffer("density_centers", torch.linspace(0, 2, config.density_bins))\n        self.register_buffer("heat_times", torch.logspace(-2, 2, config.heat_samples))\n\n    def forward(self, states: torch.Tensor, adjacency: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        # Eigensolver and graph filtering remain float32 under AMP.\n        with torch.autocast(device_type=states.device.type, enabled=False):\n            states, adjacency = states.float(), adjacency.float()\n            batch_size, node_count, _ = adjacency.shape\n            identity = torch.eye(node_count, device=adjacency.device).unsqueeze(0)\n            adjacency = torch.nan_to_num(adjacency, nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)\n            adjacency = 0.5 * (adjacency + adjacency.transpose(1, 2)) * (1 - identity)\n            degree = adjacency.sum(-1).clamp_min(1e-6)\n            normalized = adjacency * degree.rsqrt().unsqueeze(-1) * degree.rsqrt().unsqueeze(-2)\n            laplacian = identity - normalized\n            laplacian = 0.5 * (laplacian + laplacian.transpose(1, 2))\n\n            if self.config.readout_mode in {"eigenvalue_only", "graph_signal_spectral"}:\n                eigenvalues = torch.linalg.eigvalsh(laplacian).clamp(0, 2)\n            else:\n                with torch.no_grad():\n                    eigenvalues = torch.linalg.eigvalsh(laplacian).clamp(0, 2)\n            density = torch.exp(\n                -0.5 * ((eigenvalues.unsqueeze(-1) - self.density_centers) / 0.08) ** 2\n            ).mean(1)\n            heat_trace = torch.exp(-eigenvalues.unsqueeze(-1) * self.heat_times).mean(1)\n\n            if self.config.readout_mode == "eigenvalue_only":\n                descriptor = torch.cat((density, heat_trace), dim=-1)\n                band_weights = torch.full(\n                    (batch_size, self.config.spectral_bands),\n                    1.0 / self.config.spectral_bands,\n                    device=adjacency.device,\n                )\n            else:\n                if self.signal is None:\n                    raise RuntimeError("Graph-signal projection was not initialized")\n                # A learned channel projection creates a compact attributed\n                # graph signal. RMS normalization prevents raw signal scale\n                # from acting as a non-spectral shortcut.\n                signal = self.signal(states)\n                signal = signal / signal.square().mean(1, keepdim=True).add(1e-6).sqrt()\n\n                # T_k(L-I)X is a Chebyshev approximation of spectral filters\n                # g_b(L)X. No node state skips these graph-dependent operators.\n                shifted = laplacian - identity\n                terms = [signal, torch.bmm(shifted, signal)]\n                for _ in range(2, self.config.chebyshev_degree + 1):\n                    terms.append(2 * torch.bmm(shifted, terms[-1]) - terms[-2])\n                filtered = torch.einsum(\n                    "bkse,ck->bcse", torch.stack(terms, dim=1), self.chebyshev_coefficients\n                )\n                raw_energy = torch.log1p(filtered.square().mean(2))\n\n                if self.config.readout_mode == "graph_signal_spectral":\n                    # Fixed bands prevent an input-conditioned state gate from\n                    # becoming a covert lexical/node-embedding bypass.\n                    band_weights = torch.full(\n                        (batch_size, self.config.spectral_bands),\n                        1.0 / self.config.spectral_bands,\n                        device=adjacency.device,\n                    )\n                    energy = raw_energy.reshape(batch_size, -1)\n                else:\n                    if self.band_attention is None:\n                        raise RuntimeError("Hybrid spectral attention was not initialized")\n                    band_weights = (\n                        torch.softmax(self.band_attention(states.mean(1)), dim=-1)\n                        if USE_SPECTRAL_ATTENTION\n                        else torch.full(\n                            (batch_size, self.config.spectral_bands),\n                            1.0 / self.config.spectral_bands,\n                            device=adjacency.device,\n                        )\n                    )\n                    energy = (raw_energy * band_weights.unsqueeze(-1)).reshape(batch_size, -1)\n                descriptor = torch.cat((density, heat_trace, energy), dim=-1)\n        return descriptor, eigenvalues, band_weights\n\n\nclass RevisionEncoder(nn.Module):\n    def __init__(self, config: RevisionModelConfig) -> None:\n        super().__init__()\n        if config.hidden_dim % config.attention_heads:\n            raise ValueError("hidden_dim must be divisible by attention_heads")\n        if config.readout_mode not in {"eigenvalue_only", "graph_signal_spectral"}:\n            raise ValueError("unsupported readout_mode")\n        if config.adjacency_mode not in {"learned", "projected"} or config.pooling not in {"spectral", "mean_max", "attention", "energy_only"}:\n            raise ValueError("Unsupported controlled architecture")\n        if config.readout_mode != "hybrid" and config.use_source_lexical:\n            raise ValueError("source lexical residual is incompatible with spectral-only readouts")\n        self.config = config\n        dim = config.hidden_dim\n        self.canonical_embedding = nn.Embedding(config.canonical_vocab_size, dim, padding_idx=0)\n        self.lexical_embedding = nn.Embedding(config.lexical_buckets, dim, padding_idx=0)\n        # Start lexical fallback at roughly 20% of the canonical signal.  It\n        # can help when an identifier/literal is available, but cannot become\n        # a direct raw-node-type lookup table.\n        self.lexical_gate_logit = nn.Parameter(torch.tensor(-1.38629436))\n        self.lexical_dropout = nn.Dropout(config.lexical_dropout)\n        if config.use_source_lexical:\n            self.source_projection = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim))\n            self.source_gate_logit = nn.Parameter(torch.tensor(-0.61903921))\n        else:\n            self.source_projection = None\n            self.register_parameter("source_gate_logit", None)\n        self.input_norm = nn.LayerNorm(dim)\n        if not config.relation_indices:\n            raise ValueError("At least one input relation must be enabled.")\n        self.graph_layers = nn.ModuleList(\n            [RelationGraphLayer(dim, relations=len(config.relation_indices), dropout=config.dropout) for _ in range(2)]\n        )\n        self.slot_attention = SlotAttentionInduction(dim, config.latent_nodes, config.slot_iterations)\n        self.adjacency_query = nn.Linear(dim, dim, bias=False)\n        self.adjacency_key = nn.Linear(dim, dim, bias=False)\n        self.prior_scale = nn.Parameter(torch.tensor(1.0))\n        self.temperature_bias = nn.Parameter(torch.tensor(-0.55))\n        self.temperature_complexity = nn.Parameter(torch.tensor(1.25))\n        self.refinement = nn.ModuleList([LatentGraphRefinement(dim, config.dropout) for _ in range(2)])\n        if config.readout_mode == "eigenvalue_only" and config.pooling == "spectral":\n            self.refinement.requires_grad_(False)\n        spectral_config = replace(config, readout_mode="eigenvalue_only") if config.pooling in {"mean_max", "attention"} else config\n        self.spectral = MultiScaleSpectralExtractor(dim, spectral_config)\n        if config.adjacency_mode == "projected":\n            for module in (self.adjacency_query, self.adjacency_key):\n                module.requires_grad_(False)\n            for parameter in (self.prior_scale, self.temperature_bias, self.temperature_complexity):\n                parameter.requires_grad_(False)\n        # A common 152-D positive descriptor and the same symmetric pair head\n        # keep the readout controls comparable. Exact parameter counts are saved.\n        self.pool_projection = (nn.Linear(dim * 2 if config.pooling == "mean_max" else dim,\n                                         config.readout_descriptor_dim)\n                                if config.pooling in {"mean_max", "attention"} else None)\n        self.pool_attention = nn.Linear(dim, 1) if config.pooling == "attention" else None\n        self.output = (\n            nn.Sequential(nn.LayerNorm(dim * 2), nn.Linear(dim * 2, dim), nn.GELU(), nn.LayerNorm(dim))\n            if config.readout_mode == "hybrid"\n            else None\n        )\n\n    def forward(self, batch, temperature: float = 0.30):\n        canonical = self.canonical_embedding(batch.canonical_ids.long())\n        if self.config.use_node_lexical:\n            lexical = self.lexical_embedding(batch.lexical_ids.long()).mean(-2)\n            lexical = self.lexical_dropout(lexical) if self.training else lexical\n            canonical = canonical + torch.sigmoid(self.lexical_gate_logit) * lexical\n        input_features = self.input_norm(canonical) * batch.mask.unsqueeze(-1)\n        enabled_edges = batch.edges[:, self.config.relation_indices]\n        states = input_features\n        for layer in self.graph_layers:\n            states = layer(states, enabled_edges, batch.mask)\n\n        slots, assignment = self.slot_attention(states, batch.mask)\n        assignment = torch.nan_to_num(assignment)\n        assignment = assignment / assignment.sum(-1, keepdim=True).clamp_min(1e-6)\n        reconstruction = torch.bmm(assignment, slots)\n        reconstruction_loss = (\n            (input_features - reconstruction).square().mean(-1) * batch.mask\n        ).sum() / batch.mask.sum().clamp_min(1)\n\n        input_adjacency = enabled_edges.sum(1).clamp(max=1)\n        prior_adjacency = (batch.edges[:, self.config.prior_relation_indices].sum(1).clamp(max=1)\n                           if self.config.prior_relation_indices is not None else input_adjacency)\n        structural_prior = torch.bmm(\n            assignment.transpose(1, 2), torch.bmm(prior_adjacency, assignment)\n        )\n        structural_prior = structural_prior / structural_prior.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)\n\n        head_dim = self.config.hidden_dim // self.config.attention_heads\n        with torch.autocast(device_type=slots.device.type, enabled=False):\n            query = self.adjacency_query(slots.float()).view(\n                -1, self.config.latent_nodes, self.config.attention_heads, head_dim\n            ).transpose(1, 2)\n            key = self.adjacency_key(slots.float()).view(\n                -1, self.config.latent_nodes, self.config.attention_heads, head_dim\n            ).transpose(1, 2)\n            score = torch.matmul(query, key.transpose(-1, -2)).mean(1) / math.sqrt(head_dim)\n            score = 0.5 * (score + score.transpose(1, 2)) + self.prior_scale.float() * structural_prior.float()\n            score = torch.nan_to_num(score, nan=0.0, posinf=20.0, neginf=-20.0).clamp(-20, 20)\n            identity = torch.eye(self.config.latent_nodes, device=score.device).unsqueeze(0)\n            node_complexity = batch.mask.float().sum(-1) / self.config.max_input_nodes\n            edge_complexity = enabled_edges.float().sum(dim=(1, 2, 3)) / batch.mask.float().sum(-1).square().clamp_min(1.0)\n            complexity = (0.75 * node_complexity + 0.25 * edge_complexity.clamp(0, 1)).clamp(0, 1)\n            adaptive_temperature = ADAPTIVE_TEMPERATURE_MIN + (ADAPTIVE_TEMPERATURE_MAX - ADAPTIVE_TEMPERATURE_MIN) * torch.sigmoid(self.temperature_bias.float() + F.softplus(self.temperature_complexity.float()) * complexity) if USE_ADAPTIVE_TEMPERATURE else torch.full_like(complexity, TEMPERATURE_END)\n            # Keep a differentiable weighted graph. A fixed top-k gave\n            # every example nearly the same edge count and erased precisely the\n            # graph-to-graph spectral variation that the readout needs.\n            adjacency = torch.sigmoid(score / adaptive_temperature[:, None, None]) * (1 - identity)\n            adjacency = 0.5 * (adjacency + adjacency.transpose(1, 2))\n            if self.config.adjacency_mode == "projected":\n                adjacency = 0.5 * (structural_prior.float() + structural_prior.float().transpose(1, 2)) * (1 - identity)\n\n        projected_topology = torch.bmm(assignment, torch.bmm(adjacency.to(assignment.dtype), assignment.transpose(1, 2)))\n        projected_topology = projected_topology.clamp(1e-6, 1 - 1e-6)\n        topology_mask = batch.mask.unsqueeze(1) & batch.mask.unsqueeze(2)\n        topology_mask = topology_mask & ~torch.eye(\n            topology_mask.size(-1), device=topology_mask.device, dtype=torch.bool\n        ).unsqueeze(0)\n        topology_target = torch.maximum(input_adjacency, input_adjacency.transpose(1, 2)).to(projected_topology.dtype)\n        positive_edges = (topology_target * topology_mask).sum()\n        negative_edges = ((1 - topology_target) * topology_mask).sum()\n        topology_positive_weight = (negative_edges / positive_edges.clamp_min(1)).detach().clamp(1, 20)\n        topology_element = -(\n            topology_positive_weight * topology_target * projected_topology.log()\n            + (1 - topology_target) * (1 - projected_topology).log()\n        )\n        topology_reconstruction_loss = (\n            topology_element * topology_mask\n        ).sum() / topology_mask.sum().clamp_min(1)\n\n        latent = slots\n        for layer in self.refinement:\n            latent = layer(latent, adjacency.to(latent.dtype))\n        descriptor, eigenvalues, band_weights = self.spectral(latent, adjacency)\n        if self.config.pooling == "mean_max":\n            descriptor = F.softplus(self.pool_projection(torch.cat((latent.mean(1), latent.amax(1)), dim=-1)))\n        elif self.config.pooling == "attention":\n            weights = torch.softmax(self.pool_attention(latent), dim=1)\n            descriptor = F.softplus(self.pool_projection((latent * weights).sum(1)))\n        elif self.config.pooling == "energy_only":\n            descriptor = descriptor[:, self.config.density_bins + self.config.heat_samples:]\n        # No unused learned embedding projection. This diagnostic vector has\n        # no path to the classifier or a nonzero loss term.\n        spectral_embedding = descriptor.to(latent.dtype)\n        if self.config.readout_mode in {"eigenvalue_only", "graph_signal_spectral"}:\n            # In graph_signal_spectral mode this embedding is derived only from\n            # g_b(L)X energies and adjacency-spectrum summaries.\n            embedding = F.normalize(spectral_embedding, dim=-1)\n        else:\n            if self.output is None:\n                raise RuntimeError("Hybrid latent fusion module was not initialized")\n            embedding = F.normalize(\n                self.output(torch.cat((latent.mean(1), spectral_embedding), dim=-1)), dim=-1\n            )\n        if self.config.use_source_lexical:\n            if self.source_projection is None or self.source_gate_logit is None:\n                raise RuntimeError("Source lexical residual was not initialized")\n            source = self.lexical_embedding(batch.source_lexical_ids.long()).mean(1)\n            source = self.source_projection(source)\n            embedding = F.normalize(embedding + torch.sigmoid(self.source_gate_logit) * source, dim=-1)\n        density = adjacency.sum(dim=(1, 2)) / (\n            self.config.latent_nodes * (self.config.latent_nodes - 1)\n        )\n        density_loss = (density - self.config.target_density).square().mean()\n        connectivity_loss = F.relu(0.03 - eigenvalues[:, 1]).square().mean()\n        auxiliary = {\n            "reconstruction": reconstruction_loss,\n            "topology_reconstruction": topology_reconstruction_loss,\n            "density": density_loss,\n            "connectivity": connectivity_loss,\n            "adaptive_temperature": adaptive_temperature,\n            "spectral_attention_entropy": -(band_weights * band_weights.clamp_min(1e-8).log()).sum(-1),\n            "graph": 2 * density_loss + 0.1 * connectivity_loss,\n        }\n        if getattr(self, "capture_diagnostics", False):\n            auxiliary.update(assignment=assignment, adjacency=adjacency, eigenvalues=eigenvalues)\n        return embedding, descriptor, auxiliary\n\n\nclass RevisionPairModel(nn.Module):\n    """Symmetric comparator over separately normalized spectral blocks only."""\n\n    def __init__(self, config: RevisionModelConfig) -> None:\n        super().__init__()\n        self.config = config\n        self.encoder = RevisionEncoder(config)\n        # Density and heat trace are always separate comparison blocks.  The\n        # attributed Chebyshev-energy block exists only in spectral-signal modes.\n        self.block_sizes = (config.density_bins, config.heat_samples)\n        if config.readout_mode != "eigenvalue_only":\n            self.block_sizes += (config.spectral_bands * config.spectral_signals,)\n        if config.pooling == "energy_only":\n            self.block_sizes = (config.spectral_bands * config.spectral_signals,)\n        pair_dim = 3 * config.readout_descriptor_dim + 3 * len(self.block_sizes)\n        pair_hidden_dim = config.hidden_dim + config.hidden_dim // 2\n        pair_bottleneck_dim = config.hidden_dim - config.hidden_dim // 8\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(pair_dim),\n            nn.Linear(pair_dim, pair_hidden_dim),\n            nn.GELU(),\n            nn.Dropout(config.dropout),\n            nn.Linear(pair_hidden_dim, pair_bottleneck_dim),\n            nn.GELU(),\n            nn.Dropout(config.dropout),\n            nn.Linear(pair_bottleneck_dim, 1),\n        )\n\n    def _spectral_pair_features(\n        self, left_spectrum: torch.Tensor, right_spectrum: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        left_blocks = torch.split(left_spectrum, self.block_sizes, dim=-1)\n        right_blocks = torch.split(right_spectrum, self.block_sizes, dim=-1)\n        interactions = []\n        summaries = []\n        block_cosines = []\n        for left_block, right_block in zip(left_blocks, right_blocks):\n            left_log = torch.log1p(left_block.clamp_min(0))\n            right_log = torch.log1p(right_block.clamp_min(0))\n            left_shape = F.normalize(left_log, p=2, dim=-1)\n            right_shape = F.normalize(right_log, p=2, dim=-1)\n            difference = (left_shape - right_shape).abs()\n            interactions.extend((difference, left_shape * right_shape, (left_log - right_log).abs()))\n            block_cosine = F.cosine_similarity(left_shape, right_shape)\n            block_cosines.append(block_cosine)\n            summaries.extend(\n                (\n                    block_cosine,\n                    difference.mean(-1),\n                    difference.square().mean(-1).add(1e-8).sqrt(),\n                )\n            )\n        # The head sees only symmetric, block-wise spectral comparisons.  It\n        # never receives an individual descriptor, code embedding, pooled node\n        # state, lexical vector, or source-token feature.\n        pair_features = torch.cat((*interactions, torch.stack(summaries, dim=-1)), dim=-1)\n        spectral_cosine = torch.stack(block_cosines, dim=-1).mean(-1)\n        return pair_features, spectral_cosine\n\n    def forward(self, left, right, temperature: float = 0.30):\n        left_embedding, left_spectrum, left_aux = self.encoder(left, temperature)\n        right_embedding, right_spectrum, right_aux = self.encoder(right, temperature)\n        spectral_pair_features, spectral_cosine = self._spectral_pair_features(\n            left_spectrum, right_spectrum\n        )\n        auxiliary = {key: 0.5 * (left_aux[key] + right_aux[key]) for key in left_aux}\n        auxiliary["spectral_cosine"] = spectral_cosine\n        # Embedding similarity is diagnostic metadata only.  It is excluded\n        # from the classifier and every non-zero training-loss term.\n        auxiliary["embedding_cosine"] = F.cosine_similarity(left_embedding, right_embedding)\n        spectral_batch = torch.cat(\n            (F.normalize(left_spectrum, dim=-1), F.normalize(right_spectrum, dim=-1)), dim=0\n        )\n        spectral_std = torch.sqrt(spectral_batch.var(dim=0, unbiased=False) + 1e-4).mean()\n        auxiliary["spectral_variance"] = F.relu(SPECTRAL_VARIANCE_FLOOR - spectral_std)\n        return self.classifier(spectral_pair_features).squeeze(-1), auxiliary\n\ndef canonical_spectra_loss(\n    logits: torch.Tensor,\n    auxiliary: dict[str, torch.Tensor],\n    labels: torch.Tensor,\n    *,\n    positive_class_weight: float = 1.0,\n    spectral_weight: float = 0.30,\n    auc_ranking_weight: float = 0.10,\n    reconstruction_weight: float = 0.05,\n    topology_reconstruction_weight: float = 0.05,\n    spectral_variance_weight: float = 0.05,\n    graph_weight: float = 0.01,\n    embedding_contrastive_weight: float = 0.0,\n    embedding_negative_margin: float = 0.20,\n    hard_negative_weight: float = 0.20,\n    hard_negative_margin: float = 0.10,\n    hard_negative_fraction: float = 0.25,\n) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n    labels = labels.float()\n    if positive_class_weight <= 0:\n        raise ValueError("positive_class_weight must be positive")\n    classification = F.binary_cross_entropy_with_logits(\n        logits,\n        labels,\n        pos_weight=torch.as_tensor(positive_class_weight, device=logits.device, dtype=logits.dtype),\n    )\n    positive = labels * (1 - auxiliary["spectral_cosine"]).square()\n    negative = (1 - labels) * F.relu(auxiliary["spectral_cosine"] - 0.25).square()\n    spectral_contrastive = (positive + negative).mean()\n    positive_logits = logits[labels > 0.5]\n    negative_logits = logits[labels < 0.5]\n    if positive_logits.numel() and negative_logits.numel():\n        auc_ranking = F.softplus(\n            -(positive_logits.unsqueeze(1) - negative_logits.unsqueeze(0))\n        ).mean()\n    else:\n        auc_ranking = logits.new_zeros(())\n\n    if embedding_contrastive_weight != 0:\n        raise ValueError("embedding_contrastive_weight must remain zero for descriptor-only SPECTRA-Siam")\n    embedding_contrastive = logits.new_zeros(())\n\n    # Hard-negative mining is based on the block-wise spectral comparison,\n    # never on a standalone embedding similarity.\n    known_negative_scores = F.relu(\n        auxiliary["spectral_cosine"][labels < 0.5] - hard_negative_margin\n    ).square()\n    if known_negative_scores.numel():\n        hard_count = max(1, int(known_negative_scores.numel() * hard_negative_fraction))\n        hard_negative = torch.topk(known_negative_scores, hard_count).values.mean()\n    else:\n        hard_negative = logits.new_zeros(())\n\n\n    total = (\n        classification\n        + spectral_weight * spectral_contrastive\n        + auc_ranking_weight * auc_ranking\n        + hard_negative_weight * hard_negative\n        + reconstruction_weight * auxiliary["reconstruction"]\n        + topology_reconstruction_weight * auxiliary["topology_reconstruction"]\n        + spectral_variance_weight * auxiliary["spectral_variance"]\n        + graph_weight * auxiliary["graph"]\n    )\n    parts = {\n        "classification": classification.detach(),\n        "spectral_contrastive": spectral_contrastive.detach(),\n        "auc_ranking": auc_ranking.detach(),\n        "embedding_contrastive": embedding_contrastive.detach(),\n        "hard_negative": hard_negative.detach(),\n        "reconstruction": auxiliary["reconstruction"].detach(),\n        "topology_reconstruction": auxiliary["topology_reconstruction"].detach(),\n        "spectral_variance": auxiliary["spectral_variance"].detach(),\n        "graph": auxiliary["graph"].detach(),\n    }\n    return total, parts\n', 'numerics.py': '"""Bounded effective-batch replay for AMP overflow, without skipping updates.\n\nThe finite-gradient path preserves the original accumulation, clipping and\noptimizer sequence. Failed attempts restore RNG and model buffers and never\nadvance the training cursor. A persistent FP32 failure is still fatal.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport torch\n\nPOLICY = "amp_batch_replay_fp32_fallback_v1"\nMAX_AMP_ATTEMPTS = 3\nLEGACY_IMPLEMENTATIONS = frozenset({"6da6f8fffdc37744ce177a3f573260a6e52f5f8909d23124bcb5aae61e6e4f7a"})\n# Filled after the numerical-only training patch is frozen. This explicitly\n# prevents later architecture/data/training edits inheriting upgrade permission.\nAPPROVED_UPGRADE_TARGET = "3b8b72701001931d3c3eb090521567b7902b05deac92b7e97f5523dd8b79d477"\n\n\ndef upgrade_target_hash():\n    from .protocol import fingerprint\n    from .versioning import TRAINING_SOURCES\n    root = Path(__file__).parent\n    return fingerprint({name: (root / name).read_text(encoding="utf-8")\n                        for name in TRAINING_SOURCES if name not in {"numerics.py", "versioning.py"}})\n\n\ndef can_upgrade(implementation_hash):\n    return implementation_hash in LEGACY_IMPLEMENTATIONS and upgrade_target_hash() == APPROVED_UPGRADE_TARGET\n\n\ndef _finite_tree(value):\n    if torch.is_tensor(value):\n        return not value.is_floating_point() or bool(torch.isfinite(value).all())\n    if isinstance(value, dict):\n        return all(_finite_tree(v) for v in value.values())\n    if isinstance(value, (tuple, list)):\n        return all(_finite_tree(v) for v in value)\n    return True\n\n\ndef validate_checkpoint_numerics(state):\n    if not _finite_tree({key: state.get(key) for key in ("model", "best_model", "optimizer")}):\n        raise ValueError("Checkpoint contains non-finite weights/optimizer state; do not resume it")\n\n\nclass NumericalStepFailure(FloatingPointError):\n    def __init__(self, diagnostic):\n        self.diagnostic = diagnostic\n        super().__init__("Non-finite loss/gradients persisted in FP32; no optimizer update was applied")\n\n\nclass _AttemptFailure(Exception):\n    def __init__(self, reason, names=()):\n        self.reason, self.names = reason, list(names)\n\n\ndef optimizer_group(model, optimizer, scaler, group, *, config, device, losses,\n                    loss_function, positive_class_weight, clock=None, on_event=None):\n    """Apply exactly one optimizer update, retrying the SAME effective batch.\n\n    Input batches remain on CPU until their forward pass. This adds at most one\n    effective batch of CPU storage, not its combined activations on the GPU.\n    """\n    from .training import rng_state, restore_rng\n\n    initial_rng = rng_state()\n    buffers = {name: buffer.detach().clone() for name, buffer in model.named_buffers()}\n    examples = sum(len(labels) for _, _, labels in group)\n    events = []\n    use_scaler = scaler.is_enabled()\n    attempts = MAX_AMP_ATTEMPTS + 1 if use_scaler else 1\n\n    def restore():\n        optimizer.zero_grad(set_to_none=True)\n        restore_rng(initial_rng)\n        with torch.no_grad():\n            for name, buffer in model.named_buffers():\n                buffer.copy_(buffers[name])\n\n    for attempt in range(attempts):\n        fp32_fallback = use_scaler and attempt == MAX_AMP_ATTEMPTS\n        scaled_attempt = use_scaler and not fp32_fallback\n        autocast_enabled = config.amp and device.type == "cuda" and not fp32_fallback\n        if attempt:\n            restore()\n        contributions = []\n        try:\n            if clock:\n                clock.check()\n            for left, right, labels in group:\n                labels = labels.to(device)\n                with torch.autocast(device_type=device.type, enabled=autocast_enabled):\n                    logits, auxiliary = model(left.to(device), right.to(device), None)\n                    loss, _ = loss_function(logits, auxiliary, labels,\n                                             positive_class_weight=positive_class_weight, **losses)\n                weighted = loss * len(labels) / examples\n                # Initialize the public scaler before checking a non-finite\n                # loss, so an explicit backoff can reset its attempt state.\n                backward_loss = scaler.scale(weighted) if scaled_attempt else weighted\n                if not bool(torch.isfinite(loss)):\n                    raise _AttemptFailure("nonfinite_loss")\n                backward_loss.backward()\n                contributions.append((float(loss.detach().cpu()) * len(labels), len(labels)))\n            if scaled_attempt:\n                scaler.unscale_(optimizer)\n            try:\n                # error_if_nonfinite checks BEFORE clipping can turn inf*0\n                # into NaN. Do not mask invalid gradients with nan_to_num.\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            except RuntimeError as error:\n                if "non-finite" not in str(error).lower():\n                    raise\n                names = [name for name, parameter in model.named_parameters()\n                         if parameter.grad is not None and not bool(torch.isfinite(parameter.grad).all())]\n                raise _AttemptFailure("nonfinite_gradient_norm", names[:12]) from error\n        except _AttemptFailure as failure:\n            # A non-finite forward loss has not run backward, so its graph can\n            # still hold GPU activations. Release it before the replay/fallback.\n            del logits, auxiliary, loss, weighted, backward_loss\n            event = {"attempt": attempt + 1, "precision": "fp32_fallback" if fp32_fallback else\n                     "amp" if scaled_attempt else "fp32", "reason": failure.reason,\n                     "nonfinite_parameters": failure.names,\n                     "loss_scale": float(scaler.get_scale()) if scaled_attempt else None}\n            if scaled_attempt:\n                # No step occurs for this failed attempt. Public explicit-scale\n                # update clears the unscale state for the replayed group.\n                new_scale = max(float(scaler.get_scale()) * scaler.get_backoff_factor(),\n                                torch.finfo(torch.float32).tiny)\n                scaler.update(new_scale=new_scale)\n                event["next_loss_scale"] = new_scale\n                event["action"] = "retry_same_group" if attempt + 1 < MAX_AMP_ATTEMPTS else "retry_same_group_fp32"\n            else:\n                event["action"] = "stop_with_recoverable_checkpoint"\n            events.append(event)\n            if on_event:\n                on_event(event)\n            if not scaled_attempt:\n                restore()\n                raise NumericalStepFailure({"policy": POLICY, "events": events}) from failure\n            continue\n        except BaseException:\n            # In particular, a session deadline must save the pre-group RNG,\n            # buffers and cursor, not a half-computed accumulation group.\n            restore()\n            raise\n\n        if scaled_attempt:\n            scaler.step(optimizer)\n            scaler.update()\n        else:\n            optimizer.step()\n        optimizer.zero_grad(set_to_none=True)\n        if fp32_fallback:\n            event = {"attempt": attempt + 1, "precision": "fp32_fallback", "action": "group_recovered"}\n            events.append(event)\n            if on_event:\n                on_event(event)\n        return contributions, events\n\n    raise AssertionError("All AMP attempts must lead to a finite update or a recorded FP32 failure")\n', 'prepare.py': '"""Build disjoint, balanced pairs from existing exported CodeNet program graphs.\n\nProblem identity remains a proxy for functionality, as in the original release.\nNo behavioral equivalence checking is claimed. Exact duplicate source and shared\nsubmission IDs join problem groups before splitting. Graph extraction is reused.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport gzip\nimport hashlib\nimport io\nimport json\nimport shutil\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .protocol import fingerprint\n\nLANGUAGES = ("python", "java", "cpp", "csharp")\nCONFIGURATIONS = tuple((a, b) for i, a in enumerate(LANGUAGES) for b in LANGUAGES[i:])\n\n\ndef _sha(text: str) -> str:\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef assign_problem_splits(codes: list[dict], seed: int = 42) -> dict[str, str]:\n    """Union duplicate sources/submissions across problems, then hash components."""\n    parent = {str(c["problem_id"]): str(c["problem_id"]) for c in codes}\n\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n\n    seen = {}\n    for code in codes:\n        problem = str(code["problem_id"])\n        source = str(code.get("code", ""))\n        keys = [("source", _sha(source))]\n        if code.get("source_code_id"):\n            keys.append(("submission", str(code["source_code_id"])))\n        for key in keys:\n            if key in seen:\n                a, b = find(problem), find(seen[key])\n                parent[max(a, b)] = min(a, b)\n            else:\n                seen[key] = problem\n    result = {}\n    for problem in parent:\n        value = int(_sha(f"revision-problem-v1/{seed}/{find(problem)}")[:16], 16) / 2**64\n        result[problem] = "train" if value < .70 else "valid" if value < .85 else "test"\n    return result\n\n\ndef pair_overlap(pairs: pd.DataFrame, codes: list[dict]) -> dict:\n    index = {str(c["code_id"]): c for c in codes}\n    sets = {}\n    for split in ("train", "valid", "test"):\n        frame = pairs[pairs.split.eq(split)]\n        ids = set(frame.left_id.astype(str)) | set(frame.right_id.astype(str))\n        sets[split] = {\n            "endpoint": ids,\n            "problem": {str(index[c]["problem_id"]) for c in ids},\n            "source_hash": {_sha(index[c]["code"]) for c in ids},\n            "submission": {str(index[c].get("source_code_id", c)) for c in ids},\n        }\n    return {f"{a}_{b}": {key: len(sets[a][key] & sets[b][key]) for key in sets[a]}\n            for a, b in (("train", "valid"), ("train", "test"), ("valid", "test"))}\n\n\ndef generate_pairs(codes: list[dict], assignments: dict[str, str], *, per_configuration: int = 10000,\n                   seed: int = 42) -> pd.DataFrame:\n    if per_configuration <= 0 or per_configuration % 40:\n        raise ValueError("Pairs per configuration must be a positive multiple of 40 for exact 70/15/15 balance")\n    rng = np.random.default_rng(seed)\n    groups = defaultdict(lambda: defaultdict(list))\n    index = {str(c["code_id"]): c for c in codes}\n    for code in codes:\n        split = assignments[str(code["problem_id"])]\n        groups[(split, code["language"])][str(code["problem_id"])].append(str(code["code_id"]))\n    rows = []\n    for split, fraction in (("train", .7), ("valid", .15), ("test", .15)):\n        per_label = int(per_configuration * fraction / 2)\n        for a, b in CONFIGURATIONS:\n            left, right = groups[(split, a)], groups[(split, b)]\n            shared = sorted(p for p in left.keys() & right.keys() if a != b or len(left[p]) >= 2)\n            if not shared or len(set(left) | set(right)) < 2:\n                raise ValueError(f"Insufficient problems for {split}/{a}/{b}; change seed or input pool")\n            for label in (1, 0):\n                used = set()\n                problems_a, problems_b = sorted(left), sorted(right)\n                attempts = 0\n                while len(used) < per_label:\n                    attempts += 1\n                    if attempts > max(10000, per_label * 100):\n                        raise ValueError(f"Insufficient distinct pairs for {split}/{a}/{b}/{label}; reduce quota")\n                    p = str(rng.choice(shared if label else problems_a))\n                    q = p if label else str(rng.choice(problems_b))\n                    if not label and p == q:\n                        continue\n                    x, y = str(rng.choice(left[p])), str(rng.choice(right[q]))\n                    if x == y or _sha(index[x]["code"]) == _sha(index[y]["code"]):\n                        continue\n                    key = tuple(sorted((x, y)))\n                    if key in used:\n                        continue\n                    used.add(key)\n                    configuration = a if a == b else f"{a}_{b}"\n                    rows.append({"split": split, "left_id": x, "right_id": y, "label": label,\n                                 "configuration_id": configuration,\n                                 "pair_id": _sha(f"{split}/{x}/{y}")[:24],\n                                 "pair_kind": "clone" if label else "nonclone_diff_problem"})\n    return pd.DataFrame(rows)\n\n\ndef prepare(source_zip: Path, output: Path, *, per_configuration: int = 10000, seed: int = 42) -> dict:\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Choose an empty output directory; existing artifacts are preserved: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(source_zip) as archive:\n        names = archive.namelist()\n        code_member = next(n for n in names if n.endswith("/codes.jsonl.gz"))\n        graph_member = next(n for n in names if n.endswith("/graph_spectra.jsonl.gz"))\n        with archive.open(code_member) as raw, gzip.open(raw, "rt", encoding="utf-8") as stream:\n            codes = [json.loads(line) for line in stream if line.strip()]\n        codes = [c for c in codes if c.get("language") in LANGUAGES and c.get("problem_id")\n                 and str(c.get("code", "")).strip() and not c.get("is_mutant", False)]\n        wanted = {str(c["code_id"]) for c in codes}\n        usable, coverage = set(), Counter()\n        # Reuse the existing gzip byte stream; recompressing the multi-GB graph\n        # corpus wastes substantial preparation time. Unused records are safe:\n        # the corpus loader selects only IDs referenced by the new pair table.\n        cache = output / "graph_spectra.jsonl.gz"\n        with archive.open(graph_member) as raw, cache.open("wb") as dest:\n            shutil.copyfileobj(raw, dest, length=1024 * 1024)\n        with gzip.open(cache, "rt", encoding="utf-8") as stream:\n            for line in stream:\n                row = json.loads(line)\n                code_id = str(row.get("code_id"))\n                if code_id not in wanted:\n                    continue\n                ast = row.get("graphs", {}).get("ast", {})\n                adjacency = ast.get("adjacency", ast)\n                if not adjacency.get("node_types"):\n                    coverage["missing_ast"] += 1\n                    continue\n                usable.add(code_id)\n                coverage["truncated_ast_256"] += len(adjacency["node_types"]) > 256\n                for layer in ("ast", "cfg", "ddg", "cpg"):\n                    item = row.get("graphs", {}).get(layer, {})\n                    coverage[f"{layer}_missing_spectra"] += not bool(item.get("eigenvalues"))\n        codes = [c for c in codes if str(c["code_id"]) in usable]\n        assignments = assign_problem_splits(codes, seed)\n        pairs = generate_pairs(codes, assignments, per_configuration=per_configuration, seed=seed)\n        overlap = pair_overlap(pairs, codes)\n        if any(count for values in overlap.values() for count in values.values()):\n            raise RuntimeError(f"Disjointness audit failed: {overlap}")\n        with gzip.open(output / "codes.jsonl.gz", "wt", encoding="utf-8") as stream:\n            for code in codes:\n                stream.write(json.dumps(code, ensure_ascii=False) + "\\n")\n        pairs.to_csv(output / "pairs.csv.gz", index=False, compression="gzip")\n        pd.DataFrame([{"problem_id": p, "split": s} for p, s in sorted(assignments.items())]).to_csv(output / "problem_splits.csv", index=False)\n    manifest = {\n        "format": "spectral_clean_data_v1", "protocol": "revision_problem_disjoint_v1",\n        "dataset": "CodeNet problem/submission/exact-source-disjoint", "seed": seed,\n        "input_archive": str(source_zip.resolve()), "source_codes_member_crc": archive.getinfo(code_member).CRC,\n        "source_graphs_member_crc": archive.getinfo(graph_member).CRC,\n        "codes": len(codes), "pairs": len(pairs), "overlap": overlap,\n        "coverage": dict(coverage), "user_disjoint": "unverified: user IDs unavailable in this release",\n        "label_definition": "same problem = positive, different problem = negative; no execution validation",\n        "pair_table_sha256": fingerprint(pairs.to_dict("records")),\n        "problem_assignments_sha256": fingerprint(assignments),\n        "graph_support": "common valid AST backbone; other graph-layer failures remain reported",\n    }\n    (output / "metadata.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source-zip", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--per-configuration", type=int, default=10000)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    print(json.dumps(prepare(args.source_zip, args.output, per_configuration=args.per_configuration, seed=args.seed), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'progress.py': '"""Optional tqdm adapters; leave scientific source/checkpoint identities unchanged.\n\nInstalled by notebook setup, rather than by training modules. Adapters preserve\nloader order and file contents. The benchmark retains its unwrapped loader.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom functools import wraps\nimport json\nimport math\nfrom pathlib import Path\n\nfrom tqdm.auto import tqdm\n\n_patches = []\n_state_bars = {}\n\n\ndef _patch(module, name, replacement):\n    _patches.append((module, name, getattr(module, name)))\n    setattr(module, name, replacement)\n\n\nclass _ProgressLoader:\n    def __init__(self, loader, *, initial, total, description):\n        self.loader, self.initial, self.total, self.description = loader, initial, total, description\n\n    def __len__(self):\n        return len(self.loader)\n\n    def __getattr__(self, name):\n        return getattr(self.loader, name)\n\n    def __iter__(self):\n        with tqdm(self.loader, initial=self.initial, total=self.total,\n                  desc=self.description, unit="batch", mininterval=2, leave=False) as bar:\n            yield from bar\n\n\nclass _ProgressReader:\n    def __init__(self, stream, bar):\n        self.stream, self.bar = stream, bar\n\n    def __getattr__(self, name):\n        return getattr(self.stream, name)\n\n    def _update(self):\n        # Gzip\'s underlying file position measures compressed bytes; plain text\n        # uses the buffered binary stream. Do not seek or consume extra input.\n        binary = getattr(self.stream, "buffer", self.stream)\n        physical = getattr(binary, "fileobj", binary)\n        current = min(self.bar.total, physical.tell())\n        self.bar.update(max(0, current - self.bar.n))\n\n    def __iter__(self):\n        for line in self.stream:\n            self._update()\n            yield line\n\n    def read(self, *args, **kwargs):\n        result = self.stream.read(*args, **kwargs)\n        self._update()\n        return result\n\n    def readline(self, *args, **kwargs):\n        result = self.stream.readline(*args, **kwargs)\n        self._update()\n        return result\n\n\ndef _show_state(path, state):\n    if "codes" not in state or "next_index" not in state:\n        return\n    key = str(path)\n    bar = _state_bars.get(key)\n    if bar is None:\n        bar = _state_bars[key] = tqdm(total=state["codes"], initial=state["next_index"],\n                                      desc="Tokenize" if Path(path).name == "tokens.json" else "Embed",\n                                      unit="program", mininterval=2, leave=False)\n    bar.update(max(0, state["next_index"] - bar.n))\n\n\ndef _close_states():\n    for bar in _state_bars.values():\n        bar.close()\n    _state_bars.clear()\n\n\ndef _close_after(function):\n    @wraps(function)\n    def run(*args, **kwargs):\n        try:\n            return function(*args, **kwargs)\n        finally:\n            _close_states()\n    return run\n\n\ndef install_progress():\n    """Enable notebook progress once; no optimizer, RNG, sampler or config edits."""\n    if _patches:\n        return\n    from . import data, training, transformer, embeddings\n    # Import before wrapping training._loader: measured benchmark loops keep the\n    # original alias and exclude display overhead, regardless of import order.\n    from . import benchmark\n    original_loader = training._loader\n    if isinstance(benchmark._loader, type(original_loader)):\n        benchmark._loader = original_loader\n\n    @wraps(original_loader)\n    def loader(frame, graphs, config, batches=None):\n        result = original_loader(frame, graphs, config, batches)\n        total = math.ceil(len(frame) / config.batch_size)\n        initial = total - len(result) if batches is not None else 0\n        return _ProgressLoader(result, initial=initial, total=total,\n                               description=f"Train seed {config.seed}" if batches is not None else "Evaluate")\n\n    original_open = data._open_text\n\n    @contextmanager\n    def open_text(path):\n        path = Path(path)\n        with original_open(path) as stream:\n            with tqdm(total=path.stat().st_size, desc=f"Read {path.name}", unit="B",\n                      unit_scale=True, mininterval=2, leave=False) as bar:\n                yield _ProgressReader(stream, bar)\n\n    original_replace = transformer.replace_file\n\n    @wraps(original_replace)\n    def replace_file(source, target):\n        result = original_replace(source, target)\n        if Path(target).name == "tokens.json":\n            _show_state(target, json.loads(Path(target).read_text(encoding="utf-8")))\n        return result\n\n    original_save = embeddings._json_save\n\n    @wraps(original_save)\n    def json_save(state, path):\n        result = original_save(state, path)\n        _show_state(path, state)\n        return result\n\n    _patch(training, "_loader", loader)\n    _patch(data, "_open_text", open_text)\n    _patch(transformer, "replace_file", replace_file)\n    _patch(transformer, "prepare_tokens", _close_after(transformer.prepare_tokens))\n    _patch(embeddings, "_json_save", json_save)\n    _patch(embeddings, "extract_embeddings", _close_after(embeddings.extract_embeddings))\n\n\ndef uninstall_progress():\n    """Restore ordinary loaders/readers, useful for local tests and scripts."""\n    _close_states()\n    while _patches:\n        module, name, original = _patches.pop()\n        setattr(module, name, original)\n', 'protocol.py': '"""Explicit validation policy, common metrics, and matched transfer budgets."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,\n                             f1_score, matthews_corrcoef, precision_recall_curve,\n                             precision_score, recall_score, roc_auc_score)\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    dataset: str = "codenet"\n    seed: int = 42\n    selection_metric: str = "Accuracy"\n    max_epochs: int = 100\n    patience: int = 10\n    min_epochs: int = 10\n    batch_size: int = 32\n    effective_batch_size: int = 128\n    learning_rate: float = 1e-4\n    weight_decay: float = 1e-4\n    max_nodes: int = 256\n    session_hours: float = 10.0\n    reserve_minutes: float = 45.0\n    checkpoint_minutes: float = 10.0\n    amp: bool = True\n\n    def __post_init__(self):\n        if self.selection_metric not in {"Accuracy", "F1"}:\n            raise ValueError("Selection metric must be explicit: Accuracy or F1")\n        if not 0 < self.session_hours <= 11:\n            raise ValueError("Session budget must be at most 11h, below Kaggle\'s 12h cap")\n        if not 0 < self.reserve_minutes < self.session_hours * 60:\n            raise ValueError("Reserve must fit inside the session budget")\n        if min(self.batch_size, self.effective_batch_size, self.patience, self.max_epochs) <= 0:\n            raise ValueError("Training sizes and limits must be positive")\n        if self.effective_batch_size < self.batch_size or self.effective_batch_size % self.batch_size:\n            raise ValueError("Effective batch must be an integer multiple of the physical batch")\n\n\ndef fingerprint(value) -> str:\n    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, separators=(",", ":")).encode()).hexdigest()\n\n\ndef runtime_forecast(max_epochs: int, completed_epochs: int, elapsed_seconds: float,\n                     remaining_seconds: float | None) -> dict:\n    """Planning estimate only; never changes the scientific training schedule.\n\n    Elapsed training/validation time includes work from previous sessions and\n    retries. A 15% planning margin supplements the session\'s output reserve.\n    """\n    epochs_left = max(0, max_epochs - completed_epochs)\n    result = {"completed_epochs": completed_epochs, "epochs_left_to_cap": epochs_left,\n              "planning_margin": 0.15, "estimate_only": True}\n    if completed_epochs <= 0 or elapsed_seconds <= 0:\n        return {**result, "status": "awaiting_first_validated_epoch"}\n    seconds_per_epoch = elapsed_seconds / completed_epochs\n    result.update(status="estimated", seconds_per_epoch=seconds_per_epoch,\n                  hours_to_epoch_cap=epochs_left * seconds_per_epoch / 3600)\n    if remaining_seconds is not None:\n        available = max(0.0, remaining_seconds) * 0.85\n        capacity = int(available / seconds_per_epoch)\n        result.update(estimated_more_epochs_this_session=capacity,\n                      epoch_cap_likely_fits_this_session=epochs_left <= capacity)\n    return result\n\n\ndef metrics(labels, scores, threshold: float) -> dict:\n    y, p = np.asarray(labels, dtype=np.int8), np.asarray(scores, dtype=np.float64)\n    if len(y) == 0 or len(y) != len(p) or not np.isfinite(p).all():\n        raise ValueError("Metrics require equally sized, finite, nonempty inputs")\n    pred = p >= threshold\n    two_classes = set(np.unique(y)) == {0, 1}\n    return {\n        "P": float(precision_score(y, pred, zero_division=0)),\n        "R": float(recall_score(y, pred, zero_division=0)),\n        "F1": float(f1_score(y, pred, zero_division=0)),\n        "Accuracy": float(accuracy_score(y, pred)),\n        "BalancedAccuracy": float(balanced_accuracy_score(y, pred)),\n        "MacroF1": float(f1_score(y, pred, average="macro", zero_division=0)),\n        "AUROC": float(roc_auc_score(y, p)) if two_classes else None,\n        "AUPRC_AP": float(average_precision_score(y, p)) if two_classes else None,\n        "MCC": float(matthews_corrcoef(y, pred)), "Threshold": float(threshold),\n        "Pairs": int(len(y)), "Positives": int(y.sum()),\n        "TP": int(((y == 1) & pred).sum()), "FP": int(((y == 0) & pred).sum()),\n        "TN": int(((y == 0) & ~pred).sum()), "FN": int(((y == 1) & ~pred).sum()),\n    }\n\n\ndef choose_threshold(labels, scores, primary: str) -> tuple[float, dict]:\n    """Exact validation threshold optimization; no labels from test are used."""\n    if primary not in {"Accuracy", "F1"}:\n        raise ValueError("Primary metric must be Accuracy or F1")\n    y, p = np.asarray(labels, dtype=np.int8), np.asarray(scores, dtype=np.float64)\n    if set(np.unique(y)) != {0, 1} or not np.isfinite(p).all():\n        raise ValueError("Validation requires both classes and finite probabilities")\n    precision, recall, thresholds = precision_recall_curve(y, p)\n    # Include the all-negative operating point, missing from sklearn thresholds.\n    candidates = np.r_[thresholds, np.nextafter(p.max(), np.inf)]\n    tp = np.r_[recall[:-1] * y.sum(), 0.0]\n    fp = np.r_[np.divide(tp[:-1], precision[:-1], out=np.zeros_like(tp[:-1]), where=precision[:-1] > 0) - tp[:-1], 0.0]\n    acc = (tp + (len(y) - y.sum()) - fp) / len(y)\n    f1 = np.divide(2 * tp, tp + fp + y.sum(), out=np.zeros_like(tp), where=(tp + fp + y.sum()) > 0)\n    objective, tie = (acc, f1) if primary == "Accuracy" else (f1, acc)\n    index = max(range(len(candidates)), key=lambda i: (round(objective[i], 12), round(tie[i], 12), candidates[i]))\n    threshold = float(candidates[index])\n    return threshold, metrics(y, p, threshold)\n\n\ndef matched_training_pairs(frame: pd.DataFrame, buckets: tuple[str, ...], total: int, seed: int) -> pd.DataFrame:\n    """Same total and label budget per path, balanced within each language bucket.\n\n    Fail if the exact budget is impossible; never quietly use a smaller sample.\n    Target-language exposure still needs to be reported separately by the caller.\n    """\n    buckets = tuple(sorted(set(buckets)))\n    if total <= 0 or total % (2 * len(buckets)):\n        raise ValueError("Budget must divide equally into every configuration/label bucket")\n    n = total // (2 * len(buckets))\n    parts = []\n    for bucket in buckets:\n        for label in (0, 1):\n            group = frame[frame.configuration_id.eq(bucket) & frame.label.eq(label)]\n            if len(group) < n:\n                raise ValueError(f"Insufficient {bucket}/{label}: need {n}, have {len(group)}")\n            parts.append(group.sample(n, random_state=seed))\n    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)\n\n\ndef run_identity(config: RunConfig, model_config: dict, losses: dict, data_hash: str) -> str:\n    # Session length/checkpoint cadence can change between resume sessions.\n    settings = asdict(config)\n    for key in ("session_hours", "reserve_minutes", "checkpoint_minutes"):\n        settings.pop(key)\n    return fingerprint({"training": settings, "model": model_config, "losses": losses, "data": data_hash})\n', 'report.py': '"""Aggregate completed revision runs only, with seed uncertainty and provenance."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import t\n\nfrom .protocol import fingerprint\n\n\ndef aggregate(root: Path, output: Path, minimum_seeds: int = 3) -> pd.DataFrame:\n    rows, identities = [], set()\n    for path in sorted(root.rglob("final_test.json")):\n        result = json.loads(path.read_text(encoding="utf-8"))\n        if "test" not in result or "implementation_hash" not in result:\n            continue  # modern encoder probes have a separate, explicit schema\n        if result["identity"] in identities:\n            raise ValueError(f"Duplicate result identity; choose one authoritative archive: {path}")\n        identities.add(result["identity"])\n        training = dict(result.get("protocol", {}))\n        for key in ("seed", "session_hours", "reserve_minutes", "checkpoint_minutes"):\n            training.pop(key, None)\n        rows.append({"arm": result["arm"], "seed": result["seed"], "data_hash": result["data_hash"],\n                     "data_group_hash": result.get("data_group_hash", result["data_hash"]),\n                     "implementation_hash": result["implementation_hash"], "source": str(path),\n                     "model_configuration_hash": fingerprint(result.get("model_config", {})),\n                     "training_schedule_hash": fingerprint({"training": training, "losses": result.get("losses", {})}),\n                     **{key: result["test"][key] for key in ("Accuracy", "F1", "AUPRC_AP", "AUROC", "MCC")}})\n    if not rows:\n        raise ValueError("No completed revision test results found; training-only progress is not a paper result")\n    frame = pd.DataFrame(rows)\n    if frame.duplicated(["arm", "seed", "data_hash", "implementation_hash"]).any():\n        raise ValueError("Duplicate arm/seed results")\n    if frame.implementation_hash.nunique() != 1:\n        raise ValueError("Do not combine different implementation snapshots")\n    for _, group in frame.groupby(["arm", "data_group_hash"]):\n        if group.model_configuration_hash.nunique() != 1 or group.training_schedule_hash.nunique() != 1:\n            raise ValueError("Model revision, hyperparameters or training schedule changed across seeds; freeze choices before combining")\n    output.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(output / "revision_seed_results.csv", index=False)\n    summaries = []\n    for (arm, data_hash), group in frame.groupby(["arm", "data_group_hash"]):\n        n = group.seed.nunique()\n        if n < minimum_seeds:\n            continue\n        row = {"arm": arm, "data_hash": data_hash, "seeds": n}\n        for metric in ("Accuracy", "F1", "AUPRC_AP", "AUROC", "MCC"):\n            values = group[metric].astype(float).to_numpy()\n            mean, std = float(values.mean()), float(values.std(ddof=1)) if n > 1 else float("nan")\n            half = float(t.ppf(.975, n - 1) * std / np.sqrt(n)) if n > 1 else float("nan")\n            row.update({f"{metric}_mean": mean, f"{metric}_std": std,\n                        f"{metric}_ci95_low": mean - half, f"{metric}_ci95_high": mean + half})\n        summaries.append(row)\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(output / "revision_summary.csv", index=False)\n    (output / "README.md").write_text(\n        "Seed intervals are Student-t intervals for the mean across training seeds. "\n        "They do not represent independent problem samples or independent transfer paths. "\n        "Incomplete seed groups are retained in revision_seed_results.csv and excluded from the summary. "\n        "No degradation statistic or cross-path significance claim is inferred automatically.\\n", encoding="utf-8")\n    return summary\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--minimum-seeds", type=int, default=3)\n    args = parser.parse_args()\n    print(aggregate(args.root, args.output, args.minimum_seeds).to_string(index=False))\n', 'screening.py': '"""Collect validation-only progress; never open test results or checkpoints."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .protocol import fingerprint\n\n# Explicit compatibility exception for the audited input-only fix. External\n# baselines never call canonical_graph_from_record/lexical_source. Preserve\n# their original hashes in exports; old SPECTRA outputs remain incompatible.\nINPUT_V1_HASH = "793a3d2debc4cac7c045fb34df66f4f12f0694538e3da7a046c1b0623cee2de4"\nINPUT_V2_HASH = "8ad938f286142084e2c89ef8439bf9310db61e30019f3d61d02b90426df52e26"\nEXTERNAL_INPUT_V1_ARMS = frozenset({\n    "baseline/" + method for method in\n    ("astnn", "rtvnn", "cdlh", "deepsim", "fa_ast_ggnn", "fa_ast_gmn", "codebert", "c4_adapted", "qwen3_lora")\n})\n\n\ndef comparison_implementation_hash(state):\n    implementation = state["implementation_hash"]\n    if implementation == INPUT_V1_HASH and state["arm"] in EXTERNAL_INPUT_V1_ARMS:\n        return INPUT_V2_HASH\n    return implementation\n\n\ndef collect_validation(root, output, queue=None):\n    snapshots = {}\n    for path in sorted(Path(root).rglob("status.json")):\n        state = json.loads(path.read_text(encoding="utf-8"))\n        if not {"identity", "training", "implementation_hash", "arm"} <= state.keys():\n            continue\n        identity = state["identity"]\n        state["sources"] = [str(path)]\n        previous = snapshots.get(identity)\n        if previous:\n            a, b = previous.get("history", []), state.get("history", [])\n            n = min(len(a), len(b))\n            if fingerprint(a[:n]) != fingerprint(b[:n]):\n                raise ValueError(f"Divergent histories for the same identity: {identity}")\n            rank = lambda s: (bool(s.get("complete")), len(s.get("history", [])), s.get("optimizer_steps", 0))\n            chosen = state if rank(state) > rank(previous) else previous\n            sources = previous["sources"] + state["sources"]\n            state = {**chosen, "sources": sources}\n        snapshots[identity] = state\n    planned = pd.read_csv(queue) if queue is not None else None\n    planned_keys = (set(zip(planned.dataset, planned.arm, planned.seed.astype(int)))\n                    if planned is not None else None)\n    rows = []\n    for state in snapshots.values():\n        config = state["training"]\n        key = (config["dataset"], state["arm"], int(config["seed"]))\n        if planned_keys is not None and key not in planned_keys:\n            continue\n        valid = state.get("best_valid") or {}\n        primary = config["selection_metric"]\n        rows.append({"dataset": key[0], "arm": key[1], "seed": key[2],\n                     "identity": state["identity"], "data_hash": state["data_hash"],\n                     "implementation_hash": state["implementation_hash"],\n                     "comparison_implementation_hash": comparison_implementation_hash(state),\n                     "complete": bool(state.get("complete")), "status": state["status"],\n                     "stop_reason": state.get("stop_reason"), "primary_metric": primary,\n                     "validation_score": valid.get(primary), "best_epoch": state.get("best_epoch"),\n                     "validated_epochs": len(state.get("history", [])),\n                     "optimizer_steps": state.get("optimizer_steps"), "gpu": state.get("gpu"),\n                     "sources": json.dumps(state["sources"]),\n                     **{f"valid_{m}": valid.get(m) for m in\n                        ("Accuracy", "F1", "AUROC", "AUPRC_AP", "MCC", "Threshold")}})\n    frame = pd.DataFrame(rows)\n    keys = ["dataset", "arm", "seed"]\n    if not frame.empty and frame.duplicated(keys).any():\n        raise ValueError("Multiple configurations for the same dataset/arm/seed; select authoritative candidate outputs first")\n    if not frame.empty:\n        for _, group in frame.groupby("dataset"):\n            if group.comparison_implementation_hash.nunique() != 1 or group.data_hash.nunique() != 1:\n                raise ValueError("Screen each dataset on one common support and training implementation")\n        # Incomplete jobs are visible in the ledger but cannot win screening.\n        frame["validation_rank"] = float("nan")\n        complete = frame.complete & frame.validation_score.notna()\n        frame.loc[complete, "validation_rank"] = frame[complete].groupby(["dataset", "seed"])[\n            "validation_score"].rank(method="min", ascending=False)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(output / "validation_screening.csv", index=False)\n    if planned is not None:\n        ledger = (planned.merge(frame, how="left", on=keys, validate="one_to_one") if not frame.empty\n                  else planned.assign(status="not_started", complete=False))\n        ledger["status"] = ledger.status.fillna("not_started")\n        ledger.to_csv(output / "run_ledger.csv", index=False)\n    (output / "README.md").write_text(\n        "Validation only. No final_test.json, predictions or progress.pt files were opened. "\n        "Only completed runs receive a rank. Resumed snapshots are deduplicated by identity; "\n        "conflicting configurations require an explicit candidate choice. Single-seed ranks "\n        "are screening evidence, not statistical superiority claims. The documented input-v2 "\n        "exception reuses input-v1 external baselines only: their preparation is unchanged. "\n        "Original implementation hashes remain in the CSV. Old SPECTRA inputs are incompatible.\\n", encoding="utf-8")\n    return frame\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--queue", type=Path)\n    args = parser.parse_args()\n    print(collect_validation(args.root, args.output, args.queue).to_string(index=False))\n', 'statistics.py': '"""Paired multi-seed comparisons with shared-problem resampling across paths.\n\nThis is an exploratory multi-membership/pigeonhole bootstrap, not a claim of\nindependent transfer paths or a distribution-free significance test. One set\nof problem weights is reused across both methods, all seeds and all paths.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .protocol import fingerprint\nfrom .screening import comparison_implementation_hash\nfrom tqdm.auto import tqdm\nfrom .io_utils import replace_file\n\n\ndef _family(label):\n    if label.startswith("transfer/"):\n        parts = label.split("/", 2)\n        return parts[1] if len(parts) == 3 else "full"  # legacy transfer/{key}\n    if label.startswith("baseline/"):\n        return "/".join(label.split("/")[:2])\n    return label\n\n\ndef read_predictions(root, methods=None):\n    runs = {}\n    for path in sorted(Path(root).rglob("final_test.json")):\n        result = json.loads(path.read_text(encoding="utf-8"))\n        if "test" not in result or not result.get("test", {}).get("Pairs"):\n            continue\n        if methods is not None and _family(result["arm"]) not in methods:\n            continue\n        transfer_file = path.parent / "transfer_metadata.json"\n        condition = json.loads(transfer_file.read_text())["key"] if transfer_file.exists() else "main"\n        key = (_family(result["arm"]), condition, int(result["seed"]))\n        if key in runs:\n            raise ValueError(f"Duplicate method/condition/seed: {key}")\n        frame = pd.read_csv(path.parent / "test_predictions.csv.gz",\n                            dtype={"pair_id": str, "left_id": str, "right_id": str,\n                                   "left_problem_id": str, "right_problem_id": str})\n        required = {"pair_id", "left_problem_id", "right_problem_id", "label", "probability"}\n        if not required.issubset(frame) or frame[list(required)].isna().any().any():\n            raise ValueError(f"Known problem IDs and pair IDs are required for clustered analysis: {path}")\n        if frame.pair_id.duplicated().any():\n            raise ValueError("Duplicate pair IDs within an evaluation")\n        frame = frame.sort_values("pair_id").reset_index(drop=True)\n        frame["prediction"] = frame.probability.ge(result["test"]["Threshold"]).astype(int)\n        runs[key] = (result, frame)\n    return runs\n\n\ndef _score(y, prediction, weights, metric):\n    if weights.sum() <= 0:\n        return np.nan\n    if metric == "Accuracy":\n        return np.average(y == prediction, weights=weights)\n    tp = np.sum(weights * ((y == 1) & (prediction == 1)))\n    fp = np.sum(weights * ((y == 0) & (prediction == 1)))\n    fn = np.sum(weights * ((y == 1) & (prediction == 0)))\n    denominator = 2 * tp + fp + fn\n    return 2 * tp / denominator if denominator else 0.0\n\n\ndef paired_comparison(root, method_a, method_b, output, *, repetitions=2000, seed=42, metric="Accuracy", clock=None):\n    if metric not in {"Accuracy", "F1"} or repetitions < 2:\n        raise ValueError("Choose Accuracy/F1 and at least two bootstrap replicates")\n    runs = read_predictions(root, {method_a, method_b})\n    left = {(condition, run_seed): value for (method, condition, run_seed), value in runs.items() if method == method_a}\n    right = {(condition, run_seed): value for (method, condition, run_seed), value in runs.items() if method == method_b}\n    if not left or set(left) != set(right):\n        raise ValueError("Both methods must have exactly the same completed conditions and seeds; no silent intersections")\n    conditions = sorted({key[0] for key in left})\n    seeds = sorted({key[1] for key in left})\n    if len(seeds) < 3 or set(left) != {(condition, run_seed) for condition in conditions for run_seed in seeds}:\n        raise ValueError("At least three shared seeds, complete for every condition, are required")\n    aligned, problems, versions, supports, protocols = {}, set(), set(), set(), {}\n    rows = []\n    for key in sorted(left):\n        a, fa = left[key]\n        b, fb = right[key]\n        versions.update((comparison_implementation_hash(a), comparison_implementation_hash(b)))\n        supports.update((a.get("data_group_hash", a["data_hash"]), b.get("data_group_hash", b["data_hash"])))\n        for result in (a, b):\n            training = dict(result.get("protocol", {}))\n            for field in ("seed", "session_hours", "reserve_minutes", "checkpoint_minutes"):\n                training.pop(field, None)\n            method_condition = (_family(result["arm"]), key[0])\n            protocols.setdefault(method_condition, set()).add(fingerprint({\n                "model": result.get("model_config", {}), "training": training, "losses": result.get("losses", {})}))\n        columns = ["pair_id", "left_problem_id", "right_problem_id", "label"]\n        if not fa[columns].equals(fb[columns]):\n            raise ValueError(f"Different test support/labels for {key}; freeze shared support before comparing")\n        problems.update(fa.left_problem_id); problems.update(fa.right_problem_id)\n        aligned[key] = (fa, fb)\n        y = fa.label.to_numpy()\n        weights = np.ones(len(y))\n        av = _score(y, fa.prediction.to_numpy(), weights, metric)\n        bv = _score(y, fb.prediction.to_numpy(), weights, metric)\n        rows.append({"condition": key[0], "seed": key[1], "method_a": av, "method_b": bv, "difference_a_minus_b": av - bv})\n    if len(versions) != 1 or len(supports) != 1:\n        raise ValueError("Mixed implementation snapshots or dataset supports")\n    if any(len(settings) != 1 for settings in protocols.values()):\n        raise ValueError("Model revisions/hyperparameters/schedules differ across seeds")\n    problem_index = {key: i for i, key in enumerate(sorted(problems))}\n    arrays = {}\n    for key, (fa, fb) in aligned.items():\n        arrays[key] = (fa.label.to_numpy(), fa.prediction.to_numpy(), fb.prediction.to_numpy(),\n                       fa.left_problem_id.map(problem_index).to_numpy(), fa.right_problem_id.map(problem_index).to_numpy())\n    rng = np.random.default_rng(seed)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    progress_file = output / "bootstrap_progress.json"\n    identity = fingerprint({"a": method_a, "b": method_b, "metric": metric, "repetitions": repetitions,\n                            "seed": seed, "inputs": {str(key): tuple(array.tolist() for array in values) for key, values in arrays.items()},\n                            "implementation": Path(__file__).read_text(encoding="utf-8")})\n    progress = {"identity": identity, "next_repetition": 0, "differences": [], "rng": rng.bit_generator.state}\n    if progress_file.exists():\n        progress = json.loads(progress_file.read_text())\n        if progress["identity"] != identity:\n            raise ValueError("Bootstrap continuation identity mismatch")\n        rng.bit_generator.state = progress["rng"]\n    differences = progress["differences"]\n    def save_progress():\n        progress["rng"] = rng.bit_generator.state\n        temporary = progress_file.with_suffix(".tmp")\n        temporary.write_text(json.dumps(progress), encoding="utf-8")\n        replace_file(temporary, progress_file)\n    for iteration in tqdm(range(progress["next_repetition"], repetitions),\n                          initial=progress["next_repetition"], total=repetitions,\n                          desc="Paired bootstrap", unit="replicate", mininterval=2):\n        if clock is not None:\n            from .training import SessionBudgetReached\n            try:\n                clock.check()\n            except SessionBudgetReached:\n                save_progress()\n                return {"status": "incomplete_session_budget", "completed_repetitions": iteration,\n                        "continuation": "Reattach the bootstrap output and copy bootstrap_progress.json to this output directory"}\n        # Resample seed identities as complete trajectories, not per-path seeds.\n        sampled_seeds = rng.choice(seeds, len(seeds), replace=True)\n        counts = rng.multinomial(len(problems), np.full(len(problems), 1 / len(problems)))\n        path_differences = []\n        for condition in conditions:\n            values = []\n            for run_seed in sampled_seeds:\n                y, a, b, p, q = arrays[(condition, run_seed)]\n                # Same-problem pairs belong to one cluster, not two copies.\n                weights = counts[p] * np.where(p == q, 1, counts[q])\n                values.append(_score(y, a, weights, metric) - _score(y, b, weights, metric))\n            path_differences.append(np.mean(values))\n        difference = np.mean(path_differences)\n        if np.isfinite(difference):\n            differences.append(float(difference))\n        progress["next_repetition"] = iteration + 1\n        if (iteration + 1) % 50 == 0:\n            save_progress()\n    save_progress()\n    if len(differences) < repetitions * .9:\n        raise ValueError("Too many empty resamples; more independent problems are needed")\n    frame = pd.DataFrame(rows)\n    estimate = float(frame.groupby("condition").difference_a_minus_b.mean().mean())\n    result = {"method_a": method_a, "method_b": method_b, "metric": metric,\n              "difference_a_minus_b": estimate,\n              "ci95_percentile": np.quantile(differences, [.025, .975]).tolist(),\n              "seeds": seeds, "conditions": len(conditions), "problems": len(problems),\n              "repetitions": repetitions, "valid_repetitions": len(differences),\n              "aggregation": "equal weight per condition, then per training seed",\n              "resampling": "seed trajectories and shared problem weights; same weights across methods and paths",\n              "limitation": "Exploratory multi-membership bootstrap; does not establish language-population generalization or independent transfer paths. No p-value reported."}\n    frame.to_csv(output / "paired_condition_seed_scores.csv", index=False)\n    pd.DataFrame({"difference_a_minus_b": differences}).to_csv(output / "bootstrap_differences.csv", index=False)\n    (output / "paired_cluster_summary.json").write_text(json.dumps(result, indent=2), encoding="utf-8")\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--method-a", default="full")\n    parser.add_argument("--method-b", required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--repetitions", type=int, default=2000)\n    parser.add_argument("--metric", choices=("Accuracy", "F1"), default="Accuracy")\n    args = parser.parse_args()\n    print(json.dumps(paired_comparison(args.root, args.method_a, args.method_b, args.output,\n                                       repetitions=args.repetitions, metric=args.metric), indent=2))\n', 'training.py': '"""Resumable training: fixed validation selection and explicit final test release.\n\nCheckpoint only at optimizer boundaries. A resumed run retains optimizer/scaler,\nPython/NumPy/Torch/CUDA RNG, epoch permutation position, and validation history.\nTime exhaustion produces an incomplete run, never a completed paper result.\n"""\nfrom __future__ import annotations\n\nimport json\nimport hashlib\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom .batching import CanonicalPairDataset, make_collate\nfrom .data import CleanDataCorpus, pair_code_ids, verify_alignment, _is_gzip_file\nfrom .model import RevisionModelConfig, RevisionPairModel, canonical_spectra_loss\nfrom .protocol import RunConfig, choose_threshold, fingerprint, metrics, run_identity, runtime_forecast\nfrom .io_utils import replace_file\n\nLOSS_WEIGHTS = {\n    "spectral_weight": .30, "auc_ranking_weight": .10, "hard_negative_weight": .20,\n    "reconstruction_weight": .05, "topology_reconstruction_weight": .05,\n    "spectral_variance_weight": .05, "graph_weight": .01,\n}\nARMS = {\n    "full": {}, "mean_max": {"pooling": "mean_max"},\n    "attention": {"pooling": "attention"}, "projected": {"adjacency_mode": "projected"},\n    "projected_ast": {"adjacency_mode": "projected", "prior_relation_indices": (0,)},\n    "projected_ddg": {"adjacency_mode": "projected", "prior_relation_indices": (2,)},\n    "density_heat": {"readout_mode": "eigenvalue_only"}, "energy_only": {"pooling": "energy_only"},\n    "classification_only": {},\n    "mean_max_classification_only": {"pooling": "mean_max"},\n    "attention_classification_only": {"pooling": "attention"},\n    "auxiliary_half": {}, "auxiliary_double": {},\n    "without_spectral_and_hard_negative": {},\n    **{f"without_{name.removesuffix(\'_weight\')}": {} for name in LOSS_WEIGHTS},\n}\n\n\ndef arm_settings(arm: str) -> tuple[dict, dict]:\n    if arm not in ARMS:\n        raise ValueError(f"Unknown arm {arm}; choose {tuple(ARMS)}")\n    losses = dict(LOSS_WEIGHTS)\n    if arm.endswith("classification_only"):\n        losses = dict.fromkeys(losses, 0.0)\n    elif arm in {"auxiliary_half", "auxiliary_double"}:\n        scale = .5 if arm == "auxiliary_half" else 2.0\n        losses = {key: scale * value for key, value in losses.items()}\n    elif arm == "without_spectral_and_hard_negative":\n        losses.update(spectral_weight=0.0, hard_negative_weight=0.0)\n    elif arm.startswith("without_"):\n        losses[arm.removeprefix("without_") + "_weight"] = 0.0\n    return dict(ARMS[arm]), losses\n\n\nclass SessionBudgetReached(RuntimeError):\n    pass\n\n\nclass SessionClock:\n    def __init__(self, config: RunConfig, started: float | None = None):\n        self.started = time.monotonic() if started is None else started\n        self.deadline = self.started + config.session_hours * 3600 - config.reserve_minutes * 60\n\n    def check(self):\n        if time.monotonic() >= self.deadline:\n            raise SessionBudgetReached("Session training/evaluation budget exhausted; resume the saved progress")\n\n\ndef atomic_save(value, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(value, temp)\n    replace_file(temp, path)\n\n\ndef rng_state() -> dict:\n    return {"python": random.getstate(), "numpy": np.random.get_state(),\n            "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n\n\ndef restore_rng(value: dict):\n    random.setstate(value["python"])\n    np.random.set_state(value["numpy"])\n    torch.set_rng_state(value["torch"].cpu())\n    if value["cuda"] is not None and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all([s.cpu() for s in value["cuda"]])\n\n\ndef _cpu_state(model):\n    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n\n\ndef _load_checkpoint(path, device):\n    # Only load progress produced by this project and deliberately reattached\n    # by the user; optimizer/RNG state requires the full trusted checkpoint.\n    return torch.load(path, map_location=device, weights_only=False)\n\n\ndef _batches(size: int, batch_size: int, seed: int, epoch: int, skip: int = 0):\n    generator = torch.Generator().manual_seed(seed + epoch)\n    order = torch.randperm(size, generator=generator).tolist()\n    return [order[i:i + batch_size] for i in range(0, size, batch_size)][skip:]\n\n\ndef _loader(frame, graphs, config, batches=None):\n    if hasattr(graphs, "pair_loader"):\n        return graphs.pair_loader(frame, config, batches)\n    kwargs = {"batch_sampler": batches} if batches is not None else {"batch_size": config.batch_size, "shuffle": False}\n    return DataLoader(CanonicalPairDataset(frame, graphs), num_workers=0, collate_fn=make_collate(config.max_nodes),\n                      generator=torch.Generator().manual_seed(config.seed + 9187),\n                      pin_memory=torch.cuda.is_available(), **kwargs)\n\n\n@torch.no_grad()\ndef predict(model, frame, graphs, config, device, clock=None):\n    model.eval()\n    probabilities = []\n    for left, right, labels in _loader(frame, graphs, config):\n        if clock is not None:\n            clock.check()\n        with torch.autocast(device_type=device.type, enabled=config.amp and device.type == "cuda"):\n            logits, _ = model(left.to(device), right.to(device), None)\n        probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())\n    return np.concatenate(probabilities)\n\n\ndef load_pair_data(clean_data: Path, *, require_disjoint: bool = True):\n    metadata_path = clean_data / "metadata.json"\n    manifest = json.loads(metadata_path.read_text(encoding="utf-8"))\n    alignment = manifest.get("ddg_ast_alignment")\n    if alignment:\n        verify_alignment(clean_data, alignment)\n    if require_disjoint:\n        if manifest.get("protocol") != "revision_problem_disjoint_v1":\n            raise ValueError("Revision runs require the audited problem-disjoint bundle. Build it with revision.prepare first.")\n        if any(n for values in manifest.get("overlap", {}).values() for n in values.values()):\n            raise ValueError("Bundle overlap audit is nonzero")\n        if not manifest.get("overlap"):\n            raise ValueError("Bundle has no disjointness audit")\n    else:\n        if manifest.get("protocol") != "revision_reference_audited_v1":\n            raise ValueError("Reference runs require an audited derived benchmark, not an unfiltered original release")\n        if not manifest.get("overlap") or any(n for group in manifest["overlap"].values() for n in group.values()):\n            raise ValueError("Reference bundle overlap audit is absent or nonzero")\n    corpus = CleanDataCorpus(clean_data)\n    frames = {name: corpus.pairs(name) for name in ("train", "valid", "test")}\n    for name, frame in frames.items():\n        if set(frame.label.unique()) != {0, 1}:\n            raise ValueError(f"{name}: both labels are required")\n    digest = hashlib.sha256()\n    import gzip\n    opener = gzip.open if _is_gzip_file(corpus.pairs_path) else open\n    with opener(corpus.pairs_path, "rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    data_hash = fingerprint({"manifest": manifest, "pairs_file_sha256": digest.hexdigest()})\n    return corpus, frames, data_hash\n\n\ndef prepare_data(clean_data: Path, config: RunConfig, *, require_disjoint: bool = True):\n    corpus, frames, data_hash = load_pair_data(clean_data, require_disjoint=require_disjoint)\n    graphs = corpus.load_graphs(pair_code_ids(*frames.values()), max_nodes=config.max_nodes)\n    for name, frame in frames.items():\n        usable = frame.left_id.isin(graphs) & frame.right_id.isin(graphs)\n        if not usable.all():\n            raise ValueError(f"{name}: {int((~usable).sum())} pairs missing graph endpoints; rebuild common support before comparing arms")\n    return frames, graphs, data_hash\n\n\ndef train_session(frames: dict, graphs: dict, config: RunConfig, model_config: RevisionModelConfig,\n                  losses: dict, work_dir: Path, *, data_hash: str, resume: Path | None = None,\n                  clock=None, model_factory=RevisionPairModel, experiment_label: str = "unspecified",\n                  data_group_hash: str | None = None, loss_function=None,\n                  allow_numerical_upgrade: bool = False) -> dict:\n    """Train one arm/seed for one session. Reading test metrics is a separate step."""\n    clock = clock or SessionClock(config)\n    accounted_at = time.monotonic()\n    work_dir.mkdir(parents=True, exist_ok=True)\n    identity = run_identity(config, asdict(model_config), losses, data_hash)\n    # Code changes invalidate optimizer continuation; never silently mix models.\n    from .versioning import training_implementation_hash\n    from .numerics import (POLICY, NumericalStepFailure, can_upgrade, optimizer_group,\n                           validate_checkpoint_numerics)\n    source_hash = training_implementation_hash()\n    identity = fingerprint({"run": identity, "implementation": source_hash, "arm": experiment_label})\n    progress = work_dir / "progress.pt"\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    random.seed(config.seed); np.random.seed(config.seed); torch.manual_seed(config.seed)\n    if device.type == "cuda":\n        torch.cuda.manual_seed_all(config.seed)\n    model = model_factory(model_config).to(device)\n    loss_function = loss_function or getattr(model, "compute_loss", canonical_spectra_loss)\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),\n                                 lr=config.learning_rate, weight_decay=config.weight_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=config.amp and device.type == "cuda")\n    state = {"identity": identity, "epoch": 1, "batch": 0, "history": [], "bad_epochs": 0,\n             "best_score": None, "best_model": None, "best_threshold": None,\n             "seen": 0, "loss_sum": 0.0, "optimizer_steps": 0, "complete": False,\n             "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),\n             "total_parameters": sum(p.numel() for p in model.parameters()),\n             "training": asdict(config), "model_config": asdict(model_config), "losses": losses,\n             "data_hash": data_hash, "implementation_hash": source_hash,\n             "data_group_hash": data_group_hash or data_hash,\n             "arm": experiment_label,\n             "numerical_policy": POLICY, "numerical_events": [], "lineage": [],\n             "training_validation_wall_seconds": 0.0,\n             "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",\n             "device": str(device), "torch_version": str(torch.__version__)}\n    checkpoint_path = progress if progress.exists() else resume\n    if checkpoint_path is not None:\n        loaded = _load_checkpoint(checkpoint_path, device)\n        if loaded["identity"] != identity:\n            old_hash = loaded.get("implementation_hash")\n            old_identity = fingerprint({"run": run_identity(config, asdict(model_config), losses, data_hash),\n                                        "implementation": old_hash, "arm": experiment_label})\n            if not (allow_numerical_upgrade and can_upgrade(old_hash) and loaded["identity"] == old_identity):\n                raise ValueError("Resume identity mismatch: data, seed, architecture, losses, protocol or implementation changed. "\n                                 "Approved numerical-only continuation requires ALLOW_NUMERICAL_UPGRADE=True.")\n            validate_checkpoint_numerics(loaded)\n            loaded.setdefault("lineage", []).append({"from_identity": loaded["identity"],\n                "from_implementation_hash": old_hash, "to_identity": identity,\n                "to_implementation_hash": source_hash, "policy": POLICY,\n                "epoch": loaded["epoch"], "batch": loaded["batch"],\n                "reason": "Explicit approved AMP-overflow fix; architecture/data/hyperparameters unchanged"})\n            loaded.update(identity=identity, implementation_hash=source_hash)\n            print(json.dumps({"status": "approved_numerical_upgrade", "lineage": loaded["lineage"][-1]}), flush=True)\n        state = loaded\n        state.setdefault("numerical_events", [])\n        state.setdefault("lineage", [])\n        state["numerical_policy"] = POLICY\n        if state.get("stop_reason") == "nonfinite_after_fp32":\n            state["stop_reason"] = None\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"])\n        restore_rng(state["rng"])\n    now = time.monotonic()\n    state["session_setup_wall_seconds"] = state.get("session_setup_wall_seconds", 0.0) + max(\n        0.0, now - getattr(clock, "started", accounted_at))\n    accounted_at = now  # model loading/data preparation are setup, not training compute\n    last_checkpoint = time.monotonic()\n\n    def save(status):\n        nonlocal last_checkpoint, accounted_at\n        now = time.monotonic()\n        state["training_validation_wall_seconds"] += now - accounted_at\n        accounted_at = now\n        state["model"] = _cpu_state(model)\n        state["optimizer"] = optimizer.state_dict()\n        state["scaler"] = scaler.state_dict()\n        state["rng"] = rng_state()\n        atomic_save(state, progress)\n        summary = {k: state[k] for k in ("identity", "epoch", "batch", "optimizer_steps", "complete", "history", "trainable_parameters")}\n        summary.update({key: state.get(key) for key in ("best_epoch", "best_valid", "stop_reason", "arm", "training",\n                                                        "model_config", "losses", "data_hash", "data_group_hash",\n                                                        "implementation_hash", "gpu", "training_validation_wall_seconds")})\n        summary.update({key: state[key] for key in ("numerical_policy", "numerical_events", "lineage")})\n        summary["session_setup_wall_seconds"] = state["session_setup_wall_seconds"]\n        summary.update(status=status, checkpoint=str(progress), test_evaluated=False,\n                       continuation="Attach this output as a Kaggle input and set RESUME_CHECKPOINT to its progress.pt")\n        deadline = getattr(clock, "deadline", None)\n        summary["runtime_forecast"] = runtime_forecast(\n            config.max_epochs, len(state["history"]), state["training_validation_wall_seconds"],\n            None if deadline is None else deadline - time.monotonic())\n        (work_dir / "status.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n        print(json.dumps({"status": status, "runtime_forecast": summary["runtime_forecast"]}), flush=True)\n        last_checkpoint = time.monotonic()\n        return summary\n\n    train = frames["train"]\n    positive = int(train.label.sum())\n    positive_weight = min(10.0, (len(train) - positive) / max(1, positive))\n    accumulation = config.effective_batch_size // config.batch_size\n    total_batches = math.ceil(len(train) / config.batch_size)\n    def numerical_event(event):\n        record = {"epoch": state["epoch"], "batch": state["batch"],\n                  "optimizer_steps": state["optimizer_steps"], **event}\n        state["numerical_events"].append(record)\n        print(json.dumps({"status": "numerical_recovery", **record}), flush=True)\n\n    try:\n        while state["epoch"] <= config.max_epochs and not state["complete"]:\n            clock.check()\n            model.train(); optimizer.zero_grad(set_to_none=True)\n            batches = _batches(len(train), config.batch_size, config.seed, state["epoch"], state["batch"])\n            loader = iter(_loader(train, graphs, config, batches))\n            while state["batch"] < total_batches:\n                clock.check()\n                group_size = min(accumulation, total_batches - state["batch"])\n                group = [next(loader) for _ in range(group_size)]\n                contributions, recovery = optimizer_group(model, optimizer, scaler, group,\n                    config=config, device=device, losses=losses, loss_function=loss_function,\n                    positive_class_weight=positive_weight, clock=clock, on_event=numerical_event)\n                for weighted_loss, examples in contributions:\n                    state["loss_sum"] += weighted_loss\n                    state["seen"] += examples\n                    state["batch"] += 1\n                state["optimizer_steps"] += 1\n                if recovery or time.monotonic() - last_checkpoint >= config.checkpoint_minutes * 60:\n                    save("training")\n            # Persist the epoch before validation: validation can be retried on\n            # resume without repeating updates if the session ends here.\n            save("validation_pending")\n            p = predict(model, frames["valid"], graphs, config, device, clock)\n            threshold, result = choose_threshold(frames["valid"].label.to_numpy(), p, config.selection_metric)\n            score = (result[config.selection_metric], result["F1"] if config.selection_metric == "Accuracy" else result["Accuracy"])\n            if state["best_score"] is None or score > tuple(state["best_score"]):\n                state.update(best_score=score, best_model=_cpu_state(model), best_threshold=threshold,\n                             best_epoch=state["epoch"], best_valid=result, bad_epochs=0)\n            else:\n                state["bad_epochs"] += 1\n            state["history"].append({"epoch": state["epoch"], "train_loss": state["loss_sum"] / max(1, state["seen"]),\n                                     "valid": result, "optimizer_steps": state["optimizer_steps"]})\n            print(json.dumps(state["history"][-1]), flush=True)\n            converged = state["epoch"] >= config.min_epochs and state["bad_epochs"] >= config.patience\n            state["epoch"] += 1; state["batch"] = 0; state["seen"] = 0; state["loss_sum"] = 0.0\n            state["complete"] = converged or state["epoch"] > config.max_epochs\n            state["stop_reason"] = "validation_patience" if converged else "maximum_epochs" if state["complete"] else None\n            save("complete" if state["complete"] else "training")\n    except SessionBudgetReached:\n        return save("incomplete_session_budget")\n    except NumericalStepFailure as error:\n        state["stop_reason"] = "nonfinite_after_fp32"\n        summary = save("failed_nonfinite")\n        diagnostic = {"epoch": state["epoch"], "batch": state["batch"],\n                      "optimizer_steps": state["optimizer_steps"], "identity": state["identity"],\n                      **error.diagnostic}\n        (work_dir / "numerical_failure.json").write_text(json.dumps(diagnostic, indent=2), encoding="utf-8")\n        raise FloatingPointError(f"Non-finite values persisted in FP32. Safe pre-group checkpoint: {summary[\'checkpoint\']}; "\n                                 f"diagnostics: {work_dir / \'numerical_failure.json\'}") from error\n    return save("complete")\n\n\ndef evaluate_final(checkpoint: Path, frames: dict, graphs: dict, config: RunConfig, output: Path,\n                   *, expected_data_hash: str, clock=None, model_factory=RevisionPairModel,\n                   model_config_factory=RevisionModelConfig) -> dict:\n    """Explicit release of test results after all model/protocol decisions are fixed."""\n    output.mkdir(parents=True, exist_ok=True)\n    state = _load_checkpoint(checkpoint, "cpu")\n    if not state["complete"] or state["best_model"] is None:\n        raise ValueError("Training is incomplete; resume it before releasing test metrics")\n    if state["data_hash"] != expected_data_hash:\n        raise ValueError("Final-evaluation data differs from the training protocol")\n    if state["training"] != asdict(config):\n        # Session duration may change across continuation jobs; scientific\n        # settings and identity remain checked during training continuation.\n        old, new = dict(state["training"]), asdict(config)\n        for key in ("session_hours", "reserve_minutes", "checkpoint_minutes"):\n            old.pop(key, None); new.pop(key, None)\n        if old != new:\n            raise ValueError("Final-evaluation configuration mismatch")\n    result_path = output / "final_test.json"\n    if result_path.exists():\n        result = json.loads(result_path.read_text(encoding="utf-8"))\n        if result["identity"] != state["identity"]:\n            raise ValueError("Existing test result belongs to a different experiment")\n        return result\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model_factory(model_config_factory(**state["model_config"])).to(device)\n    model.load_state_dict(state["best_model"])\n    started = time.monotonic()\n    try:\n        p = predict(model, frames["test"], graphs, config, device, clock)\n    except SessionBudgetReached:\n        return {"identity": state["identity"], "status": "final_test_pending_session_budget",\n                "complete": True, "test_evaluated": False, "checkpoint": str(checkpoint)}\n    result = {"identity": state["identity"], "best_epoch": state["best_epoch"],\n              "arm": state["arm"], "seed": config.seed,\n              "data_hash": state["data_hash"], "implementation_hash": state["implementation_hash"],\n              "data_group_hash": state["data_group_hash"],\n              "model_config": state["model_config"], "losses": state["losses"],\n              "pair_counts": {name: len(frame) for name, frame in frames.items()},\n              "stop_reason": state["stop_reason"], "test": metrics(frames["test"].label.to_numpy(), p, state["best_threshold"]),\n              "inference_seconds": time.monotonic() - started,\n              "trainable_parameters": state["trainable_parameters"], "protocol": state["training"]}\n    result["training_validation_wall_seconds"] = state["training_validation_wall_seconds"]\n    result["session_setup_wall_seconds"] = state.get("session_setup_wall_seconds", 0.0)\n    result["gpu"] = state["gpu"]\n    result.update({key: state.get(key) for key in ("numerical_policy", "numerical_events", "lineage")})\n    predictions = frames["test"].copy()\n    predictions["probability"] = p\n    predictions["threshold"] = state["best_threshold"]\n    predictions.to_csv(output / "test_predictions.csv.gz", index=False, compression="gzip")\n    by_language = []\n    if "configuration_id" in predictions:\n        for name, group in predictions.groupby("configuration_id"):\n            by_language.append({"configuration_id": name, **metrics(group.label, group.probability, state["best_threshold"])})\n    pd.DataFrame(by_language).to_csv(output / "test_by_language.csv", index=False)\n    # Match the persisted JSON schema on the first call too (e.g. tuple fields\n    # in model configuration become lists), so cached evaluations are identical.\n    result = json.loads(json.dumps(result))\n    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")\n    return result\n', 'transfer.py': '"""Matched data/step transfer controls; target-language participation is explicit."""\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom .protocol import fingerprint, matched_training_pairs\nfrom .transfer_matrix import all_configurations, bucket\n\n\n# Prespecified Java -> Python case study; S denotes C#, C denotes C++.\n# All five conditions share the SAME Python-Python test rows. Only the three\n# bridge conditions match target-containing training/validation label budgets.\nFOCUSED_QUOTAS = {\n    "focused|source_only": ({"J-J": 6000}, {"J-J": 1200}),\n    "focused|direct": ({"J-J": 4500, "J-P": 1500}, {"J-J": 900, "J-P": 300}),\n    "focused|two_hop": ({"J-J": 2250, "J-C": 2250, "C-P": 1500},\n                        {"J-J": 450, "J-C": 450, "C-P": 300}),\n    "focused|three_hop": ({"J-J": 1500, "J-C": 1500, "C-S": 1500, "S-P": 1500},\n                          {"J-J": 300, "J-C": 300, "C-S": 300, "S-P": 300}),\n    "focused|target_reference": ({"P-P": 6000}, {"P-P": 1200}),\n    "focused|two_hop_reinforced": ({"J-J": 1500, "J-C": 1500, "C-C": 1500, "C-P": 1500},\n                                  {"J-J": 300, "J-C": 300, "C-C": 300, "C-P": 300}),\n}\n\n\ndef _bucket_codes(bucket):\n    from .transfer_matrix import bucket as dataset_bucket\n    return dataset_bucket(*bucket.split("-")) if "-" in bucket else bucket\n\n\ndef sample_quotas(frame, quotas, seed):\n    """Exactly the stated quota, balanced within each language-pair bucket."""\n    parts = []\n    for bucket, count in sorted(quotas.items()):\n        if count <= 0 or count % 2:\n            raise ValueError("Every per-bucket quota must be positive and even")\n        code = _bucket_codes(bucket)\n        for label in (0, 1):\n            group = frame[frame.configuration_id.eq(code) & frame.label.eq(label)]\n            if len(group) < count // 2:\n                raise ValueError(f"Insufficient {bucket}/{label}: need {count // 2}, have {len(group)}")\n            parts.append(group.sample(count // 2, random_state=seed))\n    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)\n\n\ndef _selected_hash(frames):\n    # Bind identity to actual sampled support, not merely the quota description.\n    return fingerprint({split: frame[["pair_id", "left_id", "right_id", "label"]].to_dict("records")\n                        for split, frame in frames.items()})\n\n\ndef _focused_frames(frames, key, budget, seed):\n    if budget != 6000:\n        raise ValueError("The prespecified focused design requires exactly 6000 training pairs")\n    train_quotas, valid_quotas = FOCUSED_QUOTAS[key]\n    selected = {"train": sample_quotas(frames["train"], train_quotas, seed),\n                "valid": sample_quotas(frames["valid"], valid_quotas, seed + 1),\n                "test": frames["test"][frames["test"].configuration_id.eq(bucket("P", "P"))].reset_index(drop=True)}\n    if set(selected["test"].label.unique()) != {0, 1}:\n        raise ValueError("Focused transfer test requires both labels")\n    exposure = lambda quotas: sum(n for bucket, n in quotas.items() if "P" in bucket.split("-"))\n    metadata = {"key": key, "train_buckets": [_bucket_codes(b) for b in train_quotas], "test_bucket": bucket("P", "P"),\n                "train_quotas": train_quotas, "valid_quotas": valid_quotas,\n                "train_pairs": 6000, "valid_pairs": 1200,\n                "target_containing_train_pairs": exposure(train_quotas),\n                "target_containing_valid_pairs": exposure(valid_quotas),\n                "target_language_supervision": "none in task training/validation" if "source_only" in key\n                                                else "target-containing task labels",\n                "interpretation": "Java-to-Python case study; pretraining exposure is not excluded; "\n                                  "bridge language composition changes with path length",\n                "source_problem_disjoint": True, "selected_pairs_hash": _selected_hash(selected)}\n    return selected, metadata\n\n\ndef select_transfer_frames(frames: dict, key: str, budget: int, seed: int) -> tuple[dict, dict]:\n    if key in FOCUSED_QUOTAS:\n        return _focused_frames(frames, key, budget, seed)\n    configurations = {c.key: c for c in all_configurations()}\n    if key not in configurations:\n        raise ValueError(f"Unknown transfer key {key}; inspect transfer_plan.csv")\n    configuration = configurations[key]\n    if configuration.table != "table7_within" and configuration.test_bucket in configuration.train_buckets:\n        raise ValueError("Target pair configuration must be held out")\n    train = frames["train"][frames["train"].configuration_id.isin(configuration.train_buckets)]\n    train = matched_training_pairs(train, configuration.train_buckets, budget, seed)\n    valid = frames["valid"][frames["valid"].configuration_id.isin(configuration.train_buckets)].copy()\n    test = frames["test"][frames["test"].configuration_id.eq(configuration.test_bucket)].copy()\n    if any(frame.empty or set(frame.label.unique()) != {0, 1} for frame in (train, valid, test)):\n        raise ValueError("Every transfer partition must have both labels")\n    metadata = {"key": key, "train_buckets": configuration.train_buckets,\n                "test_bucket": configuration.test_bucket, "train_pairs": len(train),\n                "target_language_supervision": "bridge cross-language labels" if configuration.path else\n                                               "same-language endpoint labels",\n                "interpretation": "held-out pair configuration; target language is not unseen",\n                "source_problem_disjoint": True}\n    for column in ("left_language", "right_language"):\n        if column in train:\n            metadata[f"{column}_counts"] = train[column].value_counts().to_dict()\n    selected = {"train": train.reset_index(drop=True), "valid": valid.reset_index(drop=True),\n                "test": test.reset_index(drop=True)}\n    metadata["selected_pairs_hash"] = _selected_hash(selected)\n    return selected, metadata\n\n\ndef write_focused_plan(path):\n    rows = [{"key": key, "train_quotas": str(train), "valid_quotas": str(valid),\n             "test_bucket": "P-P", "train_pairs": sum(train.values()), "valid_pairs": sum(valid.values()),\n             "target_containing_train_pairs": sum(n for b, n in train.items() if "P" in b.split("-")),\n             "target_containing_valid_pairs": sum(n for b, n in valid.items() if "P" in b.split("-")),\n             "epochs": 30, "optimizer_steps": 1410,\n             "optional": key.endswith("reinforced")}\n            for key, (train, valid) in FOCUSED_QUOTAS.items()]\n    pd.DataFrame(rows).to_csv(path, index=False)\n\n\ndef write_plan(path):\n    rows = [{"key": c.key, "target": c.target, "path": "->".join(c.path),\n             "train_buckets": ",".join(c.train_buckets), "test_bucket": c.test_bucket,\n             "reinforcement": c.reinforcement, "matched_train_pairs": 6000,\n             "epochs": 30, "optimizer_steps_at_batch32_effective128": 1410}\n            for c in all_configurations()]\n    pd.DataFrame(rows).to_csv(path, index=False)\n', 'transfer_matrix.py': '"""The RQ3 experiment matrix: every training configuration for Tables 7-10.\n\nA configuration names the language buckets a model trains on and the bucket it\nis evaluated on. Nothing here touches a model or a dataset, so the matrix can\nbe inspected, counted, and diffed without a GPU.\n\nLanguage codes follow the paper: J=java, P=python, C=cpp, S=csharp.\nBucket ids match ``configuration_id`` in the CodeNet clean-data pairs file:\nsame-language buckets are the bare language ("java"), cross-language buckets\njoin two languages in the export\'s own fixed order ("python_java").\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom itertools import permutations\n\n# Paper symbol -> clean-data language name.\nLANGUAGES: dict[str, str] = {"J": "java", "P": "python", "C": "cpp", "S": "csharp"}\nSYMBOLS: tuple[str, ...] = ("J", "P", "C", "S")\n\n# The export writes each cross-language bucket under one fixed ordering only,\n# so the reverse spelling must map onto the same bucket id.\n_CROSS_BUCKETS: dict[frozenset[str], str] = {\n    frozenset({"java", "python"}): "python_java",\n    frozenset({"java", "cpp"}): "java_cpp",\n    frozenset({"java", "csharp"}): "java_csharp",\n    frozenset({"python", "cpp"}): "python_cpp",\n    frozenset({"python", "csharp"}): "python_csharp",\n    frozenset({"cpp", "csharp"}): "cpp_csharp",\n}\n\n\ndef bucket(left: str, right: str) -> str:\n    """Return the clean-data ``configuration_id`` for a language pair."""\n    first, second = LANGUAGES[left], LANGUAGES[right]\n    if first == second:\n        return first\n    return _CROSS_BUCKETS[frozenset({first, second})]\n\n\n@dataclass(frozen=True)\nclass Configuration:\n    """One trainable setup: which buckets to train on, which bucket to test."""\n\n    table: str\n    name: str\n    train_buckets: tuple[str, ...]\n    test_bucket: str\n    target: str\n    path: tuple[str, ...] = ()\n    reinforcement: str = ""\n    metadata: dict = field(default_factory=dict)\n\n    @property\n    def key(self) -> str:\n        """Stable identifier used for result rows and resume checkpoints."""\n        return f"{self.table}|{self.name}"\n\n\ndef table7_within() -> list[Configuration]:\n    """Within-language reference: train L-L, test L-L."""\n    return [\n        Configuration(\n            table="table7_within",\n            name=f"{symbol}-{symbol}",\n            train_buckets=(bucket(symbol, symbol),),\n            test_bucket=bucket(symbol, symbol),\n            target=symbol,\n            metadata={"setting": "within_language"},\n        )\n        for symbol in SYMBOLS\n    ]\n\n\ndef table7_cross() -> list[Configuration]:\n    """Cross-language generalization with no cross-language supervision.\n\n    Training sees both endpoint languages only as same-language pairs; the\n    unseen mixed-language bucket is the test target.\n    """\n    configurations = []\n    for index, left in enumerate(SYMBOLS):\n        for right in SYMBOLS[index + 1:]:\n            configurations.append(\n                Configuration(\n                    table="table7_cross",\n                    name=f"{left}-{right}",\n                    train_buckets=(bucket(left, left), bucket(right, right)),\n                    test_bucket=bucket(left, right),\n                    target=f"{left}{right}",\n                    metadata={"setting": "cross_language_no_supervision"},\n                )\n            )\n    return configurations\n\n\ndef table8() -> list[Configuration]:\n    """Length-1 bridges X1 -> Y: train X1-X1 and X1-Y, test Y-Y."""\n    configurations = []\n    for target in SYMBOLS:\n        for source in SYMBOLS:\n            if source == target:\n                continue\n            configurations.append(\n                Configuration(\n                    table="table8",\n                    name=f"{source}->{target}",\n                    train_buckets=(bucket(source, source), bucket(source, target)),\n                    test_bucket=bucket(target, target),\n                    target=target,\n                    path=(source, target),\n                    reinforcement="none",\n                    metadata={"bridge_length": 1},\n                )\n            )\n    return configurations\n\n\n# Reinforcement variants add same-language buckets for the intermediate\n# languages on top of the mandatory path.\n_TABLE9_VARIANTS: tuple[tuple[str, tuple[int, ...]], ...] = (\n    ("none", ()),\n    ("X2-X2", (1,)),\n)\n_TABLE10_VARIANTS: tuple[tuple[str, tuple[int, ...]], ...] = (\n    ("none", ()),\n    ("X2-X2", (1,)),\n    ("X3-X3", (2,)),\n    ("X2-X2+X3-X3", (1, 2)),\n)\n\n\ndef _bridge_configurations(\n    table: str, path_length: int, variants: tuple[tuple[str, tuple[int, ...]], ...]\n) -> list[Configuration]:\n    """Build every distinct-language path of the requested length.\n\n    A path (X1, ..., Xn, Y) trains on X1-X1 plus each consecutive hop, so the\n    target language is reachable only through the chain.\n    """\n    configurations = []\n    for target in SYMBOLS:\n        others = [symbol for symbol in SYMBOLS if symbol != target]\n        for intermediates in permutations(others, path_length):\n            path = (*intermediates, target)\n            mandatory = [bucket(path[0], path[0])]\n            mandatory += [bucket(path[index], path[index + 1]) for index in range(len(path) - 1)]\n            for label, reinforce_indices in variants:\n                extra = [bucket(path[index], path[index]) for index in reinforce_indices]\n                configurations.append(\n                    Configuration(\n                        table=table,\n                        name=f"{\'->\'.join(path)}|{label}",\n                        # dict.fromkeys keeps first-seen order while removing\n                        # any bucket a reinforcement would duplicate.\n                        train_buckets=tuple(dict.fromkeys(mandatory + extra)),\n                        test_bucket=bucket(target, target),\n                        target=target,\n                        path=path,\n                        reinforcement=label,\n                        metadata={"bridge_length": path_length},\n                    )\n                )\n    return configurations\n\n\ndef table9() -> list[Configuration]:\n    """Length-2 bridges X1 -> X2 -> Y, with and without X2-X2."""\n    return _bridge_configurations("table9", 2, _TABLE9_VARIANTS)\n\n\ndef table10() -> list[Configuration]:\n    """Length-3 bridges X1 -> X2 -> X3 -> Y across four reinforcement modes."""\n    return _bridge_configurations("table10", 3, _TABLE10_VARIANTS)\n\n\ndef all_configurations() -> list[Configuration]:\n    return [*table7_within(), *table7_cross(), *table8(), *table9(), *table10()]\n\n\ndef summary() -> dict[str, int]:\n    counts: dict[str, int] = {}\n    for configuration in all_configurations():\n        counts[configuration.table] = counts.get(configuration.table, 0) + 1\n    counts["total"] = sum(counts.values())\n    return counts\n\n\nif __name__ == "__main__":\n    for table, count in summary().items():\n        print(f"{table:16s} {count:4d}")\n', 'transformer.py': '"""Fine-tuned text baselines with the same resumable selection/test protocol.\n\nC4-inspired adaptation: CodeBERT CLS vectors and task-masked in-batch contrastive\nlearning. We mask false negatives rather than repeat other tasks as upstream\ndoes; both pair directions and all known training endpoints can supply anchors.\nThis is an adaptation, not a reproduction of the original C4 experiment.\nSources: https://github.com/Chenning-Tao/C4 (model.py and run_con.py).\nQwen pooling: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B\nLoRA: https://huggingface.co/docs/peft/en/package_reference/lora\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nfrom dataclasses import dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .io_utils import replace_file\nfrom .protocol import fingerprint\nfrom .training import SessionBudgetReached\n\n\n@dataclass(frozen=True)\nclass TransformerConfig:\n    model_id: str = "microsoft/codebert-base"\n    revision: str | None = None\n    objective: str = "pair_bce"\n    pooling: str = "cls"\n    max_length: int = 512\n    lora_rank: int = 0\n    contrastive_weight: float = .1\n    gradient_checkpointing: bool = True\n    family: str = "transformer_pair"\n\n\n@dataclass\nclass TokenBatch:\n    inputs: dict\n    tasks: torch.Tensor\n\n    def to(self, device):\n        return TokenBatch({key: value.to(device) for key, value in self.inputs.items()}, self.tasks.to(device))\n\n\nclass _Pairs(Dataset):\n    def __init__(self, frame, index):\n        self.rows = [(index[str(a)], index[str(b)], float(y))\n                     for a, b, y in frame[["left_id", "right_id", "label"]].itertuples(index=False, name=None)]\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, row):\n        return self.rows[row]\n\n\nclass TokenCorpus:\n    def __init__(self, tokens, masks, ids, tasks):\n        self.tokens, self.masks = tokens, masks\n        self.index = {key: i for i, key in enumerate(ids)}\n        self.tasks = np.asarray(tasks, dtype=np.int64)\n\n    def _batch(self, rows):\n        rows = np.asarray(rows)\n        # Fresh writable arrays avoid mmap warnings and never alter the cache.\n        return TokenBatch({"input_ids": torch.tensor(np.asarray(self.tokens[rows]), dtype=torch.long),\n                           "attention_mask": torch.tensor(np.asarray(self.masks[rows]), dtype=torch.long)},\n                          torch.tensor(self.tasks[rows]))\n\n    def pair_loader(self, frame, config, batches=None):\n        def collate(rows):\n            a, b, y = zip(*rows)\n            return self._batch(a), self._batch(b), torch.tensor(y, dtype=torch.float32)\n        kwargs = {"batch_sampler": batches} if batches is not None else {"batch_size": config.batch_size}\n        return DataLoader(_Pairs(frame, self.index), collate_fn=collate, num_workers=0,\n                          generator=torch.Generator().manual_seed(config.seed + 9187), **kwargs)\n\n\ndef task_contrastive(vectors, tasks, scale=5.0):\n    """Supervised contrastive loss; no self positives or same-task negatives."""\n    vectors = F.normalize(vectors.float(), dim=-1)\n    scores = scale * vectors @ vectors.T\n    diagonal = torch.eye(len(tasks), device=tasks.device, dtype=torch.bool)\n    known = tasks.ge(0)\n    allowed = known[:, None] & known[None, :] & ~diagonal\n    positive = tasks[:, None].eq(tasks[None, :]) & allowed\n    anchors = positive.any(1) & (allowed & ~positive).any(1)\n    if not anchors.any():\n        return vectors.sum() * 0\n    logits = scores[anchors].masked_fill(~allowed[anchors], -torch.inf)\n    log_probability = logits - torch.logsumexp(logits, dim=-1, keepdim=True)\n    positive = positive[anchors]\n    selected = log_probability.masked_fill(~positive, 0)\n    return -(selected.sum(1) / positive.sum(1)).mean()\n\n\nclass TransformerPairModel(nn.Module):\n    def __init__(self, config, *, encoder=None):\n        super().__init__()\n        if config.objective not in {"pair_bce", "c4_adapted", "pair_bce_contrastive"}:\n            raise ValueError("Unknown transformer objective")\n        if config.pooling not in {"cls", "last_token"}:\n            raise ValueError("Pooling must be explicit")\n        self.settings = config\n        if encoder is None:\n            from transformers import AutoModel\n            if not config.revision:\n                raise ValueError("Resolve and pin the model commit before constructing the baseline")\n            encoder = AutoModel.from_pretrained(config.model_id, revision=config.revision,\n                                                trust_remote_code=False, torch_dtype=torch.float32)\n            if hasattr(encoder.config, "use_cache"):\n                encoder.config.use_cache = False\n            if config.lora_rank:\n                from peft import LoraConfig, get_peft_model\n                encoder = get_peft_model(encoder, LoraConfig(\n                    r=config.lora_rank, lora_alpha=2 * config.lora_rank, lora_dropout=.05,\n                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none"))\n                encoder.enable_input_require_grads()\n            if config.gradient_checkpointing:\n                encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = encoder\n        dimension = encoder.config.hidden_size\n        if config.objective != "c4_adapted":\n            self.classifier = nn.Sequential(nn.Linear(2 * dimension + 2, 256), nn.GELU(),\n                                            nn.Dropout(.1), nn.Linear(256, 1))\n\n    def encode(self, batch):\n        hidden = self.encoder(**batch.inputs).last_hidden_state\n        if self.settings.pooling == "cls":\n            pooled = hidden[:, 0]\n        else:\n            positions = torch.arange(hidden.shape[1], device=hidden.device)[None, :]\n            last = positions.masked_fill(~batch.inputs["attention_mask"].bool(), -1).max(1).values\n            pooled = hidden[torch.arange(len(hidden), device=hidden.device), last]\n        return F.normalize(pooled.float(), dim=-1)\n\n    def forward(self, left, right, temperature=None):\n        a, b = self.encode(left), self.encode(right)\n        cosine = (a * b).sum(-1)\n        if self.settings.objective == "c4_adapted":\n            logits = 5.0 * cosine\n        else:\n            features = torch.cat(((a - b).abs(), a * b, cosine[:, None], (a - b).norm(dim=-1, keepdim=True)), -1)\n            logits = self.classifier(features).squeeze(-1)\n        return logits, {"cosine": cosine, "vectors": torch.cat((a, b)),\n                        "tasks": torch.cat((left.tasks, right.tasks))}\n\n    def compute_loss(self, logits, auxiliary, labels, *, positive_class_weight=1.0, **unused):\n        if self.settings.objective == "c4_adapted":\n            loss = task_contrastive(auxiliary["vectors"], auxiliary["tasks"])\n            return loss, {"c4_adapted_contrastive": loss.detach()}\n        loss = F.binary_cross_entropy_with_logits(logits.float(), labels.float(),\n                                                  pos_weight=logits.new_tensor(positive_class_weight))\n        if self.settings.objective == "pair_bce_contrastive":\n            cosine = auxiliary["cosine"]\n            contrastive = (labels * (1 - cosine).square() + (1 - labels) * F.relu(cosine - .2).square()).mean()\n            loss = loss + self.settings.contrastive_weight * contrastive\n        return loss, {"training_loss": loss.detach()}\n\n\ndef prepare_tokens(corpus, frames, settings, output, *, clock, resume=None):\n    """Resumable, commit-pinned tokenization; never retains model activations."""\n    from transformers import AutoConfig, AutoTokenizer\n    from .data import pair_code_ids\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path = output / "tokens.json"\n    if resume and not manifest_path.exists():\n        for name in ("tokens.json", "tokens.npy", "masks.npy"):\n            shutil.copyfile(Path(resume) / name, output / name)\n    state = json.loads(manifest_path.read_text()) if manifest_path.exists() else None\n    clock.check()\n    revision = state["revision"] if state else settings.revision\n    resolved = AutoConfig.from_pretrained(settings.model_id, revision=revision, trust_remote_code=False)._commit_hash\n    settings = replace(settings, revision=resolved)\n    ids = sorted(pair_code_ids(*frames.values()))\n    identity = fingerprint({"model": settings.model_id, "revision": resolved, "length": settings.max_length,\n                            "whitespace_normalized": settings.objective == "c4_adapted",\n                            "codes": [(key, hashlib.sha256(corpus.code_texts[key].encode()).hexdigest()) for key in ids]})\n    if state and state["identity"] != identity:\n        raise ValueError("Token-cache identity mismatch")\n    tokenizer = AutoTokenizer.from_pretrained(settings.model_id, revision=resolved, trust_remote_code=False,\n                                              padding_side="left" if settings.pooling == "last_token" else "right")\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    if state is None:\n        np.lib.format.open_memmap(output / "tokens.npy", mode="w+", dtype=np.int32,\n                                  shape=(len(ids), settings.max_length)).flush()\n        np.lib.format.open_memmap(output / "masks.npy", mode="w+", dtype=np.uint8,\n                                  shape=(len(ids), settings.max_length)).flush()\n        state = {"identity": identity, "revision": resolved, "next_index": 0, "complete": False,\n                 "codes": len(ids), "truncated_codes": 0, "max_length": settings.max_length}\n    def save():\n        temporary = manifest_path.with_suffix(".tmp")\n        temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")\n        replace_file(temporary, manifest_path)\n    tokens = np.load(output / "tokens.npy", mmap_mode="r+")\n    masks = np.load(output / "masks.npy", mmap_mode="r+")\n    try:\n        while state["next_index"] < len(ids):\n            clock.check()\n            start, stop = state["next_index"], min(state["next_index"] + 256, len(ids))\n            texts = [corpus.code_texts[key] for key in ids[start:stop]]\n            if settings.objective == "c4_adapted":\n                texts = [" ".join(text.split()) for text in texts]\n            lengths = [len(row) for row in tokenizer(texts, truncation=False)["input_ids"]]\n            batch = tokenizer(texts, padding="max_length", truncation=True, max_length=settings.max_length)\n            tokens[start:stop] = batch["input_ids"]\n            masks[start:stop] = batch["attention_mask"]\n            tokens.flush(); masks.flush()\n            state.update(next_index=stop, complete=stop == len(ids))\n            state["truncated_codes"] += sum(length > settings.max_length for length in lengths)\n            save()\n    except SessionBudgetReached:\n        save()\n        return None, settings, state\n    tasks = {key: corpus.code_metadata.get(key, {}).get("problem_id") for key in ids}\n    train_ids = pair_code_ids(frames["train"])\n    vocabulary = {key: i for i, key in enumerate(sorted({str(tasks[key]) for key in train_ids if tasks[key] is not None}))}\n    task_ids = [vocabulary.get(str(tasks[key]), -1) if key in train_ids else -1 for key in ids]\n    if settings.objective == "c4_adapted" and any(task < 0 for key, task in zip(ids, task_ids) if key in train_ids):\n        raise ValueError("C4 adaptation requires known training problem IDs; do not infer them from test labels")\n    return TokenCorpus(tokens, masks, ids, task_ids), settings, state\n', 'versioning.py': '"""Training dependency fingerprint, independent of reporting utilities.\n\nKeep this list explicit: changing a model, data loader, sampler or training\npolicy must invalidate continuation. Reporting/plotting additions must not.\nThe versioning module itself is included so changes to this policy are visible.\n"""\nfrom pathlib import Path\n\nfrom .protocol import fingerprint\n\n\nTRAINING_SOURCES = (\n    "__init__.py", "versioning.py", "canonical_types.py", "data.py",\n    "alignment.py", "batching.py", "model.py", "protocol.py", "training.py",\n    "io_utils.py", "legacy.py", "transformer.py", "jobs.py", "transfer.py", "numerics.py",\n    "transfer_matrix.py",\n)\n\n\ndef training_implementation_hash(root=None):\n    root = Path(root) if root is not None else Path(__file__).parent\n    return fingerprint({name: (root / name).read_text(encoding="utf-8")\n                        for name in TRAINING_SOURCES})\n'}
_runtime = Path("/kaggle/working/revision_runtime")
_package = _runtime / "spectral_code" / "revision"
_package.mkdir(parents=True, exist_ok=True)
(_package.parent / "__init__.py").write_text("", encoding="utf-8")
for _name, _source in _sources.items():
    (_package / _name).write_text(_source, encoding="utf-8")
sys.path.insert(0, str(_runtime))
from spectral_code.revision.progress import install_progress
install_progress()  # tqdm for data loading, train/evaluation batches and token/embedding caches

_legacy_sources = {'core.py': '"""Core models for the repository\'s paper-faithful baseline adaptations.\n\nThe code deliberately accepts tensors derived from our exported AST/CFG/DDG\ngraphs.  It reproduces the defining algorithmic ideas of the cited methods,\nwithout claiming parser- or preprocessing-equivalence with upstream projects.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\n\nimport numpy as np\nimport torch\nfrom torch import Tensor, nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\ndef children_from_edges(num_nodes: int, rows: Sequence[int], cols: Sequence[int]) -> list[list[int]]:\n    """Return a cycle-safe ordered child list for an exported AST."""\n    children = [[] for _ in range(num_nodes)]\n    seen: set[tuple[int, int]] = set()\n    for parent, child in zip(rows, cols):\n        parent, child = int(parent), int(child)\n        edge = (parent, child)\n        if 0 <= parent < num_nodes and 0 <= child < num_nodes and parent != child and edge not in seen:\n            children[parent].append(child)\n            seen.add(edge)\n    return children\n\n\ndef postorder_depths(children: Sequence[Sequence[int]]) -> np.ndarray:\n    """Compute bottom-up levels, breaking malformed back-edges safely."""\n    n = len(children)\n    state = np.zeros(n, dtype=np.int8)\n    depth = np.zeros(n, dtype=np.int16)\n\n    def visit(node: int) -> int:\n        if state[node] == 2:\n            return int(depth[node])\n        if state[node] == 1:\n            return 0\n        state[node] = 1\n        child_depths = [visit(c) for c in children[node] if state[c] != 1]\n        depth[node] = 0 if not child_depths else min(255, 1 + max(child_depths))\n        state[node] = 2\n        return int(depth[node])\n\n    for index in range(n):\n        visit(index)\n    return depth\n\n\ndef left_child_right_sibling(children: Sequence[Sequence[int]]) -> tuple[np.ndarray, np.ndarray]:\n    """Convert an n-ary AST to the lossless left-child/right-sibling binary form."""\n    n = len(children)\n    left = np.full(n, -1, dtype=np.int64)\n    right = np.full(n, -1, dtype=np.int64)\n    for parent, child_list in enumerate(children):\n        clean = [int(c) for c in child_list if 0 <= int(c) < n and int(c) != parent]\n        if not clean:\n            continue\n        left[parent] = clean[0]\n        for current, sibling in zip(clean, clean[1:]):\n            right[current] = sibling\n    return left, right\n\n\ndef deckard_subtree_vectors(\n    node_types: Sequence[int],\n    children: Sequence[Sequence[int]],\n    vocab_size: int,\n    *,\n    min_nodes: int = 3,\n) -> tuple[np.ndarray, np.ndarray]:\n    """Construct DECKARD characteristic vectors for every eligible AST subtree.\n\n    Each vector is a count of AST node categories in a rooted subtree.  The\n    final coordinate stores subtree size, matching DECKARD\'s size-aware vector\n    grouping while allowing direct scoring of benchmark-supplied pairs.\n    """\n    n = len(node_types)\n    counts = np.zeros((n, vocab_size + 1), dtype=np.float32)\n    sizes = np.ones(n, dtype=np.int32)\n    levels = postorder_depths(children)\n    # The earlier direct transcription performed one Python-level dense-vector\n    # addition per AST edge. On CodeNet that is millions of tiny NumPy calls.\n    # ``np.add.at`` performs the identical child-to-parent reductions in a\n    # compiled loop, while retaining duplicate/multi-parent edges and the\n    # cycle guard used by the original implementation.\n    parent_rows: list[int] = []\n    child_rows: list[int] = []\n    for parent, child_list in enumerate(children):\n        for child in child_list:\n            child = int(child)\n            if 0 <= child < n and levels[child] < levels[parent]:\n                parent_rows.append(parent)\n                child_rows.append(child)\n    parents = np.asarray(parent_rows, dtype=np.intp)\n    child_indices = np.asarray(child_rows, dtype=np.intp)\n    node_levels = levels.astype(np.intp, copy=False)\n    for level in range(int(levels.max(initial=0)) + 1):\n        nodes = np.flatnonzero(node_levels == level)\n        if not len(nodes):\n            continue\n        tokens = np.asarray(node_types, dtype=np.int64)[nodes]\n        valid_tokens = (tokens >= 0) & (tokens < vocab_size)\n        counts[nodes[valid_tokens], tokens[valid_tokens]] += 1.0\n        if len(parents):\n            edge_mask = node_levels[parents] == level\n            if edge_mask.any():\n                np.add.at(counts, parents[edge_mask], counts[child_indices[edge_mask]])\n                np.add.at(sizes, parents[edge_mask], sizes[child_indices[edge_mask]])\n        counts[nodes, -1] = sizes[nodes]\n    keep = sizes >= min_nodes\n    return counts[keep], sizes[keep]\n\n\ndef deckard_pair_similarity(\n    left_vectors: np.ndarray,\n    left_sizes: np.ndarray,\n    right_vectors: np.ndarray,\n    right_sizes: np.ndarray,\n    *,\n    size_tolerance: float = 0.20,\n) -> float:\n    """DECKARD-style best subtree similarity for one supplied code pair."""\n    if not len(left_vectors) or not len(right_vectors):\n        return 0.0\n    # Full-data runs retain integer subtree-category counts compactly.  Promote\n    # only this pair\'s small retained vectors for the cosine computation so\n    # uint8 storage cannot overflow a matrix product.\n    left_vectors = np.asarray(left_vectors, dtype=np.float32)\n    right_vectors = np.asarray(right_vectors, dtype=np.float32)\n    left_sizes = np.asarray(left_sizes, dtype=np.float32)\n    right_sizes = np.asarray(right_sizes, dtype=np.float32)\n    # Pair evaluation replaces the original corpus-wide LSH candidate lookup;\n    # the characteristic vectors and size filter remain unchanged.  Score all\n    # retained subtree pairs in one BLAS multiplication instead of issuing up\n    # to 32 Python/NumPy matrix products per benchmark pair.\n    compatible = (\n        np.abs(left_sizes[:, None] - right_sizes[None, :])\n        / np.maximum(left_sizes[:, None], 1.0)\n    ) <= size_tolerance\n    if not compatible.any():\n        return 0.0\n    denominator = np.maximum(\n        np.linalg.norm(left_vectors, axis=1)[:, None] * np.linalg.norm(right_vectors, axis=1)[None, :],\n        1e-8,\n    )\n    similarities = (left_vectors @ right_vectors.T) / denominator\n    return float(similarities[compatible].max(initial=0.0))\n\n\ndef _masked_max(values: Tensor, mask: Tensor, dim: int) -> Tensor:\n    masked = values.masked_fill(~mask.unsqueeze(-1), torch.finfo(values.dtype).min)\n    result = masked.max(dim=dim).values\n    return torch.where(torch.isfinite(result), result, torch.zeros_like(result))\n\n\nclass RecursiveTreeEncoder(nn.Module):\n    """Bottom-up recursive tree encoder over an exported AST."""\n\n    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)\n        self.node = nn.Linear(embed_dim, hidden_dim)\n        self.child = nn.Linear(hidden_dim, hidden_dim, bias=False)\n\n    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor) -> Tensor:\n        mask = node_types.ne(0)\n        embedded = self.embedding(node_types)\n        hidden = torch.zeros((*node_types.shape, self.node.out_features), device=node_types.device, dtype=embedded.dtype)\n        valid_edges = parents.ge(0) & children.ge(0)\n        max_depth = int(depths.max().detach().cpu()) if depths.numel() else 0\n        for level in range(max_depth + 1):\n            aggregate = torch.zeros_like(hidden)\n            valid = valid_edges & depths.gather(1, children.clamp_min(0)).lt(level + 1)\n            child_index = children.clamp_min(0).unsqueeze(-1).expand(-1, -1, hidden.size(-1))\n            child_values = hidden.gather(1, child_index) * valid.unsqueeze(-1)\n            parent_index = parents.clamp_min(0).unsqueeze(-1).expand_as(child_values)\n            aggregate.scatter_add_(1, parent_index, child_values)\n            count = torch.zeros_like(hidden[..., :1])\n            count.scatter_add_(1, parents.clamp_min(0).unsqueeze(-1), valid.unsqueeze(-1).to(hidden.dtype))\n            candidate = torch.tanh(self.node(embedded) + self.child(aggregate / count.clamp_min(1.0)))\n            update = depths.eq(level) & mask\n            hidden = torch.where(update.unsqueeze(-1), candidate, hidden)\n        return hidden * mask.unsqueeze(-1)\n\n\nclass ASTNNEncoder(nn.Module):\n    """ASTNN block sequence: recursive subtree encoding, BiGRU, max pooling."""\n\n    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 100, code_dim: int = 200):\n        super().__init__()\n        self.tree = RecursiveTreeEncoder(vocab_size, embed_dim, hidden_dim)\n        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)\n        self.projection = nn.Linear(hidden_dim * 2, code_dim)\n\n    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor, statements: Tensor) -> Tensor:\n        nodes = self.tree(node_types, parents, children, depths)\n        statement_mask = statements.ge(0)\n        indices = statements.clamp_min(0).unsqueeze(-1).expand(-1, -1, nodes.size(-1))\n        blocks = nodes.gather(1, indices) * statement_mask.unsqueeze(-1)\n        sequence, _ = self.gru(blocks)\n        pooled = _masked_max(sequence, statement_mask, 1)\n        return torch.tanh(self.projection(pooled))\n\n\nclass RtvNNEncoder(nn.Module):\n    """Recursive tree/vector encoder with a reconstruction objective."""\n\n    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192, code_dim: int = 192):\n        super().__init__()\n        self.tree = RecursiveTreeEncoder(vocab_size, embed_dim, hidden_dim)\n        self.decoder = nn.Linear(hidden_dim, embed_dim)\n        self.output = nn.Linear(hidden_dim * 2, code_dim)\n\n    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor) -> tuple[Tensor, Tensor]:\n        nodes = self.tree(node_types, parents, children, depths)\n        mask = node_types.ne(0)\n        mean = (nodes * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        maximum = _masked_max(nodes, mask, 1)\n        code = torch.tanh(self.output(torch.cat([mean, maximum], dim=-1)))\n        target = self.tree.embedding(node_types).detach()\n        reconstruction = F.mse_loss(self.decoder(nodes)[mask], target[mask]) if mask.any() else nodes.sum() * 0\n        return code, reconstruction\n\n\nclass BinaryTreeLSTMEncoder(nn.Module):\n    """Binary Tree-LSTM over left-child/right-sibling AST conversion."""\n\n    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)\n        self.gates = nn.Linear(embed_dim + hidden_dim * 2, hidden_dim * 5)\n        self.hidden_dim = hidden_dim\n\n    def _encode_embedded(\n        self, embedded: Tensor, node_types: Tensor, left: Tensor, right: Tensor, depths: Tensor\n    ) -> tuple[Tensor, Tensor]:\n        """Evaluate the bottom-up tree recurrence for already embedded nodes."""\n        batch, nodes = node_types.shape\n        h = embedded.new_zeros(batch, nodes, self.hidden_dim)\n        c = embedded.new_zeros(batch, nodes, self.hidden_dim)\n        active_nodes = node_types.ne(0)\n        max_depth = int(depths.max().detach().cpu()) if depths.numel() else 0\n        for level in range(max_depth + 1):\n            # Only nodes at this level are written; evaluating the gates for the\n            # whole padded batch and discarding the rest costs the padded width\n            # at every level.  Selecting the level\'s rows first is the same\n            # recurrence on exactly the rows that survive the update.\n            rows, columns = (depths.eq(level) & active_nodes).nonzero(as_tuple=True)\n            if rows.numel() == 0:\n                continue\n            left_rows = left[rows, columns]\n            right_rows = right[rows, columns]\n            left_ok = left_rows.ge(0).unsqueeze(-1)\n            right_ok = right_rows.ge(0).unsqueeze(-1)\n            hl = h[rows, left_rows.clamp_min(0)] * left_ok\n            hr = h[rows, right_rows.clamp_min(0)] * right_ok\n            cl = c[rows, left_rows.clamp_min(0)] * left_ok\n            cr = c[rows, right_rows.clamp_min(0)] * right_ok\n            i, fl, fr, o, u = self.gates(\n                torch.cat([embedded[rows, columns], hl, hr], -1)\n            ).chunk(5, -1)\n            new_c = torch.sigmoid(i) * torch.tanh(u) + torch.sigmoid(fl) * cl + torch.sigmoid(fr) * cr\n            new_h = torch.sigmoid(o) * torch.tanh(new_c)\n            h = h.index_put((rows, columns), new_h)\n            c = c.index_put((rows, columns), new_c)\n        return h, c\n\n    def forward(self, node_types: Tensor, left: Tensor, right: Tensor, depths: Tensor) -> tuple[Tensor, Tensor]:\n        embedded = self.embedding(node_types)\n        # A deep, padded CodeNet AST otherwise retains one BxNxH activation\n        # graph per tree level. Checkpointing preserves the forward computation\n        # and recomputes it during backward, which is safe on a 16-GB Kaggle T4.\n        if self.training and torch.is_grad_enabled():\n            return checkpoint(\n                self._encode_embedded, embedded, node_types, left, right, depths,\n                use_reentrant=False,\n            )\n        return self._encode_embedded(embedded, node_types, left, right, depths)\n\n\nclass CDLHModel(nn.Module):\n    """Binary Tree-LSTM followed by a supervised continuous hash layer."""\n\n    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192, hash_bits: int = 32):\n        super().__init__()\n        self.tree_lstm = BinaryTreeLSTMEncoder(vocab_size, embed_dim, hidden_dim)\n        self.hash = nn.Linear(hidden_dim, hash_bits)\n\n    def encode(self, node_types: Tensor, left: Tensor, right: Tensor, depths: Tensor) -> Tensor:\n        hidden, _ = self.tree_lstm(node_types, left, right, depths)\n        mask = node_types.ne(0)\n        return torch.tanh(self.hash(_masked_max(hidden, mask, 1)))\n\n    @staticmethod\n    def loss(left_hash: Tensor, right_hash: Tensor, labels: Tensor, quantization_weight: float = 0.01) -> Tensor:\n        similarity = (left_hash * right_hash).mean(-1)\n        pair_loss = F.binary_cross_entropy_with_logits(4.0 * similarity, labels.float())\n        quantization = ((left_hash.abs() - 1.0) ** 2).mean() + ((right_hash.abs() - 1.0) ** 2).mean()\n        return pair_loss + quantization_weight * quantization\n\n\ndef relation_message(hidden: Tensor, edge_src: Tensor, edge_dst: Tensor, transform: nn.Module) -> Tensor:\n    """Batched sparse relation aggregation from padded edge-index tensors."""\n    valid = edge_src.ge(0) & edge_dst.ge(0)\n    src = edge_src.clamp_min(0)\n    dst = edge_dst.clamp_min(0)\n    src_hidden = hidden.gather(1, src.unsqueeze(-1).expand(-1, -1, hidden.size(-1)))\n    # Embeddings remain float32 under AMP while Linear outputs may be fp16 or\n    # bfloat16. scatter_add_ requires an exact dtype match between destination\n    # and source, so normalize the message tensor to the recurrent state dtype.\n    messages = (transform(src_hidden) * valid.unsqueeze(-1)).to(hidden.dtype)\n    output = torch.zeros_like(hidden)\n    output.scatter_add_(1, dst.unsqueeze(-1).expand_as(messages), messages)\n    counts = torch.zeros_like(hidden[..., :1])\n    counts.scatter_add_(1, dst.unsqueeze(-1), valid.unsqueeze(-1).to(hidden.dtype))\n    return output / counts.clamp_min(1.0)\n\n\nclass DeepSimEncoder(nn.Module):\n    """Control/data-flow semantic-matrix encoder for the exported graphs.\n\n    DeepSim\'s original semantic matrix combines control- and data-flow\n    information.  The project export cannot reproduce WALA instruction\n    categories across all four languages, but it can retain both graph views\n    and their structural roles rather than silently reducing the method to a\n    CFG-only encoder.\n    """\n\n    def __init__(self, vocab_size: int, numeric_dim: int = 12, embed_dim: int = 96, hidden_dim: int = 192, code_dim: int = 192):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)\n        self.self_proj = nn.Linear(embed_dim + numeric_dim, hidden_dim)\n        self.cfg_message = nn.Linear(hidden_dim, hidden_dim, bias=False)\n        self.sequence = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)\n        self.output = nn.Linear(hidden_dim * 2, code_dim)\n\n    def forward(self, node_types: Tensor, numeric: Tensor, cfg_src: Tensor, cfg_dst: Tensor) -> Tensor:\n        mask = node_types.ne(0)\n        semantic_matrix = torch.cat([self.embedding(node_types), numeric], -1)\n        semantic_hidden = self.self_proj(semantic_matrix)\n        hidden = torch.tanh(semantic_hidden + relation_message(semantic_hidden, cfg_src, cfg_dst, self.cfg_message))\n        sequence, _ = self.sequence(hidden)\n        return F.normalize(self.output(_masked_max(sequence, mask, 1)), dim=-1)\n\n\nclass FAASTGGNN(nn.Module):\n    """Relation-aware GGNN on our AST+CFG+DDG flow-augmented graph."""\n\n    def __init__(self, vocab_size: int, relations: int = 6, hidden_dim: int = 192, code_dim: int = 192, steps: int = 5):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)\n        self.transforms = nn.ModuleList(nn.Linear(hidden_dim, hidden_dim, bias=False) for _ in range(relations))\n        self.gru = nn.GRUCell(hidden_dim, hidden_dim)\n        self.gate = nn.Linear(hidden_dim, 1)\n        self.output = nn.Linear(hidden_dim, code_dim)\n        self.steps = steps\n\n    def propagate(self, hidden: Tensor, edge_src: Tensor, edge_dst: Tensor) -> Tensor:\n        # All six directed relations share the same padded edge layout.  Fuse\n        # their linear projections and scatter operations into two kernels.\n        # This is algebraically identical to summing ``relation_message`` for\n        # every relation, but avoids six Python/CUDA dispatch chains at each\n        # GGNN step (the principal full-CodeNet bottleneck).\n        relations = len(self.transforms)\n        weights = torch.stack([transform.weight for transform in self.transforms])\n        transformed = torch.einsum("bni,roi->brno", hidden, weights).to(hidden.dtype)\n        valid = edge_src.ge(0) & edge_dst.ge(0)\n        src = edge_src.clamp_min(0)\n        dst = edge_dst.clamp_min(0)\n        messages = transformed.gather(\n            2, src.unsqueeze(-1).expand(-1, -1, -1, hidden.size(-1)),\n        ) * valid.unsqueeze(-1)\n        relation_aggregate = hidden.new_zeros(hidden.size(0), relations, hidden.size(1), hidden.size(2))\n        relation_aggregate.scatter_add_(\n            2, dst.unsqueeze(-1).expand_as(messages), messages,\n        )\n        counts = hidden.new_zeros(hidden.size(0), relations, hidden.size(1), 1)\n        counts.scatter_add_(2, dst.unsqueeze(-1), valid.unsqueeze(-1).to(hidden.dtype))\n        aggregate = (relation_aggregate / counts.clamp_min(1.0)).sum(1)\n        return self.gru(aggregate.flatten(0, 1), hidden.flatten(0, 1)).view_as(hidden)\n\n    def pool(self, hidden: Tensor, mask: Tensor) -> Tensor:\n        weights = torch.sigmoid(self.gate(hidden)) * mask.unsqueeze(-1)\n        pooled = (weights * hidden).sum(1) / weights.sum(1).clamp_min(1e-6)\n        return F.normalize(self.output(pooled), dim=-1)\n\n    def forward(self, node_types: Tensor, edge_src: Tensor, edge_dst: Tensor) -> Tensor:\n        mask = node_types.ne(0)\n        hidden = self.embedding(node_types)\n        for _ in range(self.steps):\n            hidden = self.propagate(hidden, edge_src, edge_dst) * mask.unsqueeze(-1)\n        return self.pool(hidden, mask)\n\n\nclass FAASTGMN(FAASTGGNN):\n    """FA-AST GGNN with pair-aware cross-graph matching at every step."""\n\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        hidden_dim = self.embedding.embedding_dim\n        self.match_gru = nn.GRUCell(hidden_dim * 2, hidden_dim)\n\n    @staticmethod\n    def cross_attention(left: Tensor, right: Tensor, left_mask: Tensor, right_mask: Tensor) -> tuple[Tensor, Tensor]:\n        scale = left.size(-1) ** -0.5\n        scores = torch.bmm(left, right.transpose(1, 2)) * scale\n        left_scores = scores.masked_fill(~right_mask.unsqueeze(1), -1e4)\n        right_scores = scores.transpose(1, 2).masked_fill(~left_mask.unsqueeze(1), -1e4)\n        left_match = torch.bmm(torch.softmax(left_scores, -1), right)\n        right_match = torch.bmm(torch.softmax(right_scores, -1), left)\n        return left_match, right_match\n\n    def forward_pair(\n        self,\n        left_types: Tensor,\n        left_src: Tensor,\n        left_dst: Tensor,\n        right_types: Tensor,\n        right_src: Tensor,\n        right_dst: Tensor,\n    ) -> tuple[Tensor, Tensor]:\n        lm, rm = left_types.ne(0), right_types.ne(0)\n        left, right = self.embedding(left_types), self.embedding(right_types)\n        for _ in range(self.steps):\n            lp, rp = self.propagate(left, left_src, left_dst), self.propagate(right, right_src, right_dst)\n            lx, rx = self.cross_attention(lp, rp, lm, rm)\n            left = self.match_gru(torch.cat([lp, lp - lx], -1).flatten(0, 1), left.flatten(0, 1)).view_as(left)\n            right = self.match_gru(torch.cat([rp, rp - rx], -1).flatten(0, 1), right.flatten(0, 1)).view_as(right)\n            left, right = left * lm.unsqueeze(-1), right * rm.unsqueeze(-1)\n        return self.pool(left, lm), self.pool(right, rm)\n', 'notebook_runtime.py': '"""Self-contained Kaggle runtime for paper-faithful baseline adaptations.\n\nThis file is embedded into generated notebooks by\nthe standalone Kaggle notebooks. Keep imports absolute-free:\nthe core definitions are prepended to the notebook cell before this runtime.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport datetime\nimport gc\nimport gzip\nimport hashlib\nimport json\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\ntry:\n    children_from_edges\nexcept NameError:  # Direct local import; generated notebooks prepend core.py.\n    from spectral_code.faithful_graph_baselines.core import (\n        ASTNNEncoder,\n        CDLHModel,\n        DeepSimEncoder,\n        FAASTGGNN,\n        FAASTGMN,\n        RtvNNEncoder,\n        children_from_edges,\n        deckard_pair_similarity,\n        deckard_subtree_vectors,\n        left_child_right_sibling,\n        postorder_depths,\n    )\n\n\nMETHOD_CONFIGS = {\n    "deckard": {"name": "Deckard", "batch": None, "lr": None, "weight_decay": None},\n    # Faithful models run on one GPU because their int16 corpus buffers cannot\n    # be broadcast by NCCL.  CDLH\'s level-wise binary Tree-LSTM retains both\n    # trees\' recurrent states for backward, so it needs a substantially\n    # smaller batch than the other encoders to fit safely on a 16-GB T4.\n    "astnn": {"name": "ASTNN", "batch": 32, "lr": 1e-3, "weight_decay": 0.0},\n    "rtvnn": {"name": "RtvNN", "batch": 32, "lr": 5e-4, "weight_decay": 1e-4},\n    "cdlh": {"name": "CDLH", "batch": 16, "lr": 1e-3, "weight_decay": 0.0},\n    "deepsim": {"name": "DeepSim", "batch": 32, "lr": 5e-4, "weight_decay": 1e-3},\n    "fa_ast_ggnn": {"name": "FA-AST+GGNN", "batch": 8, "lr": 5e-4, "weight_decay": 1e-4},\n    "fa_ast_gmn": {"name": "FA-AST+GMN", "batch": 4, "lr": 5e-4, "weight_decay": 1e-4},\n}\n\n# A generated CodeNet notebook supplies conservative values in its header.\n# Other datasets retain the established defaults.\nMAX_NODES = int(globals().get("MAX_NODES", 256))\nMAX_EDGES = int(globals().get("MAX_EDGES", 512))\nMAX_STATEMENTS = int(globals().get("MAX_STATEMENTS", 64))\n# The paper uses directed flow-augmented AST relations.  Keep both directions\n# explicitly: collapsing them makes parent/control/data-flow messages\n# indistinguishable from their reverse relation.\nRELATIONS = ("ast_forward", "ast_reverse", "cfg_forward", "cfg_reverse", "ddg_forward", "ddg_reverse")\n\n\ndef _seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _open_text(path: Path):\n    with path.open("rb") as probe:\n        packed = probe.read(2) == b"\\x1f\\x8b"\n    return gzip.open(path, "rt", encoding="utf-8") if packed else path.open("r", encoding="utf-8")\n\n\ndef _kaggle_working_dir() -> Path:\n    return Path(globals().get("WORK_DIR_OVERRIDE", "/kaggle/working"))\n\n\ndef _kaggle_input_dir() -> Path:\n    return Path(globals().get("KAGGLE_INPUT_OVERRIDE", "/kaggle/input"))\n\n\ndef _extract_clean_artifacts_from_zips() -> list[Path]:\n    """Materialize the minimum required clean-data files from Kaggle ZIP inputs.\n\n    Kaggle datasets can be attached either as a folder or as the final clean-data\n    ZIP.  The native GNN notebooks already support both layouts; the faithful\n    baselines must do the same.  Extracting only pairs/graph records avoids\n    unpacking unrelated large source archives.\n    """\n    import zipfile\n\n    input_root = _kaggle_input_dir()\n    if not input_root.exists():\n        return []\n    required = {\n        "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp",\n        "graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp",\n    }\n    cache_root = _kaggle_working_dir() / "faithful_baseline_input_cache"\n    extracted_roots: list[Path] = []\n    for archive_path in input_root.rglob("*.zip"):\n        try:\n            with zipfile.ZipFile(archive_path) as archive:\n                members = [\n                    info for info in archive.infolist()\n                    if not info.is_dir() and Path(info.filename).name in required\n                ]\n                basenames = {Path(info.filename).name for info in members}\n                has_pairs = any(name.startswith("pairs.csv") for name in basenames)\n                has_graphs = any(name.startswith("graph_spectra.jsonl") for name in basenames)\n                if not (has_pairs and has_graphs):\n                    continue\n                digest = hashlib.sha1(str(archive_path).encode("utf-8")).hexdigest()[:12]\n                target = cache_root / f"{archive_path.stem}_{digest}"\n                marker = target / ".complete"\n                if not marker.exists():\n                    for info in members:\n                        # Reject unsafe archive member paths even though Kaggle\n                        # attachments are expected to be trusted.\n                        relative = Path(info.filename)\n                        if relative.is_absolute() or ".." in relative.parts:\n                            continue\n                        destination = target / relative\n                        destination.parent.mkdir(parents=True, exist_ok=True)\n                        with archive.open(info, "r") as source, destination.open("wb") as destination_file:\n                            shutil.copyfileobj(source, destination_file)\n                    marker.parent.mkdir(parents=True, exist_ok=True)\n                    marker.write_text("ok", encoding="utf-8")\n                extracted_roots.append(target)\n        except zipfile.BadZipFile:\n            continue\n    return extracted_roots\n\n\ndef _find_input(dataset_key: str, *names: str) -> Path:\n    input_root = _kaggle_input_dir()\n    roots = [input_root / dataset_key, input_root, Path(".")]\n    matches = []\n    for root in roots:\n        if not root.exists():\n            continue\n        for name in names:\n            matches.extend(path for path in root.rglob(name) if path.is_file())\n    if not matches:\n        for root in _extract_clean_artifacts_from_zips():\n            for name in names:\n                matches.extend(path for path in root.rglob(name) if path.is_file())\n    if not matches:\n        visible = [str(path.relative_to(input_root)) for path in input_root.rglob("*") if path.is_file()][:80] if input_root.exists() else []\n        raise FileNotFoundError(\n            f"Could not find {names} below the Kaggle inputs or an attached clean-data ZIP. "\n            "Attach the final clean_data folder/ZIP containing pairs.csv(.gz) and "\n            "graph_spectra.jsonl(.gz). Files currently visible: " + ", ".join(visible)\n        )\n    normalized_key = "".join(character for character in dataset_key.lower() if character.isalnum())\n    key_without_version = normalized_key.removesuffix("v3")\n\n    def rank(path: Path) -> tuple[int, int, int, str]:\n        normalized_path = "".join(character for character in str(path).lower() if character.isalnum())\n        dataset_penalty = 0 if (\n            normalized_key in normalized_path or key_without_version in normalized_path\n        ) else 1\n        return dataset_penalty, len(path.parts), len(path.name), str(path)\n\n    return sorted(set(matches), key=rank)[0]\n\n\ndef _metric_dict(labels, scores, threshold: float) -> dict:\n    labels = np.asarray(labels, dtype=np.int64)\n    predictions = (np.asarray(scores) >= threshold).astype(np.int64)\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        labels, predictions, average="binary", zero_division=0\n    )\n    return {\n        "P": float(precision),\n        "R": float(recall),\n        "F1": float(f1),\n        "Acc": float(accuracy_score(labels, predictions)),\n        "MacroF1": float(f1_score(labels, predictions, average="macro", zero_division=0)),\n        "BalancedAccuracy": float(balanced_accuracy_score(labels, predictions)),\n        "ROC_AUC": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else float("nan"),\n        "TP": int(((labels == 1) & (predictions == 1)).sum()),\n        "FP": int(((labels == 0) & (predictions == 1)).sum()),\n        "TN": int(((labels == 0) & (predictions == 0)).sum()),\n        "FN": int(((labels == 1) & (predictions == 0)).sum()),\n    }\n\n\ndef _choose_threshold(labels, scores) -> tuple[float, dict]:\n    scores = np.asarray(scores, dtype=np.float32)\n    candidates = np.unique(\n        np.concatenate([np.linspace(0.01, 0.99, 199), np.quantile(scores, np.linspace(0, 1, 201))])\n    )\n    best_threshold, best_metrics = 0.5, None\n    for threshold in candidates:\n        metrics = _metric_dict(labels, scores, float(threshold))\n        key = _selection_key(metrics, labels)\n        if best_metrics is None or key > _selection_key(best_metrics, labels):\n            best_threshold, best_metrics = float(threshold), metrics\n    return best_threshold, best_metrics\n\n\ndef _validation_selection(labels) -> dict[str, int | bool | str]:\n    """Use the declared metric; ``auto`` remains a legacy fallback only."""\n    labels = np.asarray(labels, dtype=np.int64)\n    positives = int((labels == 1).sum())\n    negatives = int((labels == 0).sum())\n    balanced = positives > 0 and positives == negatives\n    declared = globals().get("VALIDATION_SELECTION_METRIC", "auto")\n    if declared not in {"Accuracy", "F1", "auto"}:\n        raise ValueError("VALIDATION_SELECTION_METRIC must be Accuracy, F1, or auto")\n    return {\n        "balanced": balanced,\n        "metric": ("Accuracy" if balanced else "F1") if declared == "auto" else declared,\n        "positives": positives,\n        "negatives": negatives,\n    }\n\n\ndef _selection_key(metrics: dict, labels) -> tuple[float, float]:\n    context = _validation_selection(labels)\n    if context["metric"] == "Accuracy":\n        return float(metrics["Acc"]), float(metrics["F1"])\n    return float(metrics["F1"]), float(metrics["BalancedAccuracy"])\n\n\ndef _limit(frame: pd.DataFrame, split: str, maximum: int | None, seed: int) -> pd.DataFrame:\n    subset = frame[frame.split == split].copy()\n    if maximum is not None and len(subset) > maximum:\n        subset = subset.sample(maximum, random_state=seed)\n    return subset.reset_index(drop=True)\n\n\ndef _adjacency(layer: dict) -> dict:\n    return layer.get("adjacency", {}) if isinstance(layer, dict) else {}\n\n\ndef _load_raw_graphs(\n    path: Path, wanted: set[str], required_layers: tuple[str, ...] | None = None,\n) -> dict[str, dict]:\n    """Load only graph layers required by the selected baseline.\n\n    CodeNet has more than 135K endpoints and four stored graph views per\n    endpoint. Keeping unused views as Python objects can exhaust Kaggle host\n    RAM before a model is constructed.\n    """\n    allowed = set(required_layers) if required_layers is not None else None\n    graphs = {}\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="Loading exported program graphs", unit="code"):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            code_id = str(row.get("code_id"))\n            if code_id in wanted:\n                graphs[code_id] = {\n                    key: _adjacency(value)\n                    for key, value in row.get("graphs", {}).items()\n                    if allowed is None or key in allowed\n                }\n    return graphs\n\n\ndef _load_deckard_vectors_streaming(\n    path: Path, code_ids: list[str], train_ids: set[str],\n) -> tuple[dict[str, tuple[np.ndarray, np.ndarray]], set[str]]:\n    """Build DECKARD subtree vectors in two streaming passes.\n\n    DECKARD keeps only 32 small uint8 vectors per endpoint, but the shared\n    loader first materializes every exported AST as decoded Python objects.\n    At CodeNet\'s scale that intermediate alone can exceed Kaggle host RAM, so\n    the run dies or thrashes long before scoring.  Reading each line once and\n    reducing it immediately keeps the peak at the retained vectors.\n    """\n    wanted = set(code_ids)\n    frequency: dict[str, int] = {}\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="DECKARD vocabulary pass", unit="code", leave=False):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            if str(row.get("code_id")) not in train_ids:\n                continue\n            adjacency = _adjacency(row.get("graphs", {}).get("ast"))\n            for node_type in adjacency.get("node_types", [])[:MAX_NODES]:\n                value = str(node_type)\n                frequency[value] = frequency.get(value, 0) + 1\n    ordered = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))\n    vocab = {"<pad>": 0, "<unk>": 1, **{value: index + 2 for index, (value, _) in enumerate(ordered)}}\n\n    vectors: dict[str, tuple[np.ndarray, np.ndarray]] = {}\n    usable: set[str] = set()\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="DECKARD subtree vectors", unit="code"):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            code_id = str(row.get("code_id"))\n            if code_id not in wanted:\n                continue\n            adjacency = _adjacency(row.get("graphs", {}).get("ast"))\n            if not adjacency.get("node_types"):\n                continue\n            types, *_, children = _tree_arrays(adjacency, vocab)\n            n = int((types != 0).sum())\n            subtree_vectors, sizes = deckard_subtree_vectors(\n                types[:n], children, len(vocab), min_nodes=3,\n            )\n            order = np.argsort(sizes)[-32:]\n            # Category counts and subtree sizes are bounded by MAX_NODES, so\n            # uint8 is lossless; deckard_pair_similarity promotes to float32.\n            vectors[code_id] = (\n                subtree_vectors[order].astype(np.uint8, copy=False),\n                sizes[order].astype(np.uint8, copy=False),\n            )\n            usable.add(code_id)\n    return vectors, usable\n\n\n_TREE_ARRAY_SPEC = (\n    ("node_types", MAX_NODES, np.int32),\n    ("edge_parents", MAX_EDGES, np.int16),\n    ("edge_children", MAX_EDGES, np.int16),\n    ("depths", MAX_NODES, np.int16),\n    ("binary_depths", MAX_NODES, np.int16),\n    ("left_child", MAX_NODES, np.int16),\n    ("right_sibling", MAX_NODES, np.int16),\n    ("statements", MAX_STATEMENTS, np.int16),\n)\n\n\ndef _load_tree_arrays_streaming(\n    path: Path, code_ids: list[str], train_ids: set[str],\n) -> tuple[dict[str, np.ndarray], dict[str, int], set[str], dict[str, int]]:\n    """Pack ASTNN/RtvNN/CDLH tensors without retaining decoded ASTs.\n\n    These baselines need one AST view per endpoint, but the shared loader\n    decodes every exported graph into Python objects and holds them all before\n    packing.  At CodeNet\'s endpoint count that peak can exceed Kaggle host RAM\n    even though the packed arrays themselves are modest.\n    """\n    wanted = set(code_ids)\n    frequency: dict[str, int] = {}\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="AST vocabulary pass", unit="code", leave=False):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            if str(row.get("code_id")) not in train_ids:\n                continue\n            adjacency = _adjacency(row.get("graphs", {}).get("ast"))\n            for node_type in adjacency.get("node_types", [])[:MAX_NODES]:\n                value = str(node_type)\n                frequency[value] = frequency.get(value, 0) + 1\n    ordered = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))\n    vocab = {"<pad>": 0, "<unk>": 1, **{value: index + 2 for index, (value, _) in enumerate(ordered)}}\n\n    id_to_row = {code_id: index for index, code_id in enumerate(code_ids)}\n    count = len(code_ids)\n    arrays = {\n        name: np.zeros((count, width), dtype=dtype)\n        for name, width, dtype in _TREE_ARRAY_SPEC\n    }\n    for name in ("edge_parents", "edge_children", "left_child", "right_sibling", "statements"):\n        arrays[name].fill(-1)\n    usable: set[str] = set()\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="Packing ASTs", unit="code"):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            code_id = str(row.get("code_id"))\n            target = id_to_row.get(code_id)\n            if target is None:\n                continue\n            adjacency = _adjacency(row.get("graphs", {}).get("ast"))\n            if not adjacency.get("node_types"):\n                continue\n            packed = _tree_arrays(adjacency, vocab)[:8]\n            for (name, _width, _dtype), value in zip(_TREE_ARRAY_SPEC, packed):\n                arrays[name][target] = value\n            usable.add(code_id)\n    return arrays, vocab, usable, id_to_row\n\n\ndef _load_fa_arrays_streaming(\n    path: Path, code_ids: list[str], train_ids: set[str], required_layers: tuple[str, ...],\n) -> tuple[dict[str, np.ndarray], dict[str, int], set[str], dict[str, int]]:\n    """Build FA-AST tensors in two streaming passes without retaining raw JSON.\n\n    FA-AST needs four graph views per CodeNet endpoint.  Holding their decoded\n    Python dictionaries before packing can exceed Kaggle host RAM.  Its type\n    vocabulary is the only dependency between the two passes, so first count\n    training CPG types and then fill the final compact arrays directly.\n    """\n    wanted = set(code_ids)\n    frequency: dict[str, int] = {}\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="FA-AST vocabulary pass", unit="code", leave=False):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            if str(row.get("code_id")) not in train_ids:\n                continue\n            cpg = _adjacency(row.get("graphs", {}).get("cpg"))\n            for node_type in cpg.get("node_types", [])[:MAX_NODES]:\n                value = str(node_type)\n                frequency[value] = frequency.get(value, 0) + 1\n    ordered = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))\n    vocab = {"<pad>": 0, "<unk>": 1, **{value: index + 2 for index, (value, _) in enumerate(ordered)}}\n    id_to_row = {code_id: index for index, code_id in enumerate(code_ids)}\n    count = len(code_ids)\n    arrays = {\n        "node_types": np.zeros((count, MAX_NODES), dtype=np.int32),\n        "edge_src": np.full((count, len(RELATIONS), MAX_EDGES), -1, dtype=np.int16),\n        "edge_dst": np.full((count, len(RELATIONS), MAX_EDGES), -1, dtype=np.int16),\n    }\n    usable: set[str] = set()\n    with _open_text(path) as source:\n        for line in tqdm(source, desc="FA-AST compact graph pass", unit="code"):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            code_id = str(row.get("code_id"))\n            target = id_to_row.get(code_id)\n            if target is None:\n                continue\n            graphs = {\n                name: _adjacency(value)\n                for name, value in row.get("graphs", {}).items()\n                if name in required_layers\n            }\n            if not all(graphs.get(name, {}).get("node_types") for name in required_layers):\n                continue\n            node_types, edge_src, edge_dst = _flow_arrays(graphs, vocab)\n            arrays["node_types"][target] = node_types\n            arrays["edge_src"][target] = edge_src\n            arrays["edge_dst"][target] = edge_dst\n            usable.add(code_id)\n    return arrays, vocab, usable, id_to_row\n\n\ndef _training_type_vocabulary(raw: dict[str, dict], train_ids: set[str], graph_view: str) -> dict[str, int]:\n    frequency: dict[str, int] = {}\n    for code_id in train_ids:\n        adjacency = raw.get(code_id, {}).get(graph_view, {})\n        for node_type in adjacency.get("node_types", [])[:MAX_NODES]:\n            value = str(node_type)\n            frequency[value] = frequency.get(value, 0) + 1\n    ordered = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))\n    return {"<pad>": 0, "<unk>": 1, **{value: index + 2 for index, (value, _) in enumerate(ordered)}}\n\n\ndef _statement_roots(types: list[str], children: list[list[int]]) -> list[int]:\n    indegree = [0] * len(types)\n    for child_list in children:\n        for child in child_list:\n            indegree[child] += 1\n    roots = [index for index, degree in enumerate(indegree) if degree == 0]\n    methods = [index for index in roots if types[index].upper() == "METHOD"] or roots[:1]\n    statements = []\n    for method in methods:\n        direct = children[method]\n        blocks = [node for node in direct if types[node].upper() == "BLOCK"]\n        for parent in blocks or [method]:\n            statements.extend(children[parent])\n    return list(dict.fromkeys(statements or methods or [0]))[:MAX_STATEMENTS]\n\n\ndef _tree_arrays(adjacency: dict, vocab: dict[str, int]):\n    raw_types = [str(value) for value in adjacency.get("node_types", [])]\n    n = min(int(adjacency.get("num_nodes", 0) or 0), len(raw_types), MAX_NODES)\n    rows, cols = [], []\n    for parent, child in zip(adjacency.get("row", []), adjacency.get("col", [])):\n        parent, child = int(parent), int(child)\n        if parent < n and child < n and len(rows) < MAX_EDGES:\n            rows.append(parent)\n            cols.append(child)\n    children = children_from_edges(n, rows, cols)\n    depth = postorder_depths(children)\n    left, right = left_child_right_sibling(children)\n    binary_children = [list(filter(lambda value: value >= 0, (int(left[index]), int(right[index])))) for index in range(n)]\n    binary_depth = postorder_depths(binary_children)\n    types = np.zeros(MAX_NODES, dtype=np.int32)\n    types[:n] = [vocab.get(value, 1) for value in raw_types[:n]]\n    parents = np.full(MAX_EDGES, -1, dtype=np.int16)\n    child_array = np.full(MAX_EDGES, -1, dtype=np.int16)\n    parents[:len(rows)], child_array[:len(cols)] = rows, cols\n    depths = np.zeros(MAX_NODES, dtype=np.int16)\n    binary_depths = np.zeros(MAX_NODES, dtype=np.int16)\n    left_array = np.full(MAX_NODES, -1, dtype=np.int16)\n    right_array = np.full(MAX_NODES, -1, dtype=np.int16)\n    depths[:n], binary_depths[:n], left_array[:n], right_array[:n] = depth, binary_depth, left, right\n    statements = np.full(MAX_STATEMENTS, -1, dtype=np.int16)\n    roots = _statement_roots(raw_types[:n], children) if n else []\n    statements[:len(roots)] = roots\n    return types, parents, child_array, depths, binary_depths, left_array, right_array, statements, children\n\n\ndef _map_relation(layer: dict, universe_ids: list[str]) -> tuple[np.ndarray, np.ndarray]:\n    lookup = {str(node_id): index for index, node_id in enumerate(universe_ids)}\n    layer_ids = [str(value) for value in layer.get("node_ids", [])]\n    source = np.full(MAX_EDGES, -1, dtype=np.int16)\n    target = np.full(MAX_EDGES, -1, dtype=np.int16)\n    edges = []\n    for row, col in zip(layer.get("row", []), layer.get("col", [])):\n        row, col = int(row), int(col)\n        if row < len(layer_ids) and col < len(layer_ids):\n            u, v = lookup.get(layer_ids[row]), lookup.get(layer_ids[col])\n            if u is not None and v is not None and u < MAX_NODES and v < MAX_NODES:\n                edges.append((u, v))\n                if len(edges) >= MAX_EDGES:\n                    break\n    if edges:\n        source[:len(edges)], target[:len(edges)] = zip(*edges)\n    return source, target\n\n\ndef _flow_universe(graphs: dict, cpg: dict) -> list[int]:\n    """Rank CPG node positions by how many relation edges they carry.\n\n    A graph larger than MAX_NODES must still be represented by a connected\n    subgraph.  Keeping an arbitrary node prefix instead leaves nearly every\n    edge with an endpoint outside the retained window, because each exported\n    layer orders its nodes independently.  The surviving graph is then\n    edgeless, message passing degenerates to a constant, and the model cannot\n    train at all.\n    """\n    node_ids = [str(value) for value in cpg.get("node_ids", [])]\n    if len(node_ids) <= MAX_NODES:\n        return list(range(len(node_ids)))\n    position = {node_id: index for index, node_id in enumerate(node_ids)}\n    degree = np.zeros(len(node_ids), dtype=np.int64)\n    neighbours: dict[int, list[int]] = {}\n    for name in ("ast", "cfg", "ddg"):\n        layer = graphs.get(name, {})\n        layer_ids = [str(value) for value in layer.get("node_ids", [])]\n        for row, col in zip(layer.get("row", []), layer.get("col", [])):\n            row, col = int(row), int(col)\n            if row >= len(layer_ids) or col >= len(layer_ids):\n                continue\n            source = position.get(layer_ids[row])\n            target = position.get(layer_ids[col])\n            if source is None or target is None or source == target:\n                continue\n            degree[source] += 1\n            degree[target] += 1\n            neighbours.setdefault(source, []).append(target)\n            neighbours.setdefault(target, []).append(source)\n    # Grow a connected region outward from the busiest node.  Selecting the\n    # globally highest-degree nodes alone tends to pick mutually distant hubs,\n    # whose connecting edges all fall outside the window.\n    order = np.lexsort((np.arange(len(node_ids)), -degree))\n    selected: set[int] = set()\n    for seed in order:\n        if len(selected) >= MAX_NODES:\n            break\n        seed = int(seed)\n        if seed in selected:\n            continue\n        queue, selected = [seed], selected | {seed}\n        while queue and len(selected) < MAX_NODES:\n            current = queue.pop(0)\n            ranked_neighbours = sorted(\n                set(neighbours.get(current, ())), key=lambda node: (-degree[node], node)\n            )\n            for node in ranked_neighbours:\n                if len(selected) >= MAX_NODES:\n                    break\n                if node not in selected:\n                    selected.add(node)\n                    queue.append(node)\n    return sorted(selected)\n\n\ndef _flow_arrays(graphs: dict, vocab: dict[str, int]):\n    cpg = graphs.get("cpg") or graphs.get("ast") or {}\n    raw_ids = [str(value) for value in cpg.get("node_ids", [])]\n    raw_types = [str(value) for value in cpg.get("node_types", [])]\n    selected = _flow_universe(graphs, cpg)\n    universe_ids = [raw_ids[index] for index in selected]\n    universe_types = [raw_types[index] for index in selected if index < len(raw_types)]\n    types = np.zeros(MAX_NODES, dtype=np.int32)\n    types[:len(universe_types)] = [vocab.get(value, 1) for value in universe_types]\n    sources = np.full((len(RELATIONS), MAX_EDGES), -1, dtype=np.int16)\n    targets = np.full_like(sources, -1)\n    for relation, (name, reverse) in enumerate((\n        ("ast", False), ("ast", True),\n        ("cfg", False), ("cfg", True),\n        ("ddg", False), ("ddg", True),\n    )):\n        src, dst = _map_relation(graphs.get(name, {}), universe_ids)\n        sources[relation], targets[relation] = (dst, src) if reverse else (src, dst)\n    return types, sources, targets\n\n\ndef _deepsim_arrays(graphs: dict, vocab: dict[str, int]):\n    cfg = graphs.get("cfg", {})\n    cfg_ids = [str(value) for value in cfg.get("node_ids", [])][:MAX_NODES]\n    cfg_types = [str(value) for value in cfg.get("node_types", [])][:len(cfg_ids)]\n    types = np.zeros(MAX_NODES, dtype=np.int32)\n    types[:len(cfg_types)] = [vocab.get(value, 1) for value in cfg_types]\n    src, dst = _map_relation(cfg, cfg_ids)\n    ddg_src, ddg_dst = _map_relation(graphs.get("ddg", {}), cfg_ids)\n    # [CFG in/out/degree/branch/active/position,\n    #  DDG in/out/degree/source/sink/active].  This is the portable analogue\n    # of the original DeepSim control/data-flow semantic matrix.\n    numeric = np.zeros((MAX_NODES, 12), dtype=np.float32)\n    for u, v in zip(src[src >= 0], dst[dst >= 0]):\n        numeric[int(u), 0] += 1\n        numeric[int(v), 1] += 1\n    numeric[:, 2] = numeric[:, 0] + numeric[:, 1]\n    numeric[:, 3] = (numeric[:, 0] > 1).astype(np.float32)\n    numeric[:, 4] = (numeric[:, 2] > 0).astype(np.float32)\n    if cfg_ids:\n        numeric[:len(cfg_ids), 5] = np.linspace(0, 1, len(cfg_ids), dtype=np.float32)\n    for u, v in zip(ddg_src[ddg_src >= 0], ddg_dst[ddg_dst >= 0]):\n        numeric[int(u), 6] += 1\n        numeric[int(v), 7] += 1\n    numeric[:, 8] = numeric[:, 6] + numeric[:, 7]\n    numeric[:, 9] = (numeric[:, 6] > 0).astype(np.float32)\n    numeric[:, 10] = (numeric[:, 7] > 0).astype(np.float32)\n    numeric[:, 11] = (numeric[:, 8] > 0).astype(np.float32)\n    numeric[:, (0, 1, 2, 6, 7, 8)] = np.log1p(numeric[:, (0, 1, 2, 6, 7, 8)])\n    return types, numeric, src, dst\n\n\nclass _PairRows(Dataset):\n    def __init__(self, frame: pd.DataFrame, id_to_row: dict[str, int]):\n        self.left = frame.left_id.map(id_to_row).to_numpy(np.int64)\n        self.right = frame.right_id.map(id_to_row).to_numpy(np.int64)\n        self.labels = frame.label.to_numpy(np.float32)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, index):\n        return self.left[index], self.right[index], self.labels[index]\n\n\nclass _DepthBucketBatchSampler:\n    """Randomize batches while keeping similarly deep CDLH trees together.\n\n    A Tree-LSTM batch executes up to its deepest tree level.  Ordinary random\n    batching therefore makes a single deep AST force every shallow pair in the\n    batch through extra padded levels.  This sampler retains every row and the\n    configured logical batch size, randomizes both bucket order and row order\n    each epoch, and only changes batch composition to reduce that wasted work.\n    """\n\n    def __init__(self, pair_depths: np.ndarray, batch_size: int, seed: int) -> None:\n        self.order = np.argsort(np.asarray(pair_depths), kind="stable")\n        self.batch_size = max(1, int(batch_size))\n        self.seed = int(seed)\n        self.epoch = 0\n\n    def __len__(self) -> int:\n        return int(np.ceil(len(self.order) / self.batch_size))\n\n    def __iter__(self):\n        rng = np.random.default_rng(self.seed + self.epoch)\n        self.epoch += 1\n        buckets = [\n            self.order[start:start + self.batch_size]\n            for start in range(0, len(self.order), self.batch_size)\n        ]\n        for bucket_index in rng.permutation(len(buckets)):\n            # Preserve depth locality while avoiding a fixed order inside ties.\n            yield rng.permutation(buckets[int(bucket_index)]).tolist()\n\n\ndef _pair_tree_depths(frame: pd.DataFrame, id_to_row: dict[str, int], tree_depths: np.ndarray) -> np.ndarray:\n    left = frame.left_id.map(id_to_row).to_numpy(np.int64)\n    right = frame.right_id.map(id_to_row).to_numpy(np.int64)\n    return np.maximum(tree_depths[left], tree_depths[right])\n\n\ndef _sort_pairs_by_tree_depth(frame: pd.DataFrame, id_to_row: dict[str, int], tree_depths: np.ndarray) -> pd.DataFrame:\n    """Reorder evaluation rows only; labels/languages remain aligned with scores."""\n    order = np.argsort(_pair_tree_depths(frame, id_to_row, tree_depths), kind="stable")\n    return frame.iloc[order].reset_index(drop=True)\n\n\ndef _pair_row_indices(frame: pd.DataFrame, id_to_row: dict[str, int]) -> tuple[np.ndarray, np.ndarray]:\n    return (\n        frame.left_id.map(id_to_row).to_numpy(np.int64),\n        frame.right_id.map(id_to_row).to_numpy(np.int64),\n    )\n\n\ndef _endpoint_anchors(left_rows: np.ndarray, right_rows: np.ndarray) -> np.ndarray:\n    """Choose the more frequently occurring endpoint as each pair\'s anchor."""\n    counts = np.bincount(np.concatenate((left_rows, right_rows)))\n    return np.where(counts[left_rows] >= counts[right_rows], left_rows, right_rows)\n\n\nclass _EndpointGroupedDepthBatchSampler:\n    """Full-data CDLH batches with a repeated endpoint and comparable depth.\n\n    The model deduplicates endpoint rows *within* each batch.  Keeping pairs\n    with their most frequent endpoint together therefore avoids repeated\n    Tree-LSTM calls while preserving every pair exactly once per epoch.\n    """\n\n    def __init__(\n        self, left_rows: np.ndarray, right_rows: np.ndarray, pair_depths: np.ndarray,\n        batch_size: int, seed: int,\n    ) -> None:\n        self.batch_size = max(1, int(batch_size))\n        self.seed = int(seed)\n        self.epoch = 0\n        anchors = _endpoint_anchors(left_rows, right_rows)\n        # Stable depth sorting within an anchor keeps the max Tree-LSTM level\n        # of each logical batch low in addition to enabling endpoint reuse.\n        order = np.lexsort((np.asarray(pair_depths), anchors))\n        sorted_anchors = anchors[order]\n        boundaries = np.r_[0, np.flatnonzero(np.diff(sorted_anchors)) + 1, len(order)]\n        self.groups = [order[boundaries[i]:boundaries[i + 1]] for i in range(len(boundaries) - 1)]\n        self.row_count = len(order)\n\n    def __len__(self) -> int:\n        return int(np.ceil(self.row_count / self.batch_size))\n\n    def __iter__(self):\n        rng = np.random.default_rng(self.seed + self.epoch)\n        self.epoch += 1\n        full_batches: list[np.ndarray] = []\n        remainder: list[np.ndarray] = []\n        for group_index in rng.permutation(len(self.groups)):\n            group = self.groups[int(group_index)]\n            complete = len(group) - (len(group) % self.batch_size)\n            for start in range(0, complete, self.batch_size):\n                full_batches.append(group[start:start + self.batch_size])\n            if complete < len(group):\n                remainder.append(group[complete:])\n        for batch_index in rng.permutation(len(full_batches)):\n            yield rng.permutation(full_batches[int(batch_index)]).tolist()\n        if remainder:\n            tail = rng.permutation(np.concatenate(remainder))\n            for start in range(0, len(tail), self.batch_size):\n                yield tail[start:start + self.batch_size].tolist()\n\n\ndef _sort_pairs_by_endpoint_and_depth(\n    frame: pd.DataFrame, id_to_row: dict[str, int], tree_depths: np.ndarray,\n) -> pd.DataFrame:\n    """Evaluation ordering that maximizes exact within-batch CDLH reuse."""\n    left_rows, right_rows = _pair_row_indices(frame, id_to_row)\n    order = np.lexsort((np.asarray(_pair_tree_depths(frame, id_to_row, tree_depths)), _endpoint_anchors(left_rows, right_rows)))\n    return frame.iloc[order].reset_index(drop=True)\n\n\nclass _PairModel(nn.Module):\n    def __init__(self, method: str, arrays: dict[str, np.ndarray], vocab_size: int):\n        super().__init__()\n        self.method = method\n        for name, value in arrays.items():\n            self.register_buffer(name, torch.from_numpy(value), persistent=False)\n        if method == "astnn":\n            self.encoder = ASTNNEncoder(vocab_size)\n            code_dim = 200\n        elif method == "rtvnn":\n            self.encoder = RtvNNEncoder(vocab_size)\n            code_dim = 192\n        elif method == "cdlh":\n            self.encoder = CDLHModel(vocab_size)\n            code_dim = 64\n        elif method == "deepsim":\n            self.encoder = DeepSimEncoder(vocab_size)\n            code_dim = 192\n        elif method == "fa_ast_ggnn":\n            self.encoder = FAASTGGNN(vocab_size)\n            code_dim = 192\n        elif method == "fa_ast_gmn":\n            self.encoder = FAASTGMN(vocab_size)\n            code_dim = 192\n        else:\n            raise ValueError(method)\n        if method == "astnn":\n            # The official clone model applies one symmetric linear decision\n            # layer to the absolute difference between code vectors.\n            self.classifier = nn.Linear(code_dim, 1)\n        elif method != "cdlh":\n            self.classifier = nn.Sequential(\n                nn.Linear(code_dim * 2 + 2, 192), nn.ReLU(), nn.Dropout(0.2), nn.Linear(192, 1)\n            )\n\n    def _tree(self, rows: Tensor):\n        args = (\n            self.node_types[rows].long(), self.edge_parents[rows].long(),\n            self.edge_children[rows].long(), self.depths[rows].long()\n        )\n        if self.method == "astnn":\n            return self.encoder(*args, self.statements[rows].long()), args[0].new_zeros((), dtype=torch.float32)\n        if self.method == "rtvnn":\n            return self.encoder(*args)\n        return self.encoder.encode(\n            args[0], self.left_child[rows].long(), self.right_sibling[rows].long(),\n            self.binary_depths[rows].long()\n        ), args[0].new_zeros((), dtype=torch.float32)\n\n    def _single(self, rows: Tensor):\n        if self.method in {"astnn", "rtvnn", "cdlh"}:\n            return self._tree(rows)\n        if self.method == "deepsim":\n            return self.encoder(\n                self.node_types[rows].long(), self.numeric[rows].float(),\n                self.edge_src[rows].long(), self.edge_dst[rows].long()\n            ), self.node_types[rows].new_zeros((), dtype=torch.float32)\n        return self.encoder(\n            self.node_types[rows].long(), self.edge_src[rows].long(), self.edge_dst[rows].long()\n        ), self.node_types[rows].new_zeros((), dtype=torch.float32)\n\n    def _flow(self, rows: Tensor) -> tuple[Tensor, Tensor, Tensor]:\n        """Return the exact non-padding envelope for a flow-graph batch.\n\n        Packing uses fixed 128-node / 512-edge tensors so the corpus can live\n        in one compact buffer.  Running GGNN/GMN over their trailing padding\n        is mathematically unnecessary: padded nodes are masked and padded\n        edges are invalid.  Cropping therefore preserves the baseline\'s\n        outputs while substantially reducing CodeNet\'s message passing and\n        GMN attention work.\n        """\n        node_types = self.node_types[rows].long()\n        edge_src = self.edge_src[rows].long()\n        edge_dst = self.edge_dst[rows].long()\n        nodes = max(1, int(node_types.ne(0).sum(1).max().item()))\n        valid_edges = edge_src.ge(0) & edge_dst.ge(0)\n        # Crop to the last occupied slot, not to the largest edge *count*.\n        # Relations are packed in discovery order, so a sparse relation can\n        # hold its edges beyond a denser relation\'s count; cropping by count\n        # silently dropped those edges and left whole relations empty.\n        if valid_edges.any():\n            positions = valid_edges.nonzero()[:, -1]\n            edges = int(positions.max().item()) + 1\n        else:\n            edges = 1\n        return node_types[:, :nodes], edge_src[:, :, :edges], edge_dst[:, :, :edges]\n\n    def forward(self, left_rows: Tensor, right_rows: Tensor):\n        if self.method == "fa_ast_gmn":\n            left_types, left_src, left_dst = self._flow(left_rows)\n            right_types, right_src, right_dst = self._flow(right_rows)\n            left, right = self.encoder.forward_pair(\n                left_types, left_src, left_dst, right_types, right_src, right_dst,\n            )\n            auxiliary = left.new_zeros(())\n        elif self.method == "cdlh":\n            # CDLH has no stochastic encoder layer.  If a code occurs in more\n            # than one pair of the logical batch, one Tree-LSTM evaluation and\n            # multiple indexed uses have exactly the same loss and accumulated\n            # gradient as evaluating that identical tree repeatedly.  Grouped\n            # full-data batches below make this reuse substantial on AtCoder\n            # and XGLUE without caching across optimizer steps.\n            rows = torch.cat((left_rows, right_rows), dim=0)\n            unique_rows, inverse = torch.unique(rows, sorted=False, return_inverse=True)\n            unique_codes, auxiliary = self._single(unique_rows)\n            left = unique_codes.index_select(0, inverse[:len(left_rows)])\n            right = unique_codes.index_select(0, inverse[len(left_rows):])\n        else:\n            if self.method == "fa_ast_ggnn":\n                left_types, left_src, left_dst = self._flow(left_rows)\n                right_types, right_src, right_dst = self._flow(right_rows)\n                left = self.encoder(left_types, left_src, left_dst)\n                right = self.encoder(right_types, right_src, right_dst)\n                left_aux = right_aux = left.new_zeros(())\n            else:\n                left, left_aux = self._single(left_rows)\n                right, right_aux = self._single(right_rows)\n            auxiliary = left_aux + right_aux\n        if self.method == "cdlh":\n            logits = 4.0 * (left * right).mean(-1)\n        elif self.method == "astnn":\n            logits = self.classifier(torch.abs(left - right)).squeeze(-1)\n        else:\n            features = torch.cat([\n                torch.abs(left - right), left * right,\n                (left * right).sum(-1, keepdim=True), torch.norm(left - right, dim=-1, keepdim=True)\n            ], -1)\n            logits = self.classifier(features).squeeze(-1)\n        return logits, auxiliary, left, right\n\n\ndef _pack_method(method: str, raw: dict[str, dict], code_ids: list[str], train_ids: set[str]):\n    graph_view = "cfg" if method == "deepsim" else "cpg" if method.startswith("fa_ast") else "ast"\n    vocab = _training_type_vocabulary(raw, train_ids, graph_view)\n    if method in {"astnn", "rtvnn", "cdlh"}:\n        names = (\n            "node_types", "edge_parents", "edge_children", "depths", "binary_depths",\n            "left_child", "right_sibling", "statements"\n        )\n        rows = [_tree_arrays(raw[code_id]["ast"], vocab)[:8] for code_id in tqdm(code_ids, desc=f"Packing {method} ASTs")]\n    elif method == "deepsim":\n        names = ("node_types", "numeric", "edge_src", "edge_dst")\n        rows = [_deepsim_arrays(raw[code_id], vocab) for code_id in tqdm(code_ids, desc="Packing CFG semantic matrices")]\n    else:\n        names = ("node_types", "edge_src", "edge_dst")\n        rows = [_flow_arrays(raw[code_id], vocab) for code_id in tqdm(code_ids, desc="Packing typed FA-AST relations")]\n    arrays = {name: np.stack([row[index] for row in rows]) for index, name in enumerate(names)}\n    return arrays, vocab\n\n\ndef _is_cuda_oom(error: BaseException) -> bool:\n    return isinstance(error, torch.OutOfMemoryError) or (\n        isinstance(error, RuntimeError) and "out of memory" in str(error).lower()\n    )\n\n\n@torch.no_grad()\ndef _predict_rows_adaptive(model, left, right, device):\n    """Predict one batch, bisecting only a batch that does not fit the GPU."""\n    try:\n        logits, _, _, _ = model(left.to(device), right.to(device))\n        return torch.sigmoid(logits).float().cpu()\n    except RuntimeError as error:\n        if not _is_cuda_oom(error) or len(left) <= 1:\n            raise\n        del error\n        gc.collect()\n        torch.cuda.empty_cache()\n        middle = len(left) // 2\n        return torch.cat((\n            _predict_rows_adaptive(model, left[:middle], right[:middle], device),\n            _predict_rows_adaptive(model, left[middle:], right[middle:], device),\n        ))\n\n\n@torch.no_grad()\ndef _predict(model, frame, id_to_row, batch_size, device):\n    model.eval()\n    output = []\n    loader = DataLoader(_PairRows(frame, id_to_row), batch_size=batch_size, shuffle=False, num_workers=0)\n    for left, right, _ in tqdm(loader, desc="Predict", leave=False):\n        output.append(_predict_rows_adaptive(model, left, right, device).numpy())\n    return np.concatenate(output) if output else np.empty(0, dtype=np.float32)\n\n\ndef _adaptive_train_step(\n    model, method, optimizer, scaler, left, right, labels, device,\n    maximum_microbatch=None,\n):\n    """Run a logical batch and halve only its microbatch after a CUDA OOM.\n\n    Gradient weighting preserves the same mean loss as the unsplit logical\n    batch.  Most batches therefore retain CDLH\'s fast batch size of 16, while\n    an unusually deep AST can safely fall back to 8, 4, 2, or 1.\n    """\n    logical_rows = len(labels)\n    microbatch = min(logical_rows, maximum_microbatch or logical_rows)\n    while True:\n        optimizer.zero_grad(set_to_none=True)\n        weighted_loss = 0.0\n        chunk_left = chunk_right = chunk_labels = None\n        logits = auxiliary = left_code = right_code = None\n        chunk_loss = scaled_loss = None\n        try:\n            for start in range(0, logical_rows, microbatch):\n                stop = min(logical_rows, start + microbatch)\n                chunk_left = left[start:stop].to(device)\n                chunk_right = right[start:stop].to(device)\n                chunk_labels = labels[start:stop].to(device)\n                with torch.amp.autocast("cuda", enabled=device.type == "cuda"):\n                    logits, auxiliary, left_code, right_code = model(chunk_left, chunk_right)\n                    auxiliary = auxiliary.mean()\n                    if method == "cdlh":\n                        chunk_loss = CDLHModel.loss(left_code, right_code, chunk_labels)\n                    else:\n                        chunk_loss = nn.functional.binary_cross_entropy_with_logits(logits, chunk_labels)\n                    if method == "rtvnn":\n                        chunk_loss = chunk_loss + 0.05 * auxiliary\n                    weight = len(chunk_labels) / logical_rows\n                    scaled_loss = chunk_loss * weight\n                scaler.scale(scaled_loss).backward()\n                weighted_loss += float(chunk_loss.detach().cpu()) * len(chunk_labels)\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            return weighted_loss, microbatch\n        except RuntimeError as error:\n            if not _is_cuda_oom(error) or microbatch <= 1:\n                raise\n            previous = microbatch\n            microbatch = max(1, microbatch // 2)\n            optimizer.zero_grad(set_to_none=True)\n            chunk_left = chunk_right = chunk_labels = None\n            logits = auxiliary = left_code = right_code = None\n            chunk_loss = scaled_loss = None\n            del error\n            gc.collect()\n            torch.cuda.empty_cache()\n            print(f"[CUDA OOM recovery] retrying logical batch with microbatch {previous} -> {microbatch}")\n\n\ndef _gpu_execution_metadata() -> dict[str, object]:\n    """Describe the actual accelerator layout in every result manifest."""\n    if not torch.cuda.is_available():\n        return {\n            "device": "cpu", "available_gpu_count": 0, "used_gpu_count": 0,\n            "data_parallel": False, "gpus": [],\n        }\n    count = torch.cuda.device_count()\n    return {\n        "device": "cuda:0",\n        "available_gpu_count": count,\n        "used_gpu_count": 1,\n        "data_parallel": False,\n        "parallelism_note": (\n            "Faithful graph baselines keep corpus tensors resident on one GPU; "\n            "per-batch DataParallel replication would dominate runtime."\n        ),\n        "gpus": [torch.cuda.get_device_name(index) for index in range(count)],\n    }\n\n\ndef _run_deckard(raw, train, valid, test, id_to_row, train_ids, vectors=None):\n    if vectors is None:\n        vocab = _training_type_vocabulary(raw, train_ids, "ast")\n        vectors = {}\n        total_codes = len(id_to_row)\n        print(f"[Deckard] computing subtree vectors for {total_codes:,} code endpoints on CPU...", flush=True)\n        for index, code_id in enumerate(tqdm(id_to_row, desc="DECKARD subtree vectors"), start=1):\n            # The raw JSON graph is needed exactly once. Free it immediately:\n            # CodeNet has >134k endpoints, and retaining raw graphs alongside every\n            # code\'s subtree vectors can exceed Kaggle host memory before scoring.\n            graph = raw.pop(code_id)\n            types, *_, children = _tree_arrays(graph["ast"], vocab)\n            n = int((types != 0).sum())\n            subtree_vectors, sizes = deckard_subtree_vectors(types[:n], children, len(vocab), min_nodes=3)\n            order = np.argsort(sizes)[-32:]\n            # Category counts and subtree sizes are bounded by MAX_NODES (128 for\n            # CodeNet), so uint8 is lossless. Cosine math promotes each retained\n            # pair back to float32 in deckard_pair_similarity.\n            vectors[code_id] = (\n                subtree_vectors[order].astype(np.uint8, copy=False),\n                sizes[order].astype(np.uint8, copy=False),\n            )\n            if index % 10_000 == 0 or index == total_codes:\n                print(f"[Deckard] subtree vectors: {index:,}/{total_codes:,}", flush=True)\n        del raw\n        gc.collect()\n\n    def scores(frame):\n        print(f"[Deckard] scoring {len(frame):,} pairs...", flush=True)\n        return np.asarray([\n            deckard_pair_similarity(*vectors[left], *vectors[right])\n            for left, right in tqdm(\n                zip(frame.left_id, frame.right_id), total=len(frame),\n                desc="DECKARD pair scoring", leave=False,\n            )\n        ], dtype=np.float32)\n\n    valid_scores = scores(valid)\n    threshold, valid_metrics = _choose_threshold(valid.label, valid_scores)\n    return threshold, valid_metrics, scores(test), 0, [], None\n\n\ndef run_faithful_baseline(dataset_key: str, method: str) -> dict:\n    if method not in METHOD_CONFIGS:\n        raise ValueError(f"Unknown method {method!r}")\n    config = dict(METHOD_CONFIGS[method])\n    codenet = dataset_key.lower().replace("_", "-") == "codenet-4l"\n    if codenet:\n        # CodeNet has far more graph endpoints. CDLH evaluates two checkpointed\n        # Tree-LSTMs per pair, so these logical batches are safe on one 16-GB T4.\n        codenet_batches = {\n            "astnn": 16, "rtvnn": 16, "cdlh": 4, "deepsim": 16,\n            "fa_ast_ggnn": 4, "fa_ast_gmn": 2,\n        }\n        if method in codenet_batches:\n            config["batch"] = min(int(config["batch"]), codenet_batches[method])\n    # Older generated notebook headers contained only pair caps.  Keep the\n    # shared runtime backward compatible while making the effective schedule\n    # explicit in every output manifest.\n    run_config = dict(globals().get("RUN_CONFIG", {}))\n    max_train_pairs = run_config.get("max_train_pairs")\n    max_valid_pairs = run_config.get("max_valid_pairs")\n    max_test_pairs = run_config.get("max_test_pairs")\n    epochs = max(1, int(run_config.get("epochs", 4)))\n    patience = max(1, int(run_config.get("patience", 2)))\n    run_profile = str(globals().get("RUN_PROFILE", "final_full"))\n    seed = 42\n    _seed_everything(seed)\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    started = time.perf_counter()\n    print(f"[{config[\'name\']}] resolving input artifacts for {dataset_key}...", flush=True)\n    pairs_path = _find_input(dataset_key, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")\n    graphs_path = _find_input(\n        dataset_key,\n        "graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp",\n    )\n    # Detect compression from file bytes, not the extension: Kaggle sometimes\n    # materializes a gzip payload as a `.tmp` file.\n    with _open_text(pairs_path) as pairs_file:\n        pairs = pd.read_csv(pairs_file, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})\n    print(f"[{config[\'name\']}] loaded {len(pairs):,} benchmark pairs; selecting configured splits...", flush=True)\n    # The CodeNet main-table release is deliberately a 50k-clone / 50k-\n    # different-problem-nonclone benchmark.  Failing here prevents a Kaggle\n    # attachment of the older 12k scope-study ZIP from silently producing a\n    # non-comparable row in the CodeNet column.\n    if dataset_key.lower().replace("_", "-") == "codenet-4l":\n        observed = pairs.groupby("split").size().astype(int).to_dict()\n        expected = {"train": 70_000, "valid": 15_000, "test": 15_000}\n        if observed != expected:\n            raise RuntimeError(\n                "CodeNet baselines require the final clone50k_diff50k clean-data release "\n                f"with split counts {expected}; attached input has {observed}."\n            )\n    if "gpt" in dataset_key.lower() and (pairs.left_id == pairs.right_id).any():\n        raise RuntimeError("Identity/self pairs are forbidden in the rebuilt benchmark")\n    train = _limit(pairs, "train", max_train_pairs, seed)\n    valid = _limit(pairs, "valid", max_valid_pairs, seed + 1)\n    test = _limit(pairs, "test", max_test_pairs, seed + 2)\n    code_ids = sorted(set(train.left_id) | set(train.right_id) | set(valid.left_id) | set(valid.right_id) | set(test.left_id) | set(test.right_id))\n    train_ids = set(train.left_id) | set(train.right_id)\n    required = ("cfg", "ddg") if method == "deepsim" else ("ast", "cfg", "ddg", "cpg") if method.startswith("fa_ast") else ("ast",)\n    print(\n        f"[{config[\'name\']}] loading {required} graph layer(s) for {len(code_ids):,} code endpoints...",\n        flush=True,\n    )\n    prepacked_arrays = None\n    prepacked_vocab = None\n    prepacked_id_to_row = None\n    deckard_vectors = None\n    if method == "deckard":\n        print(f"[{config[\'name\']}] streaming ASTs directly into subtree vectors...", flush=True)\n        deckard_vectors, usable = _load_deckard_vectors_streaming(\n            graphs_path, code_ids, train_ids,\n        )\n        raw = None\n    elif method.startswith("fa_ast"):\n        print(f"[{config[\'name\']}] streaming FA-AST graphs directly into compact arrays...", flush=True)\n        prepacked_arrays, prepacked_vocab, usable, prepacked_id_to_row = _load_fa_arrays_streaming(\n            graphs_path, code_ids, train_ids, required,\n        )\n        raw = None\n    elif method in {"astnn", "rtvnn", "cdlh"}:\n        print(f"[{config[\'name\']}] streaming ASTs directly into compact arrays...", flush=True)\n        prepacked_arrays, prepacked_vocab, usable, prepacked_id_to_row = _load_tree_arrays_streaming(\n            graphs_path, code_ids, train_ids,\n        )\n        raw = None\n    else:\n        raw = _load_raw_graphs(graphs_path, set(code_ids), required)\n        usable = {\n            code_id for code_id in code_ids\n            if all(raw.get(code_id, {}).get(view, {}).get("node_types") for view in required)\n        }\n    train = train[train.left_id.isin(usable) & train.right_id.isin(usable)].reset_index(drop=True)\n    valid = valid[valid.left_id.isin(usable) & valid.right_id.isin(usable)].reset_index(drop=True)\n    test = test[test.left_id.isin(usable) & test.right_id.isin(usable)].reset_index(drop=True)\n    empty_splits = [name for name, frame in (("train", train), ("valid", valid), ("test", test)) if frame.empty]\n    if empty_splits:\n        raise RuntimeError(\n            f"{config[\'name\']} has no graph-evaluable rows in {empty_splits}. "\n            f"Required graph layers: {required}; verify that the matching {dataset_key} clean-data ZIP is attached."\n        )\n    code_ids = sorted(set(train.left_id) | set(train.right_id) | set(valid.left_id) | set(valid.right_id) | set(test.left_id) | set(test.right_id))\n    id_to_row = prepacked_id_to_row if prepacked_id_to_row is not None else {code_id: index for index, code_id in enumerate(code_ids)}\n    print(\n        f"[{config[\'name\']}] usable pairs: train={len(train):,}, valid={len(valid):,}, test={len(test):,}; "\n        f"packing {len(code_ids):,} graphs...",\n        flush=True,\n    )\n\n    if method == "deckard":\n        threshold, valid_metrics, test_scores, parameters, history, best_epoch = _run_deckard(\n            raw, train, valid, test, id_to_row, train_ids, vectors=deckard_vectors,\n        )\n    else:\n        if prepacked_arrays is not None:\n            arrays, vocab = prepacked_arrays, prepacked_vocab\n        else:\n            arrays, vocab = _pack_method(method, raw, code_ids, train_ids)\n        tree_depths = (\n            arrays["binary_depths"].max(axis=1).astype(np.int16, copy=False)\n            if method == "cdlh" else None\n        )\n        fa_train_pair_nodes = None\n        if method.startswith("fa_ast"):\n            train_left_rows, train_right_rows = _pair_row_indices(train, id_to_row)\n            graph_nodes = arrays["node_types"].astype(bool).sum(1).astype(np.int16, copy=False)\n            fa_train_pair_nodes = np.maximum(\n                graph_nodes[train_left_rows], graph_nodes[train_right_rows],\n            )\n        # Neural baselines only require compact arrays after this point. Free\n        # JSON-derived Python graphs before CUDA buffers are materialized.\n        if raw is not None:\n            del raw\n        gc.collect()\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        model = _PairModel(method, arrays, len(vocab)).to(device)\n        del arrays\n        gc.collect()\n        # Keep corpus buffers resident on one GPU.  ``DataParallel`` would\n        # replicate those large buffers on every batch, making full-data runs\n        # slower despite Kaggle exposing a second T4.\n        print(f"[{config[\'name\']}] using one GPU.", flush=True)\n        parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n        if method == "cdlh":\n            assert tree_depths is not None\n            train_left_rows, train_right_rows = _pair_row_indices(train, id_to_row)\n            train_pair_depths = _pair_tree_depths(train, id_to_row, tree_depths)\n            # Evaluation is order-invariant; grouping it by endpoint enables\n            # exact within-batch encoder reuse and avoids deep ASTs inflating\n            # otherwise shallow inference batches.\n            valid = _sort_pairs_by_endpoint_and_depth(valid, id_to_row, tree_depths)\n            test = _sort_pairs_by_endpoint_and_depth(test, id_to_row, tree_depths)\n            loader = DataLoader(\n                _PairRows(train, id_to_row),\n                batch_sampler=_EndpointGroupedDepthBatchSampler(\n                    train_left_rows, train_right_rows, train_pair_depths, config["batch"], seed,\n                ),\n                num_workers=0,\n                pin_memory=device.type == "cuda",\n            )\n            unique_endpoints = len(np.unique(np.concatenate((train_left_rows, train_right_rows))))\n            print(\n                f"[{config[\'name\']}] endpoint-grouped full-data batches: {len(train):,} pairs / "\n                f"{unique_endpoints:,} train endpoints; pair depth "\n                f"min/median/max={int(train_pair_depths.min())}/{int(np.median(train_pair_depths))}/{int(train_pair_depths.max())}.",\n                flush=True,\n            )\n        else:\n            dataset = _PairRows(train, id_to_row)\n            if method.startswith("fa_ast"):\n                assert fa_train_pair_nodes is not None\n                # Graph-size buckets retain every pair and randomize their\n                # order per epoch.  Combined with _flow\'s exact cropping this\n                # prevents one large graph from making every FA-AST batch pay\n                # for its 128x128 padded attention/message-passing envelope.\n                loader = DataLoader(\n                    dataset,\n                    batch_sampler=_DepthBucketBatchSampler(fa_train_pair_nodes, config["batch"], seed),\n                    num_workers=0, pin_memory=device.type == "cuda",\n                )\n                print(\n                    f"[{config[\'name\']}] size-bucketed FA-AST batches; graph nodes "\n                    f"min/median/max={int(fa_train_pair_nodes.min())}/"\n                    f"{int(np.median(fa_train_pair_nodes))}/{int(fa_train_pair_nodes.max())}.",\n                    flush=True,\n                )\n            else:\n                loader = DataLoader(\n                    dataset, batch_size=config["batch"], shuffle=True,\n                    num_workers=0, pin_memory=device.type == "cuda",\n                )\n        history, best = [], None\n        training_microbatch = config["batch"]\n        minimum_microbatch = config["batch"]\n        bad_epochs = 0\n        for epoch in range(1, epochs + 1):\n            print(\n                f"[{config[\'name\']}] epoch {epoch}/{epochs} starting "\n                f"(batch={config[\'batch\']}, train pairs={len(train):,})...",\n                flush=True,\n            )\n            model.train()\n            total_loss, total_rows = 0.0, 0\n            for left, right, labels in tqdm(loader, desc=f"{config[\'name\']} epoch {epoch}"):\n                batch_loss, used_microbatch = _adaptive_train_step(\n                    model, method, optimizer, scaler, left, right, labels, device,\n                    maximum_microbatch=training_microbatch,\n                )\n                training_microbatch = min(training_microbatch, used_microbatch)\n                minimum_microbatch = min(minimum_microbatch, used_microbatch)\n                total_loss += batch_loss\n                total_rows += len(labels)\n            valid_scores = _predict(model, valid, id_to_row, config["batch"], device)\n            candidate_threshold, candidate_metrics = _choose_threshold(valid.label, valid_scores)\n            row = {"epoch": epoch, "loss": total_loss / max(1, total_rows), **candidate_metrics, "threshold": candidate_threshold}\n            history.append(row)\n            key = _selection_key(candidate_metrics, valid.label)\n            if best is None or key > best[0]:\n                best = (key, copy.deepcopy(model.state_dict()), epoch, candidate_threshold, candidate_metrics)\n                bad_epochs = 0\n            else:\n                bad_epochs += 1\n                if bad_epochs >= patience:\n                    break\n        _, state, best_epoch, threshold, valid_metrics = best\n        model.load_state_dict(state)\n        print(f"[{config[\'name\']}] evaluating validation-selected epoch {best_epoch} on test split...", flush=True)\n        test_scores = _predict(model, test, id_to_row, config["batch"], device)\n\n    test_metrics = _metric_dict(test.label, test_scores, threshold)\n    runtime = time.perf_counter() - started\n    result = {\n        "Method": config["name"], "Implementation": "paper-faithful adaptation on repository-exported graphs",\n        "BestEpoch": best_epoch, "BestValidF1": valid_metrics["F1"], "BestValidAcc": valid_metrics["Acc"],\n        "ValidationSelectionMetric": _validation_selection(valid.label)["metric"],\n        "ValidationBalanced": _validation_selection(valid.label)["balanced"],\n        "ValidationPositives": _validation_selection(valid.label)["positives"],\n        "ValidationNegatives": _validation_selection(valid.label)["negatives"], **test_metrics,\n        "Threshold": threshold, "TrainPairs": len(train), "ValidPairs": len(valid), "TestPairs": len(test),\n        "TrainableParameters": parameters, "RuntimeSeconds": runtime, "RuntimeMinutes": runtime / 60,\n        "RunProfile": run_profile, "Seed": seed, "Execution": _gpu_execution_metadata(),\n    }\n    working = Path(globals().get("WORK_DIR_OVERRIDE", "/kaggle/working"))\n    working.mkdir(parents=True, exist_ok=True)\n    stem = f"{dataset_key}_{method}_faithful"\n    pd.DataFrame([result]).to_csv(working / f"{stem}_results.csv", index=False)\n    if history:\n        pd.DataFrame(history).to_csv(working / f"{stem}_history.csv", index=False)\n    manifest = {\n        "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),\n        "dataset_key": dataset_key, "method": config["name"], "implementation": result["Implementation"],\n        "input_graphs": list(required), "self_pair_count": 0, "hyperparameters": config,\n        "profile": run_profile, "configured_epochs": epochs, "patience": patience, "seed": seed,\n        "execution": _gpu_execution_metadata(), "global_batch_size": config["batch"],\n        "minimum_training_microbatch": minimum_microbatch if method != "deckard" else None,\n        "selected_epoch": best_epoch, "selected_threshold": threshold, "result": result,\n    }\n    (working / f"{stem}_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    record_language_breakdown(test, test_scores, threshold, dataset=dataset_key, method=config["name"])\n    print(pd.DataFrame([result]))\n    return result\n', '__init__.py': '"""Paper-faithful adaptations of clone-detection baselines.\n\nThe implementations retain this project\'s exported program graphs while\nmatching the defining representation and learning logic of each baseline.\nThey are adaptations, not byte-for-byte executions of upstream repositories.\n"""\n\nfrom .core import (\n    ASTNNEncoder,\n    BinaryTreeLSTMEncoder,\n    CDLHModel,\n    DeepSimEncoder,\n    FAASTGGNN,\n    FAASTGMN,\n    RtvNNEncoder,\n    deckard_pair_similarity,\n    deckard_subtree_vectors,\n    left_child_right_sibling,\n)\n\n__all__ = [\n    "ASTNNEncoder",\n    "BinaryTreeLSTMEncoder",\n    "CDLHModel",\n    "DeepSimEncoder",\n    "FAASTGGNN",\n    "FAASTGMN",\n    "RtvNNEncoder",\n    "deckard_pair_similarity",\n    "deckard_subtree_vectors",\n    "left_child_right_sibling",\n]\n'}
_legacy_package = _runtime / "spectral_code" / "faithful_graph_baselines"
_legacy_package.mkdir(parents=True, exist_ok=True)
for _name, _source in _legacy_sources.items():
    (_legacy_package / _name).write_text(_source, encoding="utf-8")


In [ ]:
# Supported HF environment; preserve Kaggle's installed CUDA PyTorch.
def _spectra_prepare_hf():
    import importlib
    from importlib import metadata
    import json
    from pathlib import Path
    import subprocess
    import sys
    import tempfile
    from packaging.specifiers import SpecifierSet

    required = {'transformers': '==4.57.3', 'huggingface-hub': '==0.36.0', 'tokenizers': '==0.22.1'}
    def installed(name):
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            return None

    needs = [name for name, spec in required.items()
             if installed(name) is None or installed(name) not in SpecifierSet(spec)]
    if needs:
        changed_modules = {name.replace("-", "_") for name in needs}
        already_loaded = any(name.split(".")[0] in changed_modules for name in sys.modules)
        torch_version = metadata.version("torch")
        with tempfile.TemporaryDirectory(prefix="spectra_hf_") as directory:
            constraints = Path(directory) / "keep_kaggle_torch.txt"
            constraints.write_text("torch==" + torch_version + "\n", encoding="utf-8")
            subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                            "--constraint", str(constraints),
                            *(name + spec for name, spec in required.items())], check=True)
        importlib.invalidate_caches()
        if already_loaded:
            raise RuntimeError("HF packages were corrected, but old modules are loaded. "
                               "Restart the kernel/session and Run All, or use a fresh Save & Run All. "
                               "Do not continue this kernel after changing packages.")
    actual = {name: installed(name) for name in required}
    if any(actual[name] is None or actual[name] not in SpecifierSet(spec)
           for name, spec in required.items()):
        raise RuntimeError("The supported HF environment could not be installed: " + str(actual))
    actual["torch"] = metadata.version("torch")
    print(json.dumps({"status": "hf_environment_ready", "versions": actual}), flush=True)

_spectra_prepare_hf()
import json
from spectral_code.revision.jobs import run_external
summary = run_external(method=METHOD, dataset=DATASET, seed=SEED, started=SESSION_STARTED,
                       clean_data=CLEAN_DATA_DIR, session_hours=SESSION_HOURS,
                       max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
                       batch_size=BATCH_SIZE, effective_batch_size=EFFECTIVE_BATCH_SIZE,
                       learning_rate=LEARNING_RATE, transfer_key=TRANSFER_KEY,
                       matched_pairs=MATCHED_TRAIN_PAIRS, resume_checkpoint=RESUME_CHECKPOINT,
                       resume_tokens=RESUME_TOKENS_DIR, model_revision=MODEL_REVISION,
                       final_test=FINAL_TEST, measure_cost=MEASURE_COST,
                       allow_numerical_upgrade=ALLOW_NUMERICAL_UPGRADE)
print(json.dumps(summary, indent=2))
